# 🧠 Text-to-SQL: Tradução de Linguagem Natural para SQL

**Projeto de NLP — Seq2Seq com Atenção de Bahdanau + Fine-Tuning com T5**

Este notebook implementa um pipeline completo de tradução de perguntas em linguagem natural (português e inglês) para queries SQL, com:

- **Parte 1:** Setup de dados e dependencias
- **Parte 2:** Análise exploratória dos dados base
- **Parte 3:** Pré-processamento de dados e vocabulários
- **Parte 4:** Arquitetura Seq2Seq com atenção de Bahdanau
- **Parte 5:** Treinamento de LSTM com dados base e comparação PT vs EN
- **Parte 6:** Incremento de dados EN, EDA, retreinamento de LSTM e comparação PT vs EN
- **Parte 7:** Fine-Tuning com T5 e variantes (com e sem schema do banco)

---
**Datasets utilizados (dados antes de limpeza e verificação de duplicatas):**
- `xlangai/spider` — Spider EN: 8.034 registros em inglês
- `Boakpe/spider-test-portuguese` — Spider PT: 2147 registros em português
- `C4AI` — 2 pares PT extraídos de notebooks
- `WikiSQL` - 74.059 registros em inglês


## ⚙️ PARTE 1 — Setup de Dados e Dependencias

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C1 — INSTALAÇÃO DE DEPENDÊNCIAS                                         ║
# ║  Instala todas as bibliotecas necessárias para o projeto.                ║
# ║  sys.executable garante que o pip do ambiente correto seja usado.        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sys

!{sys.executable} -m pip install -q \
    torch \
    datasets \
    sacrebleu \
    nltk \
    pandas \
    matplotlib \
    seaborn \
    tqdm \
    huggingface_hub \
    scikit-learn \
    transformers \
    sentencepiece \
    accelerate \
    sqlparse

# transformers + sentencepiece: necessários para fine-tuning com T5
# accelerate: otimiza treino em GPU com transformers
# scikit-learn: usado para train_test_split estratificado
print('✅ Dependências instaladas.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C2 — IMPORTS                                                            ║
# ║  Importações organizadas em blocos por categoria:                        ║
# ║  stdlib → dados → PyTorch → HuggingFace → NLP → avaliação                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os
import re
import json
import random
import unicodedata
import warnings
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# HuggingFace
from datasets import load_dataset, get_dataset_split_names, DatasetDict

# NLP
import nltk
from nltk.tokenize import word_tokenize
from sacrebleu.metrics import BLEU

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print(f'PyTorch {torch.__version__} | CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C3 — REPRODUTIBILIDADE, DEVICE E HIPERPARÂMETROS                        ║
# ║                                                                          ║
# ║  SEED fixa todos os geradores aleatórios (CPU e GPU) para resultados     ║
# ║  reprodutíveis entre execuções.                                          ║
# ║                                                                          ║
# ║  CFG centraliza todos os hiperparâmetros em um único dicionário,         ║
# ║  facilitando experimentos e documentação.                                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True   # garante determinismo na GPU
    torch.backends.cudnn.benchmark     = False  # desativa otimizações não-determinísticas

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

# ── Tokens especiais (ordem fixa: PAD=0, SOS=1, EOS=2, UNK=3) ─────────────
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

# ── Hiperparâmetros ────────────────────────────────────────────────────────
CFG = {
    # Dados
    'max_src_len'    : 60,    # comprimento máximo da pergunta (tokens)
    'max_tgt_len'    : 100,   # aumentado para suportar SQLs com WITH/subqueries
    'min_freq'       : 3,     # frequência mínima para entrar no vocabulário
    'val_split'      : 0.15,  # fração de validação quando não há split nativo
    'test_split'     : 0.15,  # fração de teste

    # Modelo Seq2Seq (LSTM)
    'embed_dim'      : 256,   # dimensão dos embeddings
    'hidden_dim'     : 256,   # dimensão das células LSTM
    'num_layers'     : 2,     # camadas LSTM
    'dropout'        : 0.3,
    'bidirectional'  : True,  # encoder bidirecional

    # Treinamento
    'batch_size'     : 64,
    'learning_rate'  : 3e-4,
    'num_epochs'     : 50,
    'clip_grad'      : 1.0,
    'teacher_forcing': 0.5,

    # Early Stopping
    'patience'       : 8,
    'lr_factor'      : 0.5,
    'lr_patience'    : 3,

    # Modo de treinamento comparativo
    # 'pt'   → só português
    # 'en'   → só inglês
    # 'both' → ambos (treina dois modelos separados para comparação)
    'lang_mode'      : 'both',

    # Fine-Tuning (T5)
    'ft_model_name'  : 'mrm8488/t5-base-finetuned-wikiSQL',  # T5 já pré-treinado em SQL
    'ft_max_src_len' : 256,   # maior pois inclui schema do banco
    'ft_max_tgt_len' : 128,
    'ft_batch_size'  : 16,
    'ft_epochs'      : 5,
    'ft_lr'          : 5e-5,

    # I/O
    'checkpoint_dir' : './checkpoints',
    'vocab_dir'      : './checkpoints/vocabs',
}

Path(CFG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
Path(CFG['vocab_dir']).mkdir(parents=True, exist_ok=True)
print('✅ Configurações carregadas.')
print(f"   Modo de treinamento: {CFG['lang_mode'].upper()}")

# ── CFG específico para ALL ──────────────────
CFG_ALL_OVERRIDE = {
    **CFG,
    'learning_rate': 1e-4,   # reduzir de 3e-4 para 1e-4 no ALL pós-warmup
}

# ── CFG específico para EN ──────────────────
CFG_EN_OVERRIDE = {
    **CFG,
    'dropout'        : 0.5,   # era 0.3 — mais regularização
    'learning_rate'  : 1e-4,  # era 3e-4 — convergência mais lenta e estável
    'teacher_forcing': 0.7,   # era 0.5 — mais supervisão no início
}

print('── Configurações por modelo ─────────────────────────────────────')
print(f'  ALL  │ dropout={CFG["dropout"]}  │ lr={CFG["learning_rate"]:.0e}'
      f'  │ patience={CFG["patience"]}  │ tf={CFG["teacher_forcing"]}')
print(f'  EN   │ dropout={CFG_EN_OVERRIDE["dropout"]}  │ lr={CFG_EN_OVERRIDE["learning_rate"]:.0e}'
      f'  │ patience={CFG_EN_OVERRIDE["patience"]}  │ tf={CFG_EN_OVERRIDE["teacher_forcing"]}')
print(f'  PT   │ dropout={CFG["dropout"]}  │ lr={CFG["learning_rate"]:.0e}'
      f'  │ patience={CFG["patience"]}  │ tf={CFG["teacher_forcing"]}')
print()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C4 — INTROSPECÇÃO E CARREGAMENTO DOS DATASETS (PT e EN)                 ║
# ║                                                                          ║
# ║  Mapa de datasets utilizados:                                            ║
# ║  ┌─────────────────────────────────┬──────────────────────────────────┐  ║
# ║  │ xlangai/spider (EN)             │ train(7000) + validation(1034)   │  ║
# ║  │ Boakpe/spider-test-portuguese   │ test(2147) → splits gerados      │  ║
# ║  │ Marchanjo/spider-pt             │ IGNORADO (arquivos corrompidos)  │  ║
# ║  └─────────────────────────────────┴──────────────────────────────────┘  ║
# ║                                                                          ║
# ║  A função inspect_dataset usa get_dataset_split_names para verificar     ║
# ║  em tempo de execução quais splits existem, sem assumir nada.            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from datasets import load_dataset, get_dataset_split_names

DATASETS_CONFIG = {
    'en': {
        'hf_id'  : 'xlangai/spider',
        'splits' : ['train', 'validation'],
        'q_col'  : 'question',
        'sql_col': 'query',
        'lang'   : 'en',
    },
    'pt': {
        'hf_id'  : 'Boakpe/spider-test-portuguese',
        'splits' : ['test'],
        'q_col'  : 'question',
        'sql_col': 'query',
        'lang'   : 'pt',
    },
}

def inspect_dataset(hf_id: str) -> List[str]:
    """Consulta o HuggingFace Hub para listar splits disponíveis."""
    try:
        splits = get_dataset_split_names(hf_id)
        print(f'  ✅ [{hf_id}] splits: {splits}')
        return splits
    except Exception as e:
        print(f'  ⚠️  [{hf_id}] erro na introspecção: {e}')
        return []

print('🔍 Inspecionando splits disponíveis no HuggingFace Hub...\n')
for lang, cfg in DATASETS_CONFIG.items():
    found = inspect_dataset(cfg['hf_id'])
    if found:
        DATASETS_CONFIG[lang]['splits'] = found

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C5 — CARREGAMENTO E CONVERSÃO PARA DATAFRAME                            ║
# ║                                                                          ║
# ║  PROBLEMA: Boakpe/spider-test-portuguese tem coluna 'sql' com tipos      ║
# ║  mistos (list e str na mesma coluna), causando ArrowInvalid no           ║
# ║  pipeline padrão do HuggingFace datasets.                                ║
# ║                                                                          ║
# ║  SOLUÇÃO: load_dataset com features=None falha → usar download manual    ║
# ║  via huggingface_hub.hf_hub_download + pd.read_json diretamente,         ║
# ║  que tolera tipos mistos. A coluna 'query' é extraída da coluna          ║
# ║  'question' (pergunta PT) e 'query' ou 'sql' (SQL).                      ║
# ║                                                                          ║
# ║  Fallback em 3 camadas:                                                  ║
# ║    1. load_dataset padrão (funciona em alguns ambientes)                 ║
# ║    2. hf_hub_download + pd.read_json (contorna o Arrow)                  ║
# ║    3. requests direto à API raw do HuggingFace                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import requests
from huggingface_hub import hf_hub_download

PT_HF_ID   = 'Boakpe/spider-test-portuguese'
PT_FILE    = 'test_pt-br.json'   # arquivo real no repositório


def load_pt_spider_robust() -> pd.DataFrame:
    """
    Carrega o Spider PT contornando o bug de tipos mistos na coluna 'sql'.
    Tenta 3 estratégias em ordem crescente de robustez.
    """

    # ── Estratégia 1: pipeline padrão HuggingFace ────────────────────────
    print('  Tentativa 1: load_dataset padrão...')
    try:
        ds = load_dataset(PT_HF_ID, split='test')
        df = pd.DataFrame({
            'question': ds['question'],
            'query'   : ds['query'],
            'split'   : 'test',
            'lang'    : 'pt',
            'source'  : PT_HF_ID,
        })
        print(f'  ✅ Estratégia 1 OK: {len(df):,} registros')
        return df
    except Exception as e:
        print(f'  ⚠️  Estratégia 1 falhou: {type(e).__name__}')

    # ── Estratégia 2: download direto + pd.read_json ──────────────────────
    print('  Tentativa 2: hf_hub_download + pd.read_json...')
    try:
        local_path = hf_hub_download(
            repo_id   = PT_HF_ID,
            filename  = PT_FILE,
            repo_type = 'dataset',
        )
        print(f'  Arquivo baixado: {local_path}')

        # pd.read_json tolera tipos mistos — não usa Arrow
        raw = pd.read_json(local_path)
        print(f'  Colunas disponíveis: {raw.columns.tolist()}')

        # Identificar a coluna com a query SQL
        # O dataset pode ter 'query' ou 'sql' dependendo da versão
        sql_col = None
        for candidate in ['query', 'sql', 'SQL', 'Query']:
            if candidate in raw.columns:
                sql_col = candidate
                break

        if sql_col is None:
            raise ValueError(
                f'Nenhuma coluna SQL encontrada. '
                f'Colunas disponíveis: {raw.columns.tolist()}'
            )

        # A coluna sql pode ser lista (estrutura Spider) ou string
        # Se for lista, pegamos o primeiro elemento (query principal)
        def extract_query(val):
            if isinstance(val, list):
                # Pode ser lista de strings ou lista de dicts
                if len(val) == 0:
                    return ''
                first = val[0]
                if isinstance(first, dict):
                    # Spider format: {'query': '...', 'query_toks': [...]}
                    return str(first.get('query', first.get('SQL', '')))
                return str(first)
            return str(val) if val is not None else ''

        queries = raw[sql_col].apply(extract_query)

        # Identificar coluna de pergunta
        q_col = None
        for candidate in ['question', 'Question', 'pergunta', 'query_pt']:
            if candidate in raw.columns:
                q_col = candidate
                break
        if q_col is None:
            raise ValueError(
                f'Coluna de pergunta não encontrada. '
                f'Colunas: {raw.columns.tolist()}'
            )

        df = pd.DataFrame({
            'question': raw[q_col].astype(str),
            'query'   : queries,
            'split'   : 'test',
            'lang'    : 'pt',
            'source'  : PT_HF_ID,
        })
        # Remover linhas com query vazia (artefatos da leitura)
        df = df[df['query'].str.strip().str.len() > 5].reset_index(drop=True)
        print(f'  ✅ Estratégia 2 OK: {len(df):,} registros')
        return df

    except Exception as e:
        print(f'  ⚠️  Estratégia 2 falhou: {type(e).__name__}: {e}')

    # ── Estratégia 3: requests direto à API raw do HuggingFace ───────────
    print('  Tentativa 3: requests para API raw HuggingFace...')
    try:
        url = (
            f'https://huggingface.co/datasets/{PT_HF_ID}'
            f'/resolve/main/{PT_FILE}'
        )
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()

        import io
        raw = pd.read_json(io.StringIO(resp.text))
        print(f'  Colunas: {raw.columns.tolist()}')

        # Mesma lógica de extração da estratégia 2
        sql_col = next(
            (c for c in ['query', 'sql', 'SQL'] if c in raw.columns), None
        )
        q_col   = next(
            (c for c in ['question', 'pergunta', 'Question'] if c in raw.columns), None
        )
        if not sql_col or not q_col:
            raise ValueError(
                f'Colunas não encontradas. Disponíveis: {raw.columns.tolist()}'
            )

        def extract_query(val):
            if isinstance(val, list):
                if not val:
                    return ''
                first = val[0]
                return str(first.get('query', first)) if isinstance(first, dict) else str(first)
            return str(val) if val is not None else ''

        df = pd.DataFrame({
            'question': raw[q_col].astype(str),
            'query'   : raw[sql_col].apply(extract_query),
            'split'   : 'test',
            'lang'    : 'pt',
            'source'  : PT_HF_ID,
        })
        df = df[df['query'].str.strip().str.len() > 5].reset_index(drop=True)
        print(f'  ✅ Estratégia 3 OK: {len(df):,} registros')
        return df

    except Exception as e:
        print(f'  ❌ Estratégia 3 falhou: {type(e).__name__}: {e}')

    # ── Nenhuma estratégia funcionou — retornar DataFrame vazio ──────────
    print('\n  ❌ Todas as estratégias falharam.')
    print('     O projeto continuará apenas com Spider EN.')
    print('     Para usar PT, adicione manualmente o arquivo test_pt-br.json')
    print('     na pasta /content/ e reexecute esta célula.')
    return pd.DataFrame(columns=['question', 'query', 'split', 'lang', 'source'])


def load_hf_to_df(hf_id: str, splits: List[str],
                  q_col: str, sql_col: str, lang: str) -> pd.DataFrame:
    """Carregamento padrão para datasets sem problema de tipos mistos."""
    frames = []
    for split in splits:
        try:
            ds = load_dataset(hf_id, split=split)
            available_cols = ds.column_names
            if q_col not in available_cols or sql_col not in available_cols:
                print(f'  ⚠️  [{hf_id}/{split}] colunas não encontradas.')
                print(f'       Disponíveis: {available_cols}')
                continue
            df = pd.DataFrame({
                'question': ds[q_col],
                'query'   : ds[sql_col],
                'split'   : split,
                'lang'    : lang,
                'source'  : hf_id,
            })
            frames.append(df)
            print(f'  ✅ [{hf_id}] split="{split}" → {len(df):,} registros')
        except Exception as e:
            print(f'  ❌ [{hf_id}] split="{split}" falhou: {e}')

    if not frames:
        return pd.DataFrame(columns=['question', 'query', 'split', 'lang', 'source'])
    return pd.concat(frames, ignore_index=True)


print('📥 Carregando Spider EN...')
df_en = load_hf_to_df(
    hf_id   = DATASETS_CONFIG['en']['hf_id'],
    splits  = DATASETS_CONFIG['en']['splits'],
    q_col   = DATASETS_CONFIG['en']['q_col'],
    sql_col = DATASETS_CONFIG['en']['sql_col'],
    lang    = 'en',
)

print('\n📥 Carregando Spider PT (com fallback robusto)...')
df_pt = load_pt_spider_robust()

print(f'\n── Resumo ────────────────────────────────────────────────────')
print(f'Spider EN: {len(df_en):,} | splits: {df_en["split"].unique().tolist()}')
if len(df_pt) > 0:
    print(f'Spider PT: {len(df_pt):,} | splits: {df_pt["split"].unique().tolist()}')
    # Diagnóstico rápido: mostrar amostra das queries extraídas
    print('\nAmostra das queries PT extraídas:')
    for _, row in df_pt.head(3).iterrows():
        print(f'  Q: {row["question"][:60]}')
        print(f'  S: {row["query"][:60]}')
        print()
else:
    print('Spider PT: 0 registros — apenas EN será usado no treinamento.')
    print('⚠️  Modelos PT e ALL serão treinados sem dados em português.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C5B — GERAÇÃO DE SPLITS PARA O SPIDER PT                                ║
# ║                                                                          ║
# ║  O Spider PT contém apenas split 'test'. Para treinar e validar,         ║
# ║  geramos splits train/val/test de forma reproducível usando SEED.        ║
# ║  Proporção: 70% train | 15% val | 15% test                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
def create_splits(df: pd.DataFrame,
                  train_r: float = 0.70,
                  val_r:   float = 0.15,
                  seed: int = SEED) -> pd.DataFrame:
    """Gera train/val/test a partir de um DataFrame sem splits."""
    if len(df) < 10:
        print(f"⚠️ Dataset muito pequeno ({len(df)} amostras). Pulando split.")
        df['split'] = 'train'
        return df

    df = df.copy().reset_index(drop=True)
    # Primeira divisão: treino vs (val+test)
    idx_train, idx_temp = train_test_split(
        df.index, test_size=1 - train_r, random_state=seed
    )
    # Segunda divisão: val vs test (50/50 do restante)
    idx_val, idx_test = train_test_split(
        idx_temp, test_size=0.5, random_state=seed
    )
    df['split'] = 'train'
    df.loc[idx_val,  'split'] = 'validation'
    df.loc[idx_test, 'split'] = 'test'
    return df


# Aplicar apenas se o PT tiver só split 'test'
if len(df_pt['split'].unique()) == 1 and df_pt['split'].iloc[0] == 'test':
    print('⚙️  Spider PT: apenas "test" disponível. Gerando splits (70/15/15)...')
    df_pt = create_splits(df_pt)

print('\nSplits Spider PT:')
print(df_pt['split'].value_counts().to_string())

In [ ]:
!git clone https://github.com/C4AI/Integrating-Question-Answering-and-Text-to-SQL-in-Portuguese.git

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C6/C7 — CARREGAMENTO OPCIONAL DO DATASET C4AI                           ║
# ║                                                                          ║
# ║  O C4AI contém pares PT extraídos de notebooks Jupyter do repositório    ║
# ║  da USP. A extração usa regex robusto que captura múltiplos tipos        ║
# ║  de comando SQL (SELECT, WITH, INSERT, etc.).                            ║
# ║                                                                          ║
# ║  Para usar: clone o repositório antes de executar esta célula:           ║
# ║  !git clone https://github.com/C4AI/Integrating-Question-Answering-      ║
# ║             and-Text-to-SQL-in-Portuguese.git                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
C4AI_PATH = Path('./Integrating-Question-Answering-and-Text-to-SQL-in-Portuguese')

SQL_PATTERN = re.compile(
    r'(?:Question|Pergunta):\s*(.+?)\s*'
    r'(?:Query|SQL):\s*'
    r'((?:SELECT|WITH|INSERT|UPDATE|DELETE).+?)'
    r'(?=\n(?:Question|Pergunta):|\Z)',
    re.DOTALL | re.IGNORECASE
)

def extract_c4ai_pairs(repo_path: Path) -> pd.DataFrame:
    """Extrai pares question/query de notebooks Jupyter do C4AI."""
    records = []
    notebooks = list(repo_path.rglob('*.ipynb'))
    print(f'  📓 {len(notebooks)} notebooks encontrados')

    for f in tqdm(notebooks, desc='Lendo notebooks'):
        try:
            with open(f, encoding='utf-8') as fp:
                nb = json.load(fp)
        except (json.JSONDecodeError, UnicodeDecodeError) as e:
            print(f'  ⚠️  {f.name}: {e}')
            continue

        for cell in nb.get('cells', []):
            for output in cell.get('outputs', []):
                text = ''
                if 'text' in output:
                    text += ''.join(output['text'])
                elif 'data' in output:
                    text += str(output['data'])

                for question, query in SQL_PATTERN.findall(text):
                    question = question.strip().split('\n')[0].strip()
                    query    = re.split(
                        r'\n.*(?:New request|nova pergunta)',
                        query, flags=re.IGNORECASE
                    )[0].strip()
                    #if len(question) > 5 and len(query) > 10:
                    if question and query:
                        records.append({
                            'question': question,
                            'query'   : query,
                            'split'   : 'raw',
                            'lang'    : 'pt',
                            'source'  : 'c4ai',
                        })

    if not records:
        return pd.DataFrame(columns=['question','query','split','lang','source'])

    df = pd.DataFrame(records)
    df = df.drop_duplicates(subset=['question','query']).reset_index(drop=True)
    df = create_splits(df)  # gerar splits
    print(f'  ✅ {len(df):,} pares extraídos e limpos')
    return df


if C4AI_PATH.exists():
    print('📥 Extraindo pares do C4AI...')
    df_c4ai = extract_c4ai_pairs(C4AI_PATH)
else:
    print('ℹ️  C4AI não encontrado — usando apenas Spider PT e EN.')
    print(f'   Para usar: clone em {C4AI_PATH}')
    df_c4ai = pd.DataFrame(columns=['question','query','split','lang','source'])

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C8 — CONSOLIDAÇÃO, LIMPEZA E CRIAÇÃO DE TEST PARA EN                    ║
# ║                                                                          ║
# ║  PROBLEMA IDENTIFICADO: EN não possuía split 'test', impossibilitando    ║
# ║  comparação justa EN vs PT na avaliação final.                           ║
# ║                                                                          ║
# ║  SOLUÇÃO: 50% do validation EN é separado como test EN, mantendo         ║
# ║  proporção representativa.                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
frames = [df_en, df_pt]
if not df_c4ai.empty:
    frames.append(df_c4ai)

df_all = pd.concat(frames, ignore_index=True)

# Limpeza global
df_all = df_all.dropna(subset=['question', 'query'])
df_all = df_all[df_all['question'].str.strip().str.len() > 5]
df_all = df_all[df_all['query'].str.strip().str.len() > 10]
df_all = df_all.drop_duplicates(subset=['question', 'query']).reset_index(drop=True)
df_all['query'] = df_all['query'].str.strip().str.upper()

# ── Criar split test para EN (50% do validation EN) ──────────────────────
en_val_mask = (df_all['lang'] == 'en') & (df_all['split'] == 'validation')
en_val_idx  = df_all[en_val_mask].index.tolist()

idx_en_val_keep, idx_en_test = train_test_split(
    en_val_idx, test_size=0.5, random_state=SEED
)
df_all.loc[idx_en_test, 'split'] = 'test'

print('══ Dataset Consolidado ══════════════════════════')
print(f'Total de pares: {len(df_all):,}\n')
print('Por fonte:')
print(df_all['source'].value_counts().to_string())
print('\nPor idioma:')
print(df_all['lang'].value_counts().to_string())
print('\nPor split:')
print(df_all['split'].value_counts().to_string())
print('\nPor idioma × split (verificar se EN agora tem test):')
print(df_all.groupby(['lang','split']).size().unstack(fill_value=0).to_string())

## 📊 PARTE 2 — Análise Exploratória (EDA)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C9 — EDA: DISTRIBUIÇÃO DE COMPRIMENTOS                                  ║
# ║                                                                          ║
# ║  IMPORTANTE: Calculamos comprimentos por split de espaço aqui (EDA       ║
# ║  inicial). Os comprimentos reais de tokens serão recalculados após       ║
# ║  a tokenização (C12). Separamos por idioma para comparação PT vs EN.     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from pathlib import Path

# Criar diretório para salvar gráficos
output_dir = Path("./plots")
output_dir.mkdir(parents=True, exist_ok=True)

df_all['q_len_raw']   = df_all['question'].str.split().str.len()
df_all['sql_len_raw'] = df_all['query'].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (lang, label) in enumerate([('en','Inglês'), ('pt','Português')]):
    sub = df_all[df_all['lang'] == lang]

    for col, (metric, title, lim_key, color) in enumerate([
        ('q_len_raw',   'Perguntas',   'max_src_len', 'steelblue'),
        ('sql_len_raw', 'Queries SQL', 'max_tgt_len', 'seagreen'),
    ]):
        ax  = axes[row, col]
        med = sub[metric].median()
        p95 = sub[metric].quantile(0.95)

        ax.hist(sub[metric], bins=40, color=color, edgecolor='white', alpha=0.8)
        ax.axvline(med, color='red',    linestyle='--', label=f'Mediana={med:.0f}')
        ax.axvline(p95, color='orange', linestyle=':',  label=f'P95={p95:.0f}')
        ax.axvline(CFG[lim_key], color='black', linestyle='-.',
                   label=f'Limite CFG={CFG[lim_key]}')
        ax.set_title(f'[{label}] {title} (palavras)')
        ax.set_xlabel('Nº de palavras')
        ax.legend(fontsize=8)

plt.suptitle('EDA: Distribuição de Comprimentos por Idioma', fontsize=14, y=1.01)
plt.tight_layout()

# 🔽 SALVAR FIGURA
file_path_png = output_dir / "eda_comprimentos.png"

plt.savefig(file_path_png, dpi=300, bbox_inches='tight')

plt.show()

print(f"Gráfico salvo em: {file_path_png}")

print('── Estatísticas por idioma ──────────────────────')
print(df_all.groupby('lang')[['q_len_raw','sql_len_raw']].describe().round(1).to_string())

print('\n── % acima dos limites do CFG ───────────────────')
for lang in ['en', 'pt']:
    sub = df_all[df_all['lang'] == lang]
    pq  = (sub['q_len_raw']   > CFG['max_src_len']).mean() * 100
    ps  = (sub['sql_len_raw'] > CFG['max_tgt_len']).mean() * 100
    print(f'  [{lang}] Perguntas acima do limite: {pq:.1f}%')
    print(f'  [{lang}] Queries acima do limite:   {ps:.1f}%')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C10 — EDA: FREQUÊNCIA DE CLÁUSULAS SQL                                  ║
# ║                                                                          ║
# ║  Mostra quais cláusulas SQL são mais comuns em cada idioma.              ║
# ║  Bigrams (GROUP BY, ORDER BY) são normalizados antes da busca.           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from pathlib import Path

# Criar diretório para salvar gráficos
output_dir = Path("./plots")
output_dir.mkdir(parents=True, exist_ok=True)

SQL_KEYWORDS = [
    'SELECT', 'FROM', 'WHERE', 'JOIN', 'LEFT JOIN', 'INNER JOIN',
    'GROUP BY', 'ORDER BY', 'HAVING', 'LIMIT', 'UNION',
    'DISTINCT', 'COUNT', 'SUM', 'AVG', 'MAX', 'MIN',
]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

kw_data = {}
for ax, (lang, label) in zip(axes, [('en','Inglês'), ('pt','Português')]):
    sub  = df_all[df_all['lang'] == lang]
    norm = sub['query'].str.upper().str.replace(r'\s+', ' ', regex=True)

    counts = {
        kw: norm.str.contains(re.escape(kw)).sum()
        for kw in SQL_KEYWORDS
    }
    kw_data[lang] = counts
    series = pd.Series(counts).sort_values(ascending=True)

    bars = ax.barh(series.index, series.values, color='coral', edgecolor='white')
    ax.set_title(f'[{label}] Frequência de Cláusulas SQL (n={len(sub):,})')
    ax.set_xlabel('Ocorrências')

    for bar, val in zip(bars, series.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=8)

plt.tight_layout()

# 🔽 SALVAR FIGURA
file_path_png = output_dir / "eda_frequencia_sql.png"

plt.savefig(file_path_png, dpi=300, bbox_inches='tight')

plt.show()

print(f"Gráfico salvo em: {file_path_png}")

# Tabela comparativa EN vs PT
kw_df = pd.DataFrame(kw_data).fillna(0).astype(int)
kw_df['diff_%'] = ((kw_df['en'] - kw_df['pt']) / kw_df['en'].replace(0,1) * 100).round(1)

print('\n── Comparação EN vs PT ──────────────────────────')
print(kw_df.sort_values('en', ascending=False).to_string())

## 🔤 PARTE 3 — Pré-processamento e Vocabulário

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C11 — NORMALIZAÇÃO E TOKENIZAÇÃO                                        ║
# ║                                                                          ║
# ║  BUG CORRIGIDO: operador >= era separado em > e = pela regex manual.     ║
# ║  SOLUÇÃO: sqlparse — tokenizador SQL dedicado que trata operadores       ║
# ║  compostos (>=, <=, <>, !=), strings literais e palavras-chave           ║
# ║  SQL corretamente, sem regex frágil.                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sqlparse
import sqlparse.tokens as T


def normalize_text(text: str) -> str:
    """Normalização básica: unicode NFC + colapsar espaços."""
    text = str(text).strip()
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\s+', ' ', text)
    return text


def tokenize_question(text: str, lang: str = 'portuguese') -> List[str]:
    """
    Tokeniza pergunta em linguagem natural.
    lang: 'portuguese' para PT, 'english' para EN.
    """
    text = normalize_text(text.lower())
    try:
        tokens = word_tokenize(text, language=lang)
    except LookupError:
        tokens = text.split()
    return tokens


def tokenize_sql(query: str) -> List[str]:
    """
    Tokeniza SQL usando sqlparse.
    Garante que >= <= <> != sejam tokens únicos (bug corrigido).
    Strings literais ('valor') são preservadas como token único.
    Whitespace, newlines e comentários são descartados.
    """
    query   = normalize_text(query.upper())
    parsed  = sqlparse.parse(query)
    if not parsed:
        return query.split()

    # Tipos de token a ignorar
    IGNORE = {
        T.Text.Whitespace,
        T.Text.Whitespace.Newline,
        T.Newline,
        T.Comment.Single,
        T.Comment.Multiline,
        T.Comment,
    }

    tokens = []
    for token in parsed[0].flatten():
        if token.ttype in IGNORE:
            continue
        val = token.value.strip()
        if val:
            tokens.append(val)
    return tokens


# ── Verificação dos operadores compostos ──────────────────────────────────
print('── Verificação da tokenização SQL (operadores compostos) ──\n')

verification_cases = [
    {
        'lang': 'pt',
        'q'  : "Quais alunos tiraram nota acima de 7 na disciplina de Cálculo?",
        'sql': "SELECT nome FROM alunos WHERE nota > 7 AND disciplina = 'Calculo';",
        'ops': [],
    },
    {
        'lang': 'en',
        'q'  : "What students scored at least 7 in Calculus?",
        'sql': "SELECT name FROM students WHERE score >= 7 AND subject = 'Calculus';",
        'ops': ['>='],
    },
    {
        'lang': 'en',
        'q'  : "Find departments where average salary is not equal to 5000",
        'sql': "SELECT dept FROM emp GROUP BY dept HAVING AVG(salary) <> 5000;",
        'ops': ['<>'],
    },
    {
        'lang': 'en',
        'q'  : "List employees with salary below or equal to 3000",
        'sql': "SELECT name FROM emp WHERE salary <= 3000 AND dept != 'HR';",
        'ops': ['<=', '!='],
    },
]

all_ok = True
for case in verification_cases:
    nltk_lang = 'portuguese' if case['lang'] == 'pt' else 'english'
    toks_sql  = tokenize_sql(case['sql'])
    toks_q    = tokenize_question(case['q'], lang=nltk_lang)

    print(f"[{case['lang'].upper()}] SQL: {case['sql']}")
    print(f"  Tokens SQL: {toks_sql}")

    for op in case['ops']:
        if op in case['sql']:
            ok = op in toks_sql
            status = f'✅ {op} preservado' if ok else f'❌ {op} SEPARADO — bug persiste'
            if not ok:
                all_ok = False
            print(f"  Operador {op}: {status}")
    print()

print(f"{'✅ Todos os operadores compostos OK' if all_ok else '❌ Ainda há operadores sendo separados'}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C12 — TOKENIZAÇÃO DO DATASET COMPLETO                                   ║
# ║                                                                          ║
# ║  Usa tokenize_sql corrigido com sqlparse (definido em C11).              ║
# ║  Cada linha usa o tokenizador correto para seu idioma.                   ║
# ║  Após tokenizar, filtramos pares que excedem os limites do CFG.          ║
# ║  Inclui verificação automática do >= no dataset real para confirmar      ║
# ║  que o fix foi aplicado corretamente sobre os dados de produção.         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
def tokenize_row_question(row) -> List[str]:
    lang_nltk = 'portuguese' if row['lang'] == 'pt' else 'english'
    return tokenize_question(row['question'], lang=lang_nltk)


print('🔤 Tokenizando perguntas...')
tqdm.pandas(desc='  Perguntas')
df_all['q_tokens'] = df_all.progress_apply(tokenize_row_question, axis=1)

print('🔤 Tokenizando queries SQL...')
tqdm.pandas(desc='  Queries SQL')
df_all['sql_tokens'] = df_all['query'].progress_apply(tokenize_sql)

# Recalcular comprimentos com tokens reais (pós-correção)
df_all['q_len']   = df_all['q_tokens'].apply(len)
df_all['sql_len'] = df_all['sql_tokens'].apply(len)

# ── Verificação dos operadores compostos no dataset real ──────────────────
print('\n── Verificação operadores compostos no dataset real ────────')
ops_to_check = ['>=', '<=', '<>', '!=']
for op in ops_to_check:
    subset = df_all[df_all['query'].str.contains(re.escape(op), na=False)]
    if len(subset) == 0:
        print(f'  {op}: não encontrado no dataset')
        continue
    sample = subset.head(2)
    broken = 0
    for _, row in sample.iterrows():
        toks = row['sql_tokens']
        if op not in toks:
            broken += 1
    status = '✅ OK' if broken == 0 else f'❌ separado em {broken}/2 amostras'
    print(f'  {op}: {len(subset):,} ocorrências no dataset — {status}')

# ── Filtrar por comprimento máximo ────────────────────────────────────────
before = len(df_all)
mask   = (
    (df_all['q_len']   <= CFG['max_src_len']) &
    (df_all['sql_len'] <= CFG['max_tgt_len'])
)
df_all  = df_all[mask].reset_index(drop=True)
after   = len(df_all)
removed = before - after

print(f'\nPares removidos por exceder comprimento: {removed:,} ({removed/before*100:.1f}%)')
print(f'Pares restantes: {after:,}')
print('\nDistribuição final por idioma × split:')
print(df_all.groupby(['lang','split']).size().unstack(fill_value=0).to_string())

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C13 — VOCABULÁRIOS                                                      ║
# ║                                                                          ║
# ║  IMPORTANTE: vocabulário construído APENAS com dados de TREINO           ║
# ║  para evitar data leakage (vazamento de informação do val/test).         ║
# ║                                                                          ║
# ║  Funcionalidades extras:                                                 ║
# ║  - save/load: persistência em JSON                                       ║
# ║  - coverage: % de tokens do corpus cobertos pelo vocabulário             ║
# ║  - decode com skip_special: para avaliação BLEU                          ║
# ║                                                                          ║
# ║  Além do vocabulário compartilhado (ALL), criamos                        ║
# ║  vocabulários específicos por idioma:                                    ║
# ║                                                                          ║
# ║  src_vocab     → perguntas EN+PT (para modelo ALL)                       ║
# ║  src_vocab_en  → perguntas só EN  (para modelo EN-only)                  ║
# ║  src_vocab_pt  → perguntas só PT  (para modelo PT-only)                  ║
# ║  tgt_vocab     → SQL compartilhado (igual para todos — SQL é SQL)        ║
# ║                                                                          ║
# ║  Vocabulários separados eliminam ruído de tokens de um idioma            ║
# ║  dentro do modelo do outro idioma.                                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
class Vocabulary:
    """Vocabulário bidirecional token ↔ índice com persistência e diagnóstico."""

    SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

    def __init__(self, name: str, min_freq: int = 1):
        self.name      = name
        self.min_freq  = min_freq
        self.token2idx: Dict[str, int] = {}
        self.idx2token: Dict[int, str] = {}
        self._init_specials()

    def _init_specials(self):
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            self.token2idx[tok] = i
            self.idx2token[i]   = tok

    def build_from_corpus(self, token_lists: List[List[str]]):
        counter = Counter(tok for toks in token_lists for tok in toks)
        # most_common: tokens mais frequentes primeiro
        for token, freq in counter.most_common():
            if freq >= self.min_freq and token not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[token] = idx
                self.idx2token[idx]   = token
        print(f'Vocabulário [{self.name}]: {len(self):,} tokens (min_freq={self.min_freq})')

    def encode(self, tokens: List[str],
               max_len: Optional[int] = None) -> List[int]:
        ids = [self.token2idx.get(t, UNK_IDX) for t in tokens]
        return ids[:max_len] if max_len else ids

    def decode(self, indices: List[int],
               skip_special: bool = False) -> List[str]:
        special_ids = {PAD_IDX, SOS_IDX, UNK_IDX}
        tokens = []
        for i in indices:
            if i == EOS_IDX:
                break   # parar no EOS
            if skip_special and i in special_ids:
                continue
            tokens.append(self.idx2token.get(i, UNK_TOKEN))
        return tokens

    def coverage(self, token_lists: List[List[str]]) -> float:
        all_toks = [t for toks in token_lists for t in toks]
        if not all_toks:
            return 0.0
        known = sum(1 for t in all_toks if t in self.token2idx)
        return known / len(all_toks) * 100

    def save(self, path: str):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({'name': self.name, 'min_freq': self.min_freq,
                       'token2idx': self.token2idx}, f,
                      ensure_ascii=False, indent=2)
        print(f"  Vocabulário [{self.name}] salvo em '{path}'")

    @classmethod
    def load(cls, path: str) -> 'Vocabulary':
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        vocab = cls(data['name'], data['min_freq'])
        vocab.token2idx = data['token2idx']
        vocab.idx2token = {int(i): t for t, i in data['token2idx'].items()}
        print(f"  Vocabulário [{vocab.name}] carregado: {len(vocab):,} tokens")
        return vocab

    def __len__(self) -> int:
        return len(self.token2idx)

train_mask    = df_all['split'] == 'train'
train_en_mask = train_mask & (df_all['lang'] == 'en')
train_pt_mask = train_mask & (df_all['lang'] == 'pt')

print(f'Construindo vocabulários...')
print(f'  Treino ALL: {train_mask.sum():,} | EN: {train_en_mask.sum():,} | PT: {train_pt_mask.sum():,}\n')

# Vocabulário compartilhado (modelo ALL)
src_vocab = Vocabulary('source_all', min_freq=CFG['min_freq'])
src_vocab.build_from_corpus(df_all.loc[train_mask, 'q_tokens'].tolist())

# Vocabulário EN-only (para modelo EN sem ruído de tokens PT)
src_vocab_en = Vocabulary('source_en', min_freq=CFG['min_freq'])
src_vocab_en.build_from_corpus(df_all.loc[train_en_mask, 'q_tokens'].tolist())

# Vocabulário PT-only (para modelo PT sem ruído de tokens EN)
src_vocab_pt = Vocabulary('source_pt', min_freq=CFG['min_freq'])
src_vocab_pt.build_from_corpus(df_all.loc[train_pt_mask, 'q_tokens'].tolist())

# Vocabulário SQL — compartilhado para todos (SQL é independente de idioma)
tgt_vocab = Vocabulary('target_sql', min_freq=CFG['min_freq'])
tgt_vocab.build_from_corpus(df_all.loc[train_mask, 'sql_tokens'].tolist())

# ── Diagnóstico de overlap entre vocabulários EN e PT ─────────────────────
toks_en_set = set(src_vocab_en.token2idx.keys()) - set(Vocabulary.SPECIAL_TOKENS)
toks_pt_set = set(src_vocab_pt.token2idx.keys()) - set(Vocabulary.SPECIAL_TOKENS)

shared   = toks_en_set & toks_pt_set
only_en  = toks_en_set - toks_pt_set
only_pt  = toks_pt_set - toks_en_set

print(f'\n── Análise de overlap vocabulário ───────────────────────')
print(f'  Tokens exclusivos EN: {len(only_en):,}')
print(f'  Tokens exclusivos PT: {len(only_pt):,}')
print(f'  Tokens compartilhados EN∩PT: {len(shared):,}')
print(f'  Ruído eliminado no modelo EN: {len(only_pt):,} tokens PT não aparecem mais')
print(f'  Ruído eliminado no modelo PT: {len(only_en):,} tokens EN não aparecem mais')

# Cobertura
print(f'\n── Cobertura dos vocabulários ───────────────────────────')
print(f'  src_vocab    (ALL): {src_vocab.coverage(df_all["q_tokens"].tolist()):.2f}%')
print(f'  src_vocab_en  (EN): {src_vocab_en.coverage(df_all.loc[df_all["lang"]=="en", "q_tokens"].tolist()):.2f}%')
print(f'  src_vocab_pt  (PT): {src_vocab_pt.coverage(df_all.loc[df_all["lang"]=="pt", "q_tokens"].tolist()):.2f}%')
print(f'  tgt_vocab    (SQL): {tgt_vocab.coverage(df_all["sql_tokens"].tolist()):.2f}%')

# Persistir todos os vocabulários
vocab_dir = Path(CFG['vocab_dir'])
src_vocab.save(str(vocab_dir / 'src_vocab_all.json'))
src_vocab_en.save(str(vocab_dir / 'src_vocab_en.json'))
src_vocab_pt.save(str(vocab_dir / 'src_vocab_pt.json'))
tgt_vocab.save(str(vocab_dir / 'tgt_vocab.json'))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C14 — DATALOADERS                                                       ║
# ║  Text2SQLDataset: converte tokens em tensores, adiciona SOS/EOS          ║
# ║  collate_fn: padding dinâmico por batch (sem padding fixo global)        ║
# ║  build_loaders: respeita splits nativos do DataFrame                     ║
# ║                                                                          ║
# ║  CORREÇÃO: loaders EN usam src_vocab_en, loaders PT usam src_vocab_pt.   ║
# ║  Antes, todos usavam src_vocab compartilhado, fazendo o embedding do     ║
# ║  modelo EN carregar ~2k tokens PT irrelevantes.                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class Text2SQLDataset(Dataset):
    """Dataset PyTorch para pares (pergunta → SQL)."""

    def __init__(self, df: pd.DataFrame,
                 src_vocab: Vocabulary,
                 tgt_vocab: Vocabulary,
                 max_src_len: Optional[int] = None,
                 max_tgt_len: Optional[int] = None):
        self.src_vocab   = src_vocab
        self.tgt_vocab   = tgt_vocab
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.src_tokens  = df['q_tokens'].tolist()
        self.tgt_tokens  = df['sql_tokens'].tolist()

    def __len__(self) -> int:
        return len(self.src_tokens)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        src_ids = self.src_vocab.encode(self.src_tokens[idx], self.max_src_len)
        tgt_ids = ([SOS_IDX]
                   + self.tgt_vocab.encode(self.tgt_tokens[idx], self.max_tgt_len)
                   + [EOS_IDX])
        return (torch.tensor(src_ids, dtype=torch.long),
                torch.tensor(tgt_ids, dtype=torch.long))


def collate_fn(
    batch: List[Tuple[torch.Tensor, torch.Tensor]]
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Padding dinâmico — pad só até o maior elemento do batch atual."""
    src_seqs, tgt_seqs = zip(*batch)
    src_padded  = pad_sequence(src_seqs, batch_first=True, padding_value=PAD_IDX)
    tgt_padded  = pad_sequence(tgt_seqs, batch_first=True, padding_value=PAD_IDX)
    src_lengths = torch.tensor([len(s) for s in src_seqs], dtype=torch.long)
    tgt_lengths = torch.tensor([len(t) for t in tgt_seqs], dtype=torch.long)
    return src_padded, tgt_padded, src_lengths, tgt_lengths

def build_loaders(df: pd.DataFrame,
                  src_vocab: Vocabulary,
                  tgt_vocab: Vocabulary,
                  cfg: dict,
                  lang_filter: Optional[str] = None) -> Dict[str, DataLoader]:
    """
    Constrói DataLoaders para cada split respeitando a coluna 'split'.
    lang_filter: 'en' | 'pt' | None (ambos).
    src_vocab deve ser o vocabulário específico do idioma filtrado.
    """
    loaders   = {}
    split_cfg = {
        'train'     : {'shuffle': True,  'drop_last': True},
        'validation': {'shuffle': False, 'drop_last': False},
        'test'      : {'shuffle': False, 'drop_last': False},
    }
    for split, params in split_cfg.items():
        mask = df['split'] == split
        if lang_filter:
            mask &= df['lang'] == lang_filter
        sub = df[mask].reset_index(drop=True)
        if len(sub) == 0:
            print(f'  ⚠️  split="{split}" vazio para lang={lang_filter}')
            continue
        dataset = Text2SQLDataset(
            sub, src_vocab, tgt_vocab,
            max_src_len = cfg['max_src_len'],
            max_tgt_len = cfg['max_tgt_len'],
        )
        loaders[split] = DataLoader(
            dataset,
            batch_size = cfg['batch_size'],
            shuffle    = params['shuffle'],
            drop_last  = params['drop_last'],
            collate_fn = collate_fn,
            num_workers = 0,
            pin_memory  = torch.cuda.is_available(),
        )
        tag = lang_filter or 'all'
        print(f'  [{tag}] {split:12s}: {len(dataset):>5,} amostras | '
              f'{len(loaders[split]):>3} batches')
    return loaders


print('══ Construindo DataLoaders ══════════════════════════════')

# ALL usa vocabulário compartilhado
print('\n▶ Loaders ALL (vocabulário compartilhado EN+PT):')
loaders_all = build_loaders(df_all, src_vocab, tgt_vocab, CFG, lang_filter=None)

# EN usa vocabulário exclusivo EN — sem tokens PT como ruído
print('\n▶ Loaders EN (vocabulário exclusivo EN):')
loaders_en = build_loaders(df_all, src_vocab_en, tgt_vocab, CFG, lang_filter='en')

# PT usa vocabulário exclusivo PT — sem tokens EN como ruído
print('\n▶ Loaders PT (vocabulário exclusivo PT):')
loaders_pt = build_loaders(df_all, src_vocab_pt, tgt_vocab, CFG, lang_filter='pt')

# Aliases para o loop de treino
train_loader = loaders_all.get('train')
val_loader   = loaders_all.get('validation')
test_loader  = loaders_all.get('test')

# ── Verificação de splits por idioma ──────────────────────────────────────
print('\n── Verificação final de splits ──────────────────────────')
for lang in ['en', 'pt']:
    for split in ['train', 'validation', 'test']:
        n      = len(df_all[(df_all['lang'] == lang) & (df_all['split'] == split)])
        status = '✅' if n > 0 else '⚠️  vazio'
        print(f'  [{lang}] {split:12s}: {n:>5,} amostras  {status}')

print('\n✅ DataLoaders prontos.')

## 🏗️ PARTE 4 — Arquitetura Seq2Seq com Atenção de Bahdanau

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C15 — ENCODER BIDIRECIONAL                                              ║
# ║                                                                          ║
# ║  LSTM Bidirecional: processa a sequência em ambas as direções            ║
# ║  → captura contexto anterior E posterior de cada palavra.                ║
# ║                                                                          ║
# ║  Projeção linear: o encoder bidirecional produz hidden de tamanho        ║
# ║  2*hidden_dim. A projeção comprime para hidden_dim para compatibilizar   ║
# ║  com o decoder (que é unidirecional).                                    ║
# ║                                                                          ║
# ║  pack_padded_sequence: ignora tokens de padding durante o cálculo        ║
# ║  da LSTM, tornando o processo mais eficiente e correto.                  ║
# ║                                                                          ║
# ║  CORREÇÃO: _merge substituída por lógica explícita que funciona          ║
# ║  corretamente para qualquer num_layers, não apenas 2.                    ║
# ║  Antes: apenas hidden[-2] e hidden[-1] eram usados, descartando          ║
# ║  camadas intermediárias silenciosamente quando num_layers > 2.           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
class Encoder(nn.Module):
    """
    Codifica a pergunta em linguagem natural em estados ocultos.
    LSTM bidirecional + projeção para compatibilizar com decoder.
    """

    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, num_layers: int, dropout: float,
                 bidirectional: bool = True):
        super().__init__()
        self.hidden_dim    = hidden_dim
        self.num_layers    = num_layers
        self.bidirectional = bidirectional
        self.num_dirs      = 2 if bidirectional else 1

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            dropout       = dropout if num_layers > 1 else 0,
            bidirectional = bidirectional,
        )
        self.dropout = nn.Dropout(dropout)

        # Projeção: comprime saída bidirecional (2*hidden) → hidden
        # Aplicada por camada para suportar qualquer num_layers
        if bidirectional:
            self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
            self.fc_cell   = nn.Linear(hidden_dim * 2, hidden_dim)
            self.fc_output = nn.Linear(hidden_dim * 2, hidden_dim)

    def _merge_state(self, state: torch.Tensor,
                    fc: nn.Linear) -> torch.Tensor:
        """
        Combina direções forward/backward mantendo todas as camadas.

        Entrada:
            (num_layers * 2, batch, hidden_dim)

        Saída:
            (num_layers, batch, hidden_dim)
        """
        num_layers_times_dirs, batch, hidden_dim = state.size()
        num_layers = num_layers_times_dirs // 2

        # Reorganizar explicitamente
        state = state.view(num_layers, 2, batch, hidden_dim)

        fwd = state[:, 0, :, :]
        bwd = state[:, 1, :, :]

        merged = torch.cat([fwd, bwd], dim=2)  # (num_layers, batch, hidden*2)

        # Aplicar FC corretamente
        merged_flat = merged.reshape(num_layers * batch, -1)
        projected   = fc(merged_flat)

        projected = projected.view(num_layers, batch, hidden_dim)

        return torch.tanh(projected)

    def forward(
        self,
        src: torch.Tensor,         # (batch, src_len)
        src_lengths: torch.Tensor,
    ) -> Tuple[torch.Tensor, Tuple]:

        embedded = self.dropout(self.embedding(src))  # (batch, src_len, embed)

        # Garantir que src_lengths não exceda o tamanho real da sequência
        src_lengths = src_lengths.clamp(max=src.size(1))

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, src_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        outputs, (hidden, cell) = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            outputs, batch_first=True
        )
        # outputs: (batch, src_len, hidden_dim * num_dirs)

        if self.bidirectional:
            # Projetar hidden e cell de todas as camadas corretamente
            hidden  = self._merge_state(hidden, self.fc_hidden)
            cell    = self._merge_state(cell,   self.fc_cell)
            # Projetar outputs de (hidden*2) → (hidden)
            outputs = torch.tanh(self.fc_output(outputs))

        return outputs, (hidden, cell)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C16 — ATENÇÃO DE BAHDANAU (Additive Attention)                          ║
# ║                                                                          ║
# ║  Fórmula: score(s, h) = v^T · tanh(W_s·s + W_h·h)                        ║
# ║  onde s = estado do decoder, h = estado do encoder                       ║
# ║                                                                          ║
# ║  MELHORIA CRÍTICA: máscara de padding — tokens PAD no encoder            ║
# ║  recebem peso -inf antes do softmax, resultando em peso ≈ 0.             ║
# ║  Sem isso, o modelo "presta atenção" em tokens que não existem.          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
class BahdanauAttention(nn.Module):
    """Atenção aditiva com máscara de padding."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        self.W_s = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v   = nn.Linear(hidden_dim, 1,          bias=False)

    def forward(
        self,
        decoder_hidden:  torch.Tensor,    # (batch, hidden_dim)
        encoder_outputs: torch.Tensor,    # (batch, src_len, hidden_dim)
        src_mask:        Optional[torch.Tensor] = None,  # (batch, src_len)
    ) -> Tuple[torch.Tensor, torch.Tensor]:

        s = decoder_hidden.unsqueeze(1).expand_as(encoder_outputs)
        energy = self.v(
            torch.tanh(self.W_s(s) + self.W_h(encoder_outputs))
        ).squeeze(-1)  # (batch, src_len)

        # Mascarar posições de padding com -inf para que softmax → 0
        if src_mask is not None:
            energy = energy.masked_fill(src_mask == 0, float('-inf'))

        attn_weights = F.softmax(energy, dim=1)  # (batch, src_len)

        # Vetor de contexto: média ponderada dos estados do encoder
        context = torch.bmm(
            attn_weights.unsqueeze(1), encoder_outputs
        ).squeeze(1)   # (batch, hidden_dim)

        return context, attn_weights

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C17 — DECODER COM ATENÇÃO                                               ║
# ║                                                                          ║
# ║  Gera SQL token a token usando 'input feeding':                          ║
# ║  → o vetor de contexto é concatenado ao embedding antes do LSTM          ║
# ║  → forçando o modelo a considerar a atenção a cada passo                 ║
# ║                                                                          ║
# ║  Projeção final: [output + context + embedding] → vocab                  ║
# ║  Essa concatenação tripla é uma heurística clássica (Luong et al.)       ║
# ║  que melhora a geração ao preservar informação do input original.        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
class Decoder(nn.Module):
    """Decoder auto-regressivo com atenção de Bahdanau e input feeding."""

    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, num_layers: int, dropout: float):
        super().__init__()
        self.attention = BahdanauAttention(hidden_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(
            embed_dim + hidden_dim,
            hidden_dim,
            num_layers = num_layers,
            batch_first = True,
            dropout = dropout if num_layers > 1 else 0,
        )
        # Projeção: [hidden + context + embed] → vocab
        self.fc_out = nn.Linear(hidden_dim * 2 + embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward_step(
        self,
        input_token:     torch.Tensor,     # (batch,)
        hidden:          Tuple,
        encoder_outputs: torch.Tensor,     # (batch, src_len, hidden_dim)
        src_mask:        Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Tuple, torch.Tensor]:

        embedded = self.dropout(
            self.embedding(input_token.unsqueeze(1))
        )  # (batch, 1, embed)

        decoder_hidden_top = hidden[0][-1]   # último layer: (batch, hidden)
        context, attn_weights = self.attention(
            decoder_hidden_top, encoder_outputs, src_mask
        )

        rnn_input = torch.cat(
            [embedded, context.unsqueeze(1)], dim=2
        )  # (batch, 1, embed+hidden)
        output, hidden = self.rnn(rnn_input, hidden)   # (batch, 1, hidden)

        output_sq   = output.squeeze(1)    # (batch, hidden)
        embedded_sq = embedded.squeeze(1)  # (batch, embed)

        logits = self.fc_out(
            torch.cat([output_sq, context, embedded_sq], dim=1)
        )  # (batch, vocab)

        return logits, hidden, attn_weights

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C18 — SEQ2SEQ: ORQUESTRADOR ENCODER + DECODER                           ║
# ║  Teacher Forcing: com probabilidade p, usa o token REAL como próximo     ║
# ║  input do decoder (acelera treinamento). Com prob (1-p), usa o token     ║
# ║  que o modelo prediz (mais realista para inferência).                    ║
# ║                                                                          ║
# ║  src_mask: construída a partir dos tokens PAD na sequência de entrada    ║
# ║  e passada para o mecanismo de atenção.                                  ║
# ║                                                                          ║
# ║  Método translate: inferência pura sem teacher forcing,                  ║
# ║  retorna tokens decodificados como string.                               ║
# ║                                                                          ║
# ║  CORREÇÃO no método translate():                                         ║
# ║  Antes: encode(tokens, self.encoder.rnn.input_size)                      ║
# ║         → input_size = embed_dim = 256 (truncava em 256 tokens)          ║
# ║  Depois: encode(tokens, CFG['max_src_len'])                              ║
# ║         → trunca no limite correto definido no CFG (60 tokens)           ║
# ║                                                                          ║
# ║  CORREÇÃO v2 — anti-loop no translate():                                 ║
# ║  O decoder pode entrar em loop de dois tipos:                            ║
# ║    Tipo 1: token único repetido — FROM FROM FROM FROM                    ║
# ║    Tipo 2: padrão alternado repetido — T1 . T1 . T1 .                    ║
# ║  A verificação de janela deslizante captura ambos sem limitar            ║
# ║  o comprimento legítimo da query gerada.                                 ║
# ║                                                                          ║
# ║  Lógica: se os últimos N tokens formam um padrão que se repete           ║
# ║  K vezes, o decoder está em loop e deve parar.                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class Seq2Seq(nn.Module):
    """Modelo seq2seq completo: encoder bidirecional + decoder com atenção."""

    def __init__(self, encoder: Encoder, decoder: Decoder,
                 src_vocab: Vocabulary, tgt_vocab: Vocabulary,
                 device: torch.device):
        super().__init__()
        self.encoder   = encoder
        self.decoder   = decoder
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.device    = device

    def make_src_mask(self, src: torch.Tensor) -> torch.Tensor:
        return (src != PAD_IDX)

    @staticmethod
    def _detect_loop(pred_ids: List[int],
                     pattern_sizes: Tuple[int, ...] = (1, 2, 3),
                     min_repeats: int = 3) -> bool:
        for psize in pattern_sizes:
            window = psize * min_repeats
            if len(pred_ids) < window:
                continue
            tail    = pred_ids[-window:]
            pattern = tail[:psize]
            if all(tail[i:i+psize] == pattern
                   for i in range(0, window, psize)):
                return True
        return False

    def forward(
        self,
        src: torch.Tensor,
        src_lengths: torch.Tensor,
        tgt: torch.Tensor,
        teacher_forcing_ratio: float = 0.5,
    ) -> torch.Tensor:

        batch_size = src.size(0)
        tgt_len    = tgt.size(1)
        vocab_size = self.decoder.fc_out.out_features

        outputs  = torch.zeros(batch_size, tgt_len, vocab_size).to(self.device)
        src_mask = self.make_src_mask(src).to(self.device)

        encoder_outputs, hidden = self.encoder(src, src_lengths)

        input_token = tgt[:, 0]

        for t in range(1, tgt_len):
            logits, hidden, _ = self.decoder.forward_step(
                input_token, hidden, encoder_outputs, src_mask
            )
            outputs[:, t, :] = logits
            use_teacher = random.random() < teacher_forcing_ratio
            input_token = tgt[:, t] if use_teacher else logits.argmax(dim=1)

        return outputs

    @torch.no_grad()
    def translate(self, question: str, lang: str = 'pt',
                  max_len: int = 100) -> str:
        self.eval()
        nltk_lang = 'portuguese' if lang == 'pt' else 'english'
        tokens    = tokenize_question(question, lang=nltk_lang)

        # comprimento real antes do padding
        true_len = min(len(tokens), CFG['max_src_len'])
        src_ids  = self.src_vocab.encode(tokens, max_len=CFG['max_src_len'])

        src      = torch.tensor([src_ids], dtype=torch.long).to(self.device)
        src_len  = torch.tensor([true_len], dtype=torch.long)
        src_mask = self.make_src_mask(src).to(self.device)

        encoder_outputs, hidden = self.encoder(src, src_len)

        input_token = torch.tensor([SOS_IDX], dtype=torch.long).to(self.device)

        pred_ids = []
        for _ in range(max_len):
            logits, hidden, _ = self.decoder.forward_step(
                input_token, hidden, encoder_outputs, src_mask
            )
            pred_id = logits.argmax(dim=1)

            if pred_id.item() == EOS_IDX:
                break

            pred_ids.append(pred_id.item())

            if self._detect_loop(pred_ids, pattern_sizes=(1, 2, 3), min_repeats=3):
                break

            input_token = pred_id

        return ' '.join(self.tgt_vocab.decode(pred_ids, skip_special=True))

# Instanciar modelo padrão para verificação
encoder = Encoder(
    vocab_size    = len(src_vocab),
    embed_dim     = CFG['embed_dim'],
    hidden_dim    = CFG['hidden_dim'],
    num_layers    = CFG['num_layers'],
    dropout       = CFG['dropout'],
    bidirectional = CFG['bidirectional'],
)
decoder = Decoder(
    vocab_size = len(tgt_vocab),
    embed_dim  = CFG['embed_dim'],
    hidden_dim = CFG['hidden_dim'],
    num_layers = CFG['num_layers'],
    dropout    = CFG['dropout'],
)
model = Seq2Seq(encoder, decoder, src_vocab, tgt_vocab, DEVICE).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Modelo criado | Parâmetros treináveis: {n_params:,}')
print(f'   Encoder: {"Bidirecional" if CFG["bidirectional"] else "Unidirecional"}')
print(f'   Vocab src: {len(src_vocab):,} | Vocab tgt: {len(tgt_vocab):,}')

print('\n── Teste da detecção de loop ────────────────────────────────────')
casos = [
    ([1, 2, 3, 4, 4, 4],          True,  'token único repetido 3x'),
    ([1, 2, 3, 4, 5, 4, 5, 4, 5], True,  'par alternado repetido 3x'),
    ([1, 2, 3, 4, 5, 6],          False, 'sequência normal sem loop'),
    ([1, 2, 1, 2, 1],             False, 'par repetido apenas 2x (abaixo do limite)'),
]
for ids, esperado, descricao in casos:
    resultado = Seq2Seq._detect_loop(ids)
    status = '✅' if resultado == esperado else '❌'
    print(f'  {status} {descricao}: detectado={resultado}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C19 — INICIALIZAÇÃO, OTIMIZADOR E FUNÇÕES DE TREINO                     ║
# ║                                                                          ║
# ║  Inicialização de pesos:                                                 ║
# ║  - Embeddings: distribuição normal pequena (0, 0.01)                     ║
# ║  - Matrizes: Xavier Uniform (estabiliza gradientes na init)              ║
# ║  - Bias: zeros, exceto forget gate da LSTM → 1.0 (previne                ║
# ║    esquecimento prematuro nas primeiras épocas)                          ║
# ║                                                                          ║
# ║  Perplexidade: exp(loss) — mais interpretável que loss bruta.            ║
# ║  BLEU por epoch: avalia qualidade das traduções completas.               ║
# ║                                                                          ║
# ║  CORREÇÃO: adicionado WarmupScheduler para o modelo ALL.                 ║
# ║                                                                          ║
# ║  PROBLEMA IDENTIFICADO: o modelo ALL com lr=3e-4 e label_smoothing       ║
# ║  encontrou um mínimo local na época 1 e nunca mais melhorou.             ║
# ║                                                                          ║
# ║  CAUSA RAIZ: o espaço de embedding EN+PT é ~2x maior que os modelos      ║
# ║  mono-idioma. Com lr alto desde o início, o modelo faz passos grandes    ║
# ║  e cai num mínimo raso imediatamente.                                    ║
# ║                                                                          ║
# ║  SOLUÇÃO — Warmup linear:                                                ║
# ║  As primeiras N épocas usam lr crescente de 0 → lr_target.               ║
# ║  Isso permite que o modelo explore o espaço de parâmetros de forma       ║
# ║  gradual antes de fazer passos grandes.                                  ║
# ║                                                                          ║
# ║  Após o warmup, o ReduceLROnPlateau assume o controle normal.            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
def init_weights(m):
    for name, param in m.named_parameters():
        if 'embedding' in name:
            nn.init.normal_(param.data, mean=0, std=0.01)
        elif 'weight' in name and param.data.dim() >= 2:
            nn.init.xavier_uniform_(param.data)
        elif 'bias' in name:
            nn.init.zeros_(param.data)
            if 'bias_ih' in name or 'bias_hh' in name:
                hidden_size = param.data.shape[0] // 4
                param.data[hidden_size:2*hidden_size].fill_(1.0)

model.apply(init_weights)
print('✅ Pesos inicializados.')

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

optimizer = optim.Adam(model.parameters(), lr=CFG['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience = CFG['lr_patience'],
    factor   = CFG['lr_factor'],
)

class WarmupScheduler:
    """
    Scheduler de warmup linear para as primeiras épocas.

    Durante as primeiras `warmup_epochs` épocas, o LR sobe linearmente
    de `start_lr` até `target_lr`. Após isso, o scheduler principal
    (ReduceLROnPlateau) assume o controle.

    Uso: chamar .step(epoch) antes de cada época de treino.
    """
    def __init__(self, optimizer: optim.Optimizer,
                 warmup_epochs: int,
                 target_lr: float,
                 start_lr: float = 1e-6):
        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.target_lr     = target_lr
        self.start_lr      = start_lr

    def step(self, epoch: int):
        """Ajusta o LR se ainda estiver no período de warmup."""
        if epoch <= self.warmup_epochs:
            # Interpolação linear: start_lr → target_lr
            lr = self.start_lr + (self.target_lr - self.start_lr) * \
                 (epoch / self.warmup_epochs)
            for pg in self.optimizer.param_groups:
                pg['lr'] = lr

    def is_warming_up(self, epoch: int) -> bool:
        return epoch <= self.warmup_epochs

def compute_bleu_sample(model, df_subset: pd.DataFrame,
                         n_samples: int = 50,
                         lang: str = 'pt') -> float:
    """
    Calcula BLEU-1 com smooth para monitoramento durante treino.
    Usa smooth_method='exp' do sacrebleu para evitar zeros em corpus pequenos.
    """
    model.eval()
    if len(df_subset) == 0:
        return 0.0

    sub = df_subset.sample(min(n_samples, len(df_subset)), random_state=SEED)
    refs, hyps = [], []

    for _, row in sub.iterrows():
        try:
            with torch.no_grad():
                pred = model.translate(row['question'], lang=lang)
        except Exception:
            pred = ''

        ref_clean  = row['query'].lower().strip()
        pred_clean = pred.lower().strip()
        if not pred_clean:
            continue  # ignorar predições vazias

        refs.append(ref_clean)    # string direta — corpus_score([refs]) adiciona dimensão
        hyps.append(pred_clean)

    if len(hyps) == 0:
        return 0.0

    try:
        bleu = BLEU(
            tokenize='13a',
            max_ngram_order=1,
            smooth_method='exp',
            effective_order=True
        )
        score = bleu.corpus_score(hyps, [refs]).score
    except Exception:
        score = 0.0

    return score

def train_epoch(model, loader, optimizer, criterion,
                clip: float,
                tgt_vocab_size: int = None) -> Tuple[float, float]:
    """
    tgt_vocab_size: tamanho do vocabulário alvo.
    Se None, infere a partir do fc_out do modelo (correto por construção).
    """
    model.train()
    # Inferir tamanho do vocabulário do próprio modelo — sempre correto
    vocab_size = model.decoder.fc_out.out_features
    total_loss = 0.0
    bar = tqdm(loader, desc='  Treino', leave=False, ncols=90)

    for i, (src, tgt, src_len, _) in enumerate(bar):
        src, tgt, src_len = src.to(DEVICE), tgt.to(DEVICE), src_len.to(DEVICE)

        optimizer.zero_grad()

        output = model(src, src_len, tgt, CFG['teacher_forcing'])

        out_flat = output[:, 1:, :].reshape(-1, vocab_size)
        tgt_flat = tgt[:, 1:].reshape(-1)

        loss = criterion(out_flat, tgt_flat)

        # proteção contra NaN/inf
        if torch.isnan(loss) or torch.isinf(loss):
            print("⚠️ Loss inválido detectado. Pulando batch.")
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        total_loss += loss.item()
        bar.set_postfix({
            'loss': f'{total_loss/(i+1):.4f}',
            'ppl' : f'{np.exp(min(total_loss/(i+1), 10)):.1f}',
        })

    avg_loss = total_loss / len(loader)
    return avg_loss, np.exp(min(avg_loss, 10))


@torch.no_grad()
def evaluate_epoch(model, loader, criterion) -> Tuple[float, float]:
    model.eval()
    vocab_size = model.decoder.fc_out.out_features   # inferir do modelo
    total_loss = 0.0
    bar = tqdm(loader, desc='  Val', leave=False, ncols=90)

    for i, (src, tgt, src_len, _) in enumerate(bar):
        src, tgt, src_len = src.to(DEVICE), tgt.to(DEVICE), src_len.to(DEVICE)

        output = model(src, src_len, tgt, teacher_forcing_ratio=0.0)

        out_flat = output[:, 1:, :].reshape(-1, vocab_size)
        tgt_flat = tgt[:, 1:].reshape(-1)

        loss = criterion(out_flat, tgt_flat)

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        total_loss += loss.item()
        bar.set_postfix({'loss': f'{total_loss/(i+1):.4f}'})

    avg_loss = total_loss / len(loader)
    return avg_loss, np.exp(min(avg_loss, 10))

print('✅ train_epoch e evaluate_epoch corrigidos (vocab_size inferido do modelo).')

print('✅ Funções de treino e WarmupScheduler definidos.')

## 🔁 Parte 5 — Treinamento de LSTM e Comparação PT vs EN

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C20 — FUNÇÃO DE TREINAMENTO                                             ║
# ║                                                                          ║
# ║  A função train_model é genérica e recebe qualquer par de loaders.       ║
# ║  Isso permite reutilizá-la para treinar o modelo com dados EN, PT        ║
# ║  ou ambos, apenas variando os loaders de entrada.                        ║
# ║                                                                          ║
# ║  O checkpoint salvo inclui: estado do modelo, otimizador, scheduler,     ║
# ║  histórico completo e melhor val_loss — permitindo retomar o treino.     ║
# ║                                                                          ║
# ║  ADIÇÃO: parâmetro warmup_epochs — quando > 0, aplica WarmupScheduler    ║
# ║  nas primeiras épocas antes de ceder controle ao ReduceLROnPlateau.      ║
# ║                                                                          ║
# ║  LÓGICA DO WARMUP:                                                       ║
# ║  - Épocas 1..warmup_epochs: lr cresce 0 → lr_target (warmup)             ║
# ║  - O early stopping NÃO conta durante o warmup (patience resetada)       ║
# ║  - Após warmup: ReduceLROnPlateau assume normalmente                     ║
# ║                                                                          ║
# ║  Por que não contar early stopping no warmup?                            ║
# ║  Durante o warmup o lr é artificialmente baixo — a val_loss pode         ║
# ║  oscilar ou não melhorar, mas isso não significa que o modelo parou      ║
# ║  de aprender. Contar paciência aqui causaria early stop prematuro.       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    cfg: dict,
    tag: str = 'model',
    df_val_sample: Optional[pd.DataFrame] = None,
    val_lang: str = 'pt',
    warmup_epochs: int = 0,
) -> dict:
    """
    Treina o modelo com early stopping, warmup opcional e BLEU por época.

    Parâmetros:
        warmup_epochs: épocas de warmup linear (0 = desativado).
                       Recomendado para o modelo ALL: warmup_epochs=3
    """
    history = {
        'train_loss': [], 'val_loss'  : [],
        'train_ppl' : [], 'val_ppl'   : [],
        'bleu'      : [], 'lr'        : [],
    }

    best_val_loss   = float('inf')
    patience_count  = 0
    checkpoint_path = f"{cfg['checkpoint_dir']}/best_{tag}.pt"

    # Warmup scheduler (ativo apenas se warmup_epochs > 0)
    warmup = WarmupScheduler(
        optimizer,
        warmup_epochs = warmup_epochs,
        target_lr     = cfg['learning_rate'],
        start_lr      = cfg['learning_rate'] / 10,
    ) if warmup_epochs > 0 else None

    print(f'\n🚀 Iniciando treino [{tag}]...')
    print(f'   Épocas: {cfg["num_epochs"]} | Paciência: {cfg["patience"]} | '
          f'LR alvo: {cfg["learning_rate"]}')
    if warmup_epochs > 0:
        print(f'   Warmup: {warmup_epochs} épocas '
              f'({cfg["learning_rate"]/10:.1e} → {cfg["learning_rate"]:.1e})')
    print('─' * 75)

    for epoch in range(1, cfg['num_epochs'] + 1):

        # Aplicar warmup nas primeiras épocas
        in_warmup = warmup is not None and warmup.is_warming_up(epoch)
        if warmup is not None:
            warmup.step(epoch)

        train_loss, train_ppl = train_epoch(
            model, train_loader, optimizer, criterion, cfg['clip_grad']
        )
        val_loss, val_ppl = evaluate_epoch(model, val_loader, criterion)

        # BLEU em todas as épocas (com n_samples reduzido para performance)
        bleu = 0.0
        if df_val_sample is not None:
            bleu = compute_bleu_sample(
                model, df_val_sample, n_samples=20, lang=val_lang
            )

        # Adicionar após o cálculo do bleu no loop de train_model:
        if epoch == 1 and bleu == 0.0 and df_val_sample is not None:
            # Debug: mostrar exemplo de predição vs referência
            sample_row = df_val_sample.iloc[0]
            sample_pred = model.translate(sample_row['question'], lang=val_lang)
            print(f'  [DEBUG BLEU] Exemplo predição: "{sample_pred[:60]}"')
            print(f'  [DEBUG BLEU] Referência:       "{sample_row["query"][:60]}"')

        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_ppl'].append(train_ppl)
        history['val_ppl'].append(val_ppl)
        history['bleu'].append(bleu)
        history['lr'].append(current_lr)

        # ReduceLROnPlateau só atua APÓS o warmup
        if not in_warmup:
            scheduler.step(val_loss)

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss  = val_loss
            # Resetar paciência apenas fora do warmup
            if not in_warmup:
                patience_count = 0
            torch.save({
                'epoch'          : epoch,
                'model_state'    : model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'val_loss'       : val_loss,
                'history'        : history,
                'cfg'            : cfg,
            }, checkpoint_path)
        else:
            if not in_warmup:
                patience_count += 1

        warmup_flag = ' [warmup]' if in_warmup else ''
        bleu_str    = f'| BLEU-1: {bleu:.2f} ' if bleu > 0 else ''
        flag = '  ← ✅ melhor' if is_best else \
               f'  (paciência: {patience_count}/{cfg["patience"]})'

        print(
            f'Ep {epoch:02d}/{cfg["num_epochs"]} '
            f'| Train: {train_loss:.4f} (ppl={train_ppl:.1f}) '
            f'| Val: {val_loss:.4f} (ppl={val_ppl:.1f}) '
            f'| LR: {current_lr:.2e}{warmup_flag} {bleu_str}{flag}'
        )

        if not in_warmup and patience_count >= cfg['patience']:
            print(f'\n⏹️  Early stopping na época {epoch}.')
            break

    print(f'\n✅ Melhor Val Loss [{tag}]: {best_val_loss:.4f}')
    print(f'   Checkpoint: {checkpoint_path}')
    return history

def make_fresh_model(vocab_src: Vocabulary,
                     cfg: dict = CFG,
                     vocab_tgt: Vocabulary = None) -> Seq2Seq:
    """
    vocab_tgt: vocabulário alvo. Se None, usa tgt_vocab global (Spider).
    Para modelos expandidos, passar tgt_vocab_exp.
    """
    _tgt_vocab = vocab_tgt if vocab_tgt is not None else tgt_vocab

    enc = Encoder(
        vocab_size    = len(vocab_src),
        embed_dim     = cfg['embed_dim'],
        hidden_dim    = cfg['hidden_dim'],
        num_layers    = cfg['num_layers'],
        dropout       = cfg['dropout'],
        bidirectional = cfg['bidirectional'],
    )
    dec = Decoder(
        vocab_size = len(_tgt_vocab),   # usa o vocab correto
        embed_dim  = cfg['embed_dim'],
        hidden_dim = cfg['hidden_dim'],
        num_layers = cfg['num_layers'],
        dropout    = cfg['dropout'],
    )
    m = Seq2Seq(enc, dec, vocab_src, _tgt_vocab, DEVICE).to(DEVICE)
    m.apply(init_weights)
    return m

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C21 — TREINAMENTO COMPARATIVO: PORTUGUÊS vs INGLÊS vs ALL               ║
# ║                                                                          ║
# ║  Correções aplicadas nesta versão:                                       ║
# ║                                                                          ║
# ║  1. make_fresh_model(vocab_src, cfg) — recebe vocabulário e CFG          ║
# ║     explicitamente, garantindo embedding do tamanho correto por          ║
# ║     idioma e hiperparâmetros específicos por experimento.                ║
# ║                                                                          ║
# ║  2. CFG_EN_OVERRIDE — combate overfitting precoce do EN:                 ║
# ║     dropout 0.3→0.5 | lr 3e-4→1e-4 | patience 5→8 | tf 0.5→0.7           ║
# ║     Justificativa: EN parou na época 7 com divergência train/val         ║
# ║     crescendo desde época 3. Estas mudanças desaceleram a saturação.     ║
# ║                                                                          ║
# ║  3. Vocabulários separados por idioma:                                   ║
# ║     model_all → src_vocab     (2.410 tokens EN+PT)                       ║
# ║     model_en  → src_vocab_en  (1.755 tokens, sem ruído PT)               ║
# ║     model_pt  → src_vocab_pt  (721 tokens, sem ruído EN)                 ║
# ║     Todos compartilham tgt_vocab (2.105 tokens,SQL é universal)          ║
# ║                                                                          ║
# ║  4. CFG passado explicitamente ao train_model para que                   ║
# ║     teacher_forcing e patience sejam respeitados por modelo.             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

histories = {}

# ── 1. Modelo ALL (EN+PT, vocabulário compartilhado, CFG padrão) ──────────
if loaders_all.get('train'):
    print('═' * 65)
    print('▶ Modelo ALL — vocabulário compartilhado EN+PT')
    print(f'  Vocab src: {len(src_vocab):,} tokens | Vocab tgt: {len(tgt_vocab):,} tokens')
    print(f'  Amostras treino: {len(df_all[df_all["split"]=="train"]):,}')

    model_all = make_fresh_model(vocab_src=src_vocab, cfg=CFG_ALL_OVERRIDE)
    opt_all = optim.Adam(model_all.parameters(), lr=CFG_ALL_OVERRIDE['learning_rate'])
    sch_all = optim.lr_scheduler.ReduceLROnPlateau(
        opt_all,
        patience = CFG_ALL_OVERRIDE['lr_patience'],
        factor   = CFG_ALL_OVERRIDE['lr_factor'],
    )
    df_val_all = df_all[df_all['split'] == 'validation']
    histories['all'] = train_model(
        model_all,
        loaders_all['train'],
        loaders_all['validation'],
        opt_all, sch_all, criterion,
        cfg = CFG_ALL_OVERRIDE,          # CFG para ALL (lr reduzido pós-warmup)
        tag = 'all',
        df_val_sample = df_val_all,
        val_lang = 'pt',
        warmup_epochs = 3,  # warmup_epochs recomendado para ALL devido ao vocabulário maior e risco de overfitting precoce
    )

# ── 2. Modelo EN (vocabulário exclusivo EN, CFG_EN_OVERRIDE) ─────────────
if loaders_en.get('train'):
    print('\n' + '═' * 65)
    print('▶ Modelo EN — vocabulário exclusivo (sem tokens PT)')
    print(f'  Vocab src: {len(src_vocab_en):,} tokens | Vocab tgt: {len(tgt_vocab):,} tokens')
    print(f'  Tokens eliminados vs ALL: {len(src_vocab) - len(src_vocab_en):,} tokens PT removidos')
    print(f'  Amostras treino: {len(df_all[(df_all["split"]=="train") & (df_all["lang"]=="en")]):,}')
    print(f'  CFG override: dropout={CFG_EN_OVERRIDE["dropout"]} | '
          f'lr={CFG_EN_OVERRIDE["learning_rate"]:.0e} | '
          f'patience={CFG_EN_OVERRIDE["patience"]} | '
          f'tf={CFG_EN_OVERRIDE["teacher_forcing"]}')

    model_en = make_fresh_model(vocab_src=src_vocab_en, cfg=CFG_EN_OVERRIDE)
    opt_en = optim.Adam(model_en.parameters(),
                          lr=CFG_EN_OVERRIDE['learning_rate'])
    sch_en = optim.lr_scheduler.ReduceLROnPlateau(
        opt_en,
        patience = CFG_EN_OVERRIDE['lr_patience'],
        factor = CFG_EN_OVERRIDE['lr_factor'],
    )
    df_val_en = df_all[
        (df_all['split'] == 'validation') & (df_all['lang'] == 'en')
    ]
    histories['en'] = train_model(
        model_en,
        loaders_en['train'],
        loaders_en['validation'],
        opt_en, sch_en, criterion,
        cfg = CFG_EN_OVERRIDE,   # CFG específico do EN
        tag = 'en',
        df_val_sample = df_val_en,
        val_lang = 'en',
    )

# ── 3. Modelo PT (vocabulário exclusivo PT, CFG padrão) ──────────────────
if loaders_pt.get('train'):
    print('\n' + '═' * 65)
    print('▶ Modelo PT — vocabulário exclusivo (sem tokens EN)')
    print(f'  Vocab src: {len(src_vocab_pt):,} tokens | Vocab tgt: {len(tgt_vocab):,} tokens')
    print(f'  Tokens eliminados vs ALL: {len(src_vocab) - len(src_vocab_pt):,} tokens EN removidos')
    print(f'  Amostras treino: {len(df_all[(df_all["split"]=="train") & (df_all["lang"]=="pt")]):,}')

    model_pt = make_fresh_model(vocab_src=src_vocab_pt, cfg=CFG)
    opt_pt = optim.Adam(model_pt.parameters(), lr=CFG['learning_rate'])
    sch_pt = optim.lr_scheduler.ReduceLROnPlateau(
        opt_pt,
        patience = CFG['lr_patience'],
        factor = CFG['lr_factor'],
    )
    df_val_pt = df_all[
        (df_all['split'] == 'validation') & (df_all['lang'] == 'pt')
    ]
    histories['pt'] = train_model(
        model_pt,
        loaders_pt['train'],
        loaders_pt['validation'],
        opt_pt, sch_pt, criterion,
        cfg = CFG,           # CFG padrão
        tag = 'pt',
        df_val_sample = df_val_pt,
        val_lang = 'pt',
    )

# ── Verificação pós-treino ────────────────────────────────────────────────
print('\n' + '═' * 65)
print('── Resumo dos modelos treinados ─────────────────────────────────')
for key, hist in histories.items():
    cfg_used = CFG_EN_OVERRIDE if key == 'en' else CFG
    vocab_used = {'all': src_vocab, 'en': src_vocab_en, 'pt': src_vocab_pt}[key]
    print(f'\n  [{key.upper()}]')
    print(f'    Vocab src:    {len(vocab_used):,} tokens')
    print(f'    Épocas:       {len(hist["val_loss"])}')
    print(f'    Melhor val loss: {min(hist["val_loss"]):.4f}')
    print(f'    Melhor val ppl:  {min(hist["val_ppl"]):.2f}')
    if any(b > 0 for b in hist['bleu']):
        print(f'    Melhor BLEU-1:   {max(hist["bleu"]):.2f}')
    else:
        print(f'    BLEU-1:          ainda 0.0 — modelo em fase inicial')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C22 — VISUALIZAÇÃO COMPARATIVA + SALVAMENTO DOS PLOTS                   ║
# ║                                                                          ║
# ║  Plota curvas de loss, perplexidade e BLEU-1 para os três modelos.       ║
# ║  Salva cada figura em alta resolução para uso em apresentação.           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os
PLOTS_DIR = './plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

colors = {'all': 'purple', 'en': 'steelblue', 'pt': 'seagreen'}
labels = {'all': 'ALL (EN+PT)', 'en': 'English only', 'pt': 'Português only'}

# ── Figura 1: Loss + Perplexidade + BLEU (painel conjunto) ────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for key, hist in histories.items():
    epochs = range(1, len(hist['train_loss']) + 1)
    c = colors[key]
    l = labels[key]

    axes[0].plot(epochs, hist['train_loss'], c=c, linestyle='--',
                 alpha=0.6, label=f'{l} train')
    axes[0].plot(epochs, hist['val_loss'], c=c, linestyle='-',
                 label=f'{l} val', linewidth=2)

    axes[1].plot(epochs, hist['val_ppl'], c=c, label=l, linewidth=2)

    bleu_epochs = [e for e, b in zip(epochs, hist['bleu']) if b > 0]
    bleu_vals   = [b for b in hist['bleu'] if b > 0]
    if bleu_vals:
        axes[2].plot(bleu_epochs, bleu_vals, c=c, marker='o',
                     label=l, linewidth=2)

axes[0].set_title('Curva de Loss (Train vs Val)')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8)

axes[1].set_title('Perplexidade de Validação')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Perplexidade (↓ melhor)')
axes[1].legend()

axes[2].set_title('BLEU-1 de Validação (↑ melhor)')
axes[2].set_xlabel('Época')
axes[2].set_ylabel('BLEU-1')
axes[2].legend()

plt.suptitle('Comparação: EN vs PT vs ALL — Modelo Seq2Seq LSTM', fontsize=13)
plt.tight_layout()
fig.savefig(f'{PLOTS_DIR}/fig1_loss_ppl_bleu.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Salvo: {PLOTS_DIR}/fig1_loss_ppl_bleu.png')

# ── Figura 2: Loss separadas (train e val em subplots individuais) ────────
fig2, axes2 = plt.subplots(1, 3, figsize=(20, 5), sharey=False)
for ax, (key, hist) in zip(axes2, histories.items()):
    epochs = range(1, len(hist['train_loss']) + 1)
    c = colors[key]
    ax.plot(epochs, hist['train_loss'], c=c, linestyle='--',
            alpha=0.7, label='train', linewidth=2)
    ax.plot(epochs, hist['val_loss'], c=c, linestyle='-',
            label='val', linewidth=2)
    ax.set_title(f'Loss — {labels[key]}')
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.legend()
    # Marcar época de melhor val_loss
    best_ep = int(np.argmin(hist['val_loss'])) + 1
    best_vl = min(hist['val_loss'])
    ax.axvline(best_ep, color='red', linestyle=':', alpha=0.5,
               label=f'melhor val (ep {best_ep})')
    ax.annotate(f'{best_vl:.3f}',
                xy=(best_ep, best_vl),
                xytext=(best_ep + 0.3, best_vl + 0.05),
                fontsize=8, color='red')
    ax.legend(fontsize=8)

plt.suptitle('Curvas de Loss por Modelo — Seq2Seq LSTM', fontsize=13)
plt.tight_layout()
fig2.savefig(f'{PLOTS_DIR}/fig2_loss_por_modelo.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Salvo: {PLOTS_DIR}/fig2_loss_por_modelo.png')

# ── Figura 3: BLEU-1 por modelo (barras no melhor valor) ─────────────────
fig3, ax3 = plt.subplots(figsize=(8, 5))
best_bleus  = {k: max(h['bleu']) if any(b > 0 for b in h['bleu']) else 0
               for k, h in histories.items()}
bar_labels  = [labels[k] for k in best_bleus]
bar_vals    = list(best_bleus.values())
bar_colors  = [colors[k] for k in best_bleus]
bars = ax3.bar(bar_labels, bar_vals, color=bar_colors, edgecolor='white',
               width=0.5)
for bar, val in zip(bars, bar_vals):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.3,
             f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')
ax3.set_title('Melhor BLEU-1 de Validação por Modelo')
ax3.set_ylabel('BLEU-1 (↑ melhor)')
ax3.set_ylim(0, max(bar_vals) * 1.25 if bar_vals else 1)
plt.tight_layout()
fig3.savefig(f'{PLOTS_DIR}/fig3_bleu_comparativo.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Salvo: {PLOTS_DIR}/fig3_bleu_comparativo.png')

# ── Figura 4: Tabela visual (resumo como heatmap de métricas) ─────────────
resumo_data = {
    'Modelo'         : [],
    'Amostras treino': [],
    'Épocas'         : [],
    'Val Loss'       : [],
    'Val PPL'        : [],
    'Melhor BLEU-1'  : [],
}
for key in ['all', 'en', 'pt']:
    if key not in histories:
        continue
    hist = histories[key]
    n_tr = (len(df_all[df_all['split'] == 'train']) if key == 'all'
            else len(df_all[(df_all['split'] == 'train') & (df_all['lang'] == key)]))
    resumo_data['Modelo'].append(labels[key])
    resumo_data['Amostras treino'].append(n_tr)
    resumo_data['Épocas'].append(len(hist['val_loss']))
    resumo_data['Val Loss'].append(round(min(hist['val_loss']), 4))
    resumo_data['Val PPL'].append(round(min(hist['val_ppl']), 2))
    resumo_data['Melhor BLEU-1'].append(
        round(max(hist['bleu']), 2) if any(b > 0 for b in hist['bleu']) else 0.0
    )

df_resumo_plot = pd.DataFrame(resumo_data).set_index('Modelo')

fig4, ax4 = plt.subplots(figsize=(10, 2.5))
ax4.axis('off')
tbl = ax4.table(
    cellText  = df_resumo_plot.reset_index().values,
    colLabels = df_resumo_plot.reset_index().columns,
    cellLoc   = 'center',
    loc       = 'center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 2.0)
# Colorir cabeçalho
for j in range(len(df_resumo_plot.columns) + 1):
    tbl[0, j].set_facecolor('#DDDDEE')
    tbl[0, j].set_text_props(fontweight='bold')
plt.title('Resumo Comparativo — Seq2Seq LSTM (Spider)',
          fontsize=12, pad=15)
plt.tight_layout()
fig4.savefig(f'{PLOTS_DIR}/fig4_tabela_resumo.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Salvo: {PLOTS_DIR}/fig4_tabela_resumo.png')

print(f'\n✅ Todos os plots salvos em: {PLOTS_DIR}/')
print('\n── Resumo Final ──────────────────────────────────────────────')
print(df_resumo_plot.to_string())

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C23 — CARREGAMENTO DOS MODELOS TREINADOS                                ║
# ║                                                                          ║
# ║  Demonstra como recarregar os três modelos salvos a partir dos           ║
# ║  checkpoints. Esta célula pode ser executada de forma independente       ║
# ║  em qualquer sessão futura, sem necessidade de re-treinar.               ║
# ║                                                                          ║
# ║  Estrutura do checkpoint salvo:                                          ║
# ║  - model_state:     pesos do modelo (encoder + decoder + atenção)        ║
# ║  - optimizer_state: estado do Adam (momentos, lr adaptativo)             ║
# ║  - scheduler_state: estado do ReduceLROnPlateau                          ║
# ║  - val_loss:        melhor val loss atingido                             ║
# ║  - history:         histórico completo de métricas por época             ║
# ║  - cfg:             hiperparâmetros usados no treino                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
def load_model_from_checkpoint(
    checkpoint_path: str,
    vocab_src: Vocabulary,
    vocab_tgt: Vocabulary,
    cfg: dict,
    device: torch.device,
) -> Seq2Seq:
    checkpoint = torch.load(checkpoint_path, map_location=device,
                             weights_only=False)
    enc = Encoder(
        vocab_size    = len(vocab_src),
        embed_dim     = cfg['embed_dim'],
        hidden_dim    = cfg['hidden_dim'],
        num_layers    = cfg['num_layers'],
        dropout       = cfg['dropout'],
        bidirectional = cfg['bidirectional'],
    )
    dec = Decoder(
        vocab_size = len(vocab_tgt),
        embed_dim  = cfg['embed_dim'],
        hidden_dim = cfg['hidden_dim'],
        num_layers = cfg['num_layers'],
        dropout    = cfg['dropout'],
    )
    m = Seq2Seq(enc, dec, vocab_src, vocab_tgt, device).to(device)
    m.load_state_dict(checkpoint['model_state'])
    m.eval()
    epoch    = checkpoint.get('epoch', '?')
    val_loss = checkpoint.get('val_loss', float('inf'))
    print(f'  ✅ Carregado | melhor época={epoch} | val_loss={val_loss:.4f}')
    return m


print('📂 Carregando modelos dos checkpoints...\n')
model_all_loaded = load_model_from_checkpoint(
    './checkpoints/best_all.pt', src_vocab,    tgt_vocab, CFG_ALL_OVERRIDE, DEVICE)
model_en_loaded  = load_model_from_checkpoint(
    './checkpoints/best_en.pt',  src_vocab_en, tgt_vocab, CFG_EN_OVERRIDE,  DEVICE)
model_pt_loaded  = load_model_from_checkpoint(
    './checkpoints/best_pt.pt',  src_vocab_pt, tgt_vocab, CFG, DEVICE)
print('\n✅ Todos os modelos prontos para inferência.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C24 — DEMONSTRAÇÃO DE INFERÊNCIA: PERGUNTAS EM PORTUGUÊS                ║
# ║                                                                          ║
# ║  Avalia os três modelos em perguntas reais em PT.                        ║
# ║  Permite análise qualitativa dos resultados para a apresentação.         ║
# ║                                                                          ║
# ║  INTERPRETAÇÃO ESPERADA DOS RESULTADOS:                                  ║
# ║  - Estrutura SQL correta (SELECT...FROM...WHERE) = modelo aprendeu       ║
# ║    a sintaxe da linguagem                                                ║
# ║  - Entidades erradas (tabelas/colunas incorretas) = modelo não           ║
# ║    aprendeu o mapeamento semântico pergunta→esquema                      ║
# ║  - Isso motiva o Fine-Tuning com schema do banco na entrada              ║
# ║                                                                          ║
# ║  DIFERENÇA IMPORTANTE vs versão anterior:                                ║
# ║  Antes: BLEU calculado sobre 5 perguntas sintéticas de demo →            ║
# ║  resultado sempre 0 porque as referências usam nomes (singer, concert)   ║
# ║  que não aparecem nas predições do modelo sem schema.                    ║
# ║                                                                          ║
# ║  Agora: BLEU calculado sobre o conjunto de TESTE real (Spider),          ║
# ║  usando as mesmas perguntas e referências que estiveram fora do          ║
# ║  treinamento. Isso dá um número honesto do que o modelo aprendeu.        ║
# ║                                                                          ║
# ║  BLEU-1 (unigrams): mais adequado para esta fase. Com ppl ≈ 70-105       ║
# ║  o modelo ainda erra entidades mas acerta tokens SQL estruturais         ║
# ║  (SELECT, FROM, WHERE, COUNT, GROUP, BY) que o BLEU-1 captura.           ║
# ║  O BLEU-4 retornaria ~0 porque raramente 4 tokens consecutivos           ║
# ║  coincidem quando as entidades estão erradas.                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

from sacrebleu.metrics import BLEU as SacreBLEU

@torch.no_grad()
def avaliar_bleu_teste(
    model,
    df_test     : pd.DataFrame,
    lang        : str = 'pt',
    n_samples   : int = 200,
    bleu_order  : int = 1,
) -> Tuple[float, List[str], List[str]]:
    """
    Avalia BLEU-N no conjunto de teste.

    Retorna (score, hyps, refs) para análise posterior.

    A assinatura correta do sacrebleu é:
        corpus_score(hypotheses, [list_of_reference_lists])
    onde cada elemento de list_of_reference_lists é uma lista
    de strings (uma por amostra), NÃO uma lista de listas.
    """
    model.eval()
    sub = df_test.sample(min(n_samples, len(df_test)), random_state=SEED)

    hyps = []
    refs = []

    for _, row in tqdm(sub.iterrows(), total=len(sub),
                       desc=f'BLEU-{bleu_order}', leave=False):
        try:
            pred = model.translate(row['question'], lang=lang)
        except Exception:
            pred = ''

        pred_clean = pred.lower().strip()
        ref_clean  = row['query'].lower().strip()

        # Ignorar predições completamente vazias
        if not pred_clean:
            continue

        hyps.append(pred_clean)
        refs.append(ref_clean)

    if not hyps:
        return 0.0, [], []

    try:
        # refs é List[str], passado como [refs] → List[List[str]]
        # corpus_score(hyps, [[ref1, ref2, ...]]) = uma referência por amostra
        bleu  = SacreBLEU(
            tokenize      = '13a',
            max_ngram_order = bleu_order,
            smooth_method = 'exp',
            effective_order = True,
        )
        score = bleu.corpus_score(hyps, [refs]).score
    except Exception as e:
        print(f'  ⚠️  Erro BLEU: {e}')
        score = 0.0

    return score, hyps, refs


# Conjuntos de teste
df_test_pt  = df_all[(df_all['split'] == 'test') & (df_all['lang'] == 'pt')].reset_index(drop=True)
df_test_en  = df_all[(df_all['split'] == 'test') & (df_all['lang'] == 'en')].reset_index(drop=True)
df_test_all = df_all[df_all['split'] == 'test'].reset_index(drop=True)

print(f'Amostras de teste: PT={len(df_test_pt)} | EN={len(df_test_en)} | ALL={len(df_test_all)}\n')

print('── BLEU-1 no conjunto de teste (n=200 amostras) ─────────────────')
bleu_all_test, hyps_all, refs_all = avaliar_bleu_teste(
    model_all_loaded, df_test_all, lang='pt', n_samples=200)
bleu_en_test,  hyps_en,  refs_en  = avaliar_bleu_teste(
    model_en_loaded,  df_test_en,  lang='en', n_samples=200)
bleu_pt_test,  hyps_pt,  refs_pt  = avaliar_bleu_teste(
    model_pt_loaded,  df_test_pt,  lang='pt', n_samples=200)

print(f'  Modelo ALL (testado em ALL): {bleu_all_test:.2f}')
print(f'  Modelo EN  (testado em EN):  {bleu_en_test:.2f}')
print(f'  Modelo PT  (testado em PT):  {bleu_pt_test:.2f}')

# ── Exemplos de predição vs referência ────────────────────────────────────
print('\n── Exemplos: predição vs referência (modelo ALL no teste ALL) ───')
for hyp, ref in list(zip(hyps_all, refs_all))[:5]:
    print(f'  REF: {ref}')
    print(f'  HYP: {hyp}')
    print()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C25 — DEMONSTRAÇÃO DE INFERÊNCIA: PERGUNTAS EM INGLÊS                   ║
# ║                                                                          ║
# ║  Avalia o modelo EN em perguntas em inglês para comparação               ║
# ║  direta com o comportamento em português.                                ║
# ║                                                                          ║
# ║  Mantém as perguntas sintéticas de demonstração para análise visual,     ║
# ║  mas agora com interpretação correta: o objetivo aqui NÃO é obter        ║
# ║  BLEU alto, mas mostrar que o modelo aprendeu estrutura SQL sem          ║
# ║  precisar de schema na entrada.                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

perguntas_demo = {
    'pt': [
        {'q': 'Quais são os nomes de todos os cantores?',
         'ref': 'SELECT name FROM singer'},
        {'q': 'Quantos concertos foram realizados em 2014?',
         'ref': 'SELECT count(*) FROM concert WHERE year = 2014'},
        {'q': 'Qual é a média de idade dos cantores de cada país?',
         'ref': 'SELECT country, avg(age) FROM singer GROUP BY country'},
        {'q': 'Liste os nomes dos cantores com mais de 30 anos ordenados por nome.',
         'ref': 'SELECT name FROM singer WHERE age > 30 ORDER BY name'},
        {'q': 'Quantos alunos existem em cada departamento?',
         'ref': 'SELECT dept_name, count(*) FROM student GROUP BY dept_name'},
    ],
    'en': [
        {'q': 'What are the names of all singers?',
         'ref': 'SELECT name FROM singer'},
        {'q': 'How many concerts were held in 2014?',
         'ref': 'SELECT count(*) FROM concert WHERE year = 2014'},
        {'q': 'What is the average age of singers from each country?',
         'ref': 'SELECT country, avg(age) FROM singer GROUP BY country'},
        {'q': 'List singer names older than 30 ordered by name.',
         'ref': 'SELECT name FROM singer WHERE age > 30 ORDER BY name'},
        {'q': 'How many students are there in each department?',
         'ref': 'SELECT dept_name, count(*) FROM student GROUP BY dept_name'},
    ],
}

# Acumular para análise estrutural
resultados_demo = {'all': [], 'pt': [], 'en': []}

print('═' * 70)
print('  DEMONSTRAÇÃO QUALITATIVA — PT (modelos ALL e PT)')
print('═' * 70)
for i, ex in enumerate(perguntas_demo['pt'], 1):
    pred_all = model_all_loaded.translate(ex['q'], lang='pt')
    pred_pt  = model_pt_loaded.translate(ex['q'],  lang='pt')
    print(f'\n[{i}] {ex["q"]}')
    print(f'  Ref : {ex["ref"]}')
    print(f'  ALL : {pred_all or "[vazio]"}')
    print(f'  PT  : {pred_pt  or "[vazio]"}')
    resultados_demo['all'].append({'pred': pred_all, 'ref': ex['ref']})
    resultados_demo['pt'].append({'pred': pred_pt,   'ref': ex['ref']})

print('\n' + '═' * 70)
print('  DEMONSTRAÇÃO QUALITATIVA — EN (modelo EN)')
print('═' * 70)
for i, ex in enumerate(perguntas_demo['en'], 1):
    pred_en = model_en_loaded.translate(ex['q'], lang='en')
    print(f'\n[{i}] {ex["q"]}')
    print(f'  Ref: {ex["ref"]}')
    print(f'  EN : {pred_en or "[vazio]"}')
    resultados_demo['en'].append({'pred': pred_en, 'ref': ex['ref']})

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C26 — ANÁLISE QUALITATIVA E MOTIVAÇÃO PARA FINE-TUNING                  ║
# ║                                                                          ║
# ║  Sintetiza os resultados dos três modelos e demonstra quantitativamente  ║
# ║  por que o BLEU permanece em 0 e por que o Fine-Tuning é necessário.     ║
# ║                                                                          ║
# ║  CORREÇÃO CRÍTICA na função avaliar_bleu1:                               ║
# ║  Versão anterior: refs = [[r] for r in refs] + corpus_score(hyps,[refs]) ║
# ║  → criava estrutura [[[r1],[r2],...]] (3 níveis de lista) → BLEU=0       ║
# ║                                                                          ║
# ║  Versão correta: refs é List[str], passado como [refs] diretamente       ║
# ║  → corpus_score(hyps, [refs]) → estrutura correta do sacrebleu           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def avaliar_bleu1_demo(predicoes: List[str],
                       referencias: List[str]) -> float:
    """
    BLEU-1 sobre lista de predições e referências.

    sacrebleu.corpus_score(hypotheses, list_of_ref_streams)
      hypotheses      : List[str]  — uma string por amostra
      list_of_ref_streams: List[List[str]] — lista de "streams" de referência
                           Para uma referência por amostra: [refs_list]
    """
    hyps = [p.lower().strip() if p.strip() else 'select'
            for p in predicoes]
    refs = [r.lower().strip() for r in referencias]

    if not hyps:
        return 0.0
    try:
        bleu  = SacreBLEU(
            tokenize        = '13a',
            max_ngram_order = 1,
            smooth_method   = 'exp',
            effective_order = True,
        )
        # ✅ [refs] = lista contendo uma lista de strings
        return bleu.corpus_score(hyps, [refs]).score
    except Exception as e:
        print(f'  ⚠️  Erro BLEU: {e}')
        return 0.0


# BLEU-1 nas demos (predições sintéticas — esperado ser baixo: sem schema)
preds_all_demo = [r['pred'] for r in resultados_demo['all']]
preds_pt_demo  = [r['pred'] for r in resultados_demo['pt']]
preds_en_demo  = [r['pred'] for r in resultados_demo['en']]
refs_pt_demo   = [r['ref']  for r in resultados_demo['pt']]
refs_en_demo   = [r['ref']  for r in resultados_demo['en']]

bleu_all_demo = avaliar_bleu1_demo(preds_all_demo, refs_pt_demo)
bleu_pt_demo  = avaliar_bleu1_demo(preds_pt_demo,  refs_pt_demo)
bleu_en_demo  = avaliar_bleu1_demo(preds_en_demo,  refs_en_demo)

print('═' * 70)
print('  ANÁLISE QUALITATIVA DOS RESULTADOS')
print('═' * 70)

print('\n── BLEU-1 comparativo ────────────────────────────────────────────')
print('  [Conjunto de TESTE real — Spider]:')
print(f'    Modelo ALL : {bleu_all_test:.2f}')
print(f'    Modelo EN  : {bleu_en_test:.2f}')
print(f'    Modelo PT  : {bleu_pt_test:.2f}')
print()
print('  [Perguntas de DEMO sintéticas — sem schema na entrada]:')
print(f'    ALL (demo) : {bleu_all_demo:.2f}  ← baixo esperado: entidades erradas')
print(f'    PT  (demo) : {bleu_pt_demo:.2f}  ← idem')
print(f'    EN  (demo) : {bleu_en_demo:.2f}  ← idem')

# ── Análise estrutural ────────────────────────────────────────────────────
print('\n── Verificação estrutural das predições (demo) ───────────────────')
for nome, preds in [('ALL', preds_all_demo),
                    ('PT',  preds_pt_demo),
                    ('EN',  preds_en_demo)]:
    n = len(preds)
    n_select  = sum('SELECT' in p.upper() for p in preds)
    n_from    = sum('FROM'   in p.upper() for p in preds)
    n_where   = sum('WHERE'  in p.upper() for p in preds)
    n_nonempty = sum(bool(p.strip()) for p in preds)
    print(f'  [{nome}] {n_nonempty}/{n} não-vazias | '
          f'SELECT:{n_select}/{n} | FROM:{n_from}/{n} | WHERE:{n_where}/{n}')

# ── Interpretação textual ─────────────────────────────────────────────────
print("""
── Interpretação dos resultados ──────────────────────────────────

  BLEU no conjunto de teste real (C24):
    Valores de 5-35 são coerentes com ppl ≈ 70-105.
    O modelo aprendeu tokens SQL estruturais (SELECT, FROM, COUNT,
    WHERE, GROUP, ORDER) que coincidem com as referências —
    por isso o BLEU-1 não é zero. O BLEU-4 seria próximo de zero
    porque 4-grams consecutivos raramente coincidem quando nomes
    de tabelas e colunas estão errados.

  BLEU nas demos sintéticas (C25):
    As referencias usam nomes como "singer", "concert", "student".
    O modelo nunca recebeu o schema como entrada, então nunca sabe
    qual tabela usar para cada pergunta — erra sistematicamente as
    entidades. Por isso o BLEU cai drasticamente em demos sintéticas
    mesmo que o BLEU no teste real seja razoável.

  Conclusão:
    O modelo Seq2Seq aprendeu ESTRUTURA SQL (sintaxe, ordem de
    cláusulas, uso de COUNT/AVG/GROUP BY). Não aprendeu SEMÂNTICA
    (qual tabela/coluna usar para cada pergunta). Isso é resultado
    esperado e motiva o Fine-Tuning com T5 + schema na entrada.
""")

# ── Tabela final ──────────────────────────────────────────────────────────
print('── Tabela resumo final ───────────────────────────────────────────')
linhas = []
for key in ['all', 'en', 'pt']:
    if key not in histories:
        continue
    hist = histories[key]
    n_tr = (len(df_all[df_all['split'] == 'train']) if key == 'all' else
            len(df_all[(df_all['split'] == 'train') & (df_all['lang'] == key)]))
    bleu_teste = {'all': bleu_all_test, 'en': bleu_en_test,
                  'pt': bleu_pt_test}[key]
    linhas.append({
        'Modelo'          : labels[key],
        'N treino'        : f'{n_tr:,}',
        'Épocas'          : len(hist['val_loss']),
        'Val Loss'        : f'{min(hist["val_loss"]):.4f}',
        'Val PPL'         : f'{min(hist["val_ppl"]):.1f}',
        'BLEU-1 (val)'    : f'{max(hist["bleu"]):.1f}',
        'BLEU-1 (teste)'  : f'{bleu_teste:.1f}',
    })

df_final = pd.DataFrame(linhas)
print(df_final.to_string(index=False))

# Salvar tabela como figura também
fig5, ax5 = plt.subplots(figsize=(13, 2.5))
ax5.axis('off')
tbl5 = ax5.table(
    cellText  = df_final.values,
    colLabels = df_final.columns,
    cellLoc   = 'center', loc='center',
)
tbl5.auto_set_font_size(False)
tbl5.set_fontsize(9)
tbl5.scale(1.1, 2.0)
for j in range(len(df_final.columns)):
    tbl5[0, j].set_facecolor('#DDDDEE')
    tbl5[0, j].set_text_props(fontweight='bold')
plt.title('Resultados Finais — Seq2Seq LSTM (Spider)', fontsize=11, pad=12)
plt.tight_layout()
fig5.savefig(f'{PLOTS_DIR}/fig5_tabela_final.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ {PLOTS_DIR}/fig5_tabela_final.png salvo.')

## 📚 PARTE 6 — Expansão do Dataset com WikiSQL

### Motivação

Os resultados da Seção 5 demonstraram que o modelo Seq2Seq treinado do
zero com ~10.000 pares atinge perplexidade de 98–142, aprende a estrutura
sintática do SQL, mas falha no mapeamento semântico pergunta→esquema.

Uma das causas identificadas é a **escassez de dados**, especialmente em
português (~1.500 amostras de treino). Esta seção incorpora o dataset
**WikiSQL** ao pipeline existente para investigar o impacto do volume de
dados nos resultados.

### WikiSQL

| Característica | Valor |
|---|---|
| Fonte | Salesforce Research (2017) |
| Idioma | Inglês |
| Tamanho | ~80.000 pares question→SQL |
| Splits | train (61.297) + val (9.145) + test (15.878) |
| Complexidade | Queries simples (sem JOIN) sobre esquemas reais |
| HuggingFace | `wikisql` |

### Diferenças em relação ao Spider

O WikiSQL contém queries mais simples que o Spider — sem JOINs entre
tabelas, sem subqueries. Em compensação, tem ~8x mais dados. A combinação
dos dois datasets cria um corpus com diversidade de complexidade.

### O que esta seção faz

1. Carrega o WikiSQL do HuggingFace nos três splits (train/val/test)
2. Padroniza o formato para o mesmo schema do projeto (question/query/split/lang/source)
3. Realiza EDA comparativa WikiSQL vs Spider
4. Verifica duplicatas entre datasets
5. Incorpora ao `df_all` existente e reconstrói vocabulários e DataLoaders
6. Re-treina os modelos com o dataset expandido para comparação

### Nota sobre idioma

O WikiSQL é exclusivamente em inglês. Ele será adicionado ao conjunto EN
e ao conjunto ALL. O PT permanece apenas com Spider PT — não há versão
traduzida do WikiSQL de qualidade suficiente disponível publicamente.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C27 — CARREGAMENTO DO SQL-CREATE-CONTEXT (substituto do WikiSQL)        ║
# ║                                                                          ║
# ║  PROBLEMA: o dataset 'wikisql' original usa um script Python legado      ║
# ║  que versões recentes do HuggingFace datasets não suportam mais.         ║
# ║  Erro: "Dataset scripts are no longer supported"                         ║
# ║                                                                          ║
# ║  SOLUÇÃO: usar b-mc2/sql-create-context, que é:                          ║
# ║  - Derivado de WikiSQL + Spider (78.577 exemplos)                        ║
# ║  - Formato Parquet moderno — sem scripts legados                         ║
# ║  - Inclui schema CREATE TABLE como contexto por pergunta                 ║
# ║  - Queries limpas e normalizadas com SQLGlot + SQLParse                  ║
# ║  - Colunas: question | context (schema) | answer (SQL)                   ║
# ║                                                                          ║
# ║  VANTAGEM EXTRA: o campo 'context' contém o schema do banco, que será    ║
# ║  usado na Seção 7 (Fine-Tuning) para concatenar à pergunta.              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
SQL_CONTEXT_DATASET = 'b-mc2/sql-create-context'

print(f'🔍 Inspecionando {SQL_CONTEXT_DATASET} no HuggingFace...')
try:
    sc_splits = get_dataset_split_names(SQL_CONTEXT_DATASET)
    print(f'  Splits disponíveis: {sc_splits}')
except Exception as e:
    print(f'  ⚠️  Introspecção falhou: {e}')
    sc_splits = ['train']

print(f'\n📥 Carregando {SQL_CONTEXT_DATASET}...')
try:
    ds_sc = load_dataset(SQL_CONTEXT_DATASET, split='train')
    print(f'  ✅ Carregado: {len(ds_sc):,} registros')
    print(f'  Colunas: {ds_sc.column_names}')
    sample = ds_sc[0]
    print(f'\n  Amostra[0]:')
    print(f'    question: {sample["question"]}')
    print(f'    context:  {sample["context"][:80]}...')
    print(f'    answer:   {sample["answer"]}')
except Exception as e:
    raise RuntimeError(
        f'Falha ao carregar {SQL_CONTEXT_DATASET}: {e}\n'
        'Verifique sua conexão com o HuggingFace.'
    )

df_sc_raw = pd.DataFrame({
    'question': ds_sc['question'],
    'query'   : ds_sc['answer'],
    'schema'  : ds_sc['context'],
    'lang'    : 'en',
    'source'  : 'sql-create-context',
    'split'   : 'raw',
})
print(f'\n── DataFrame criado: {len(df_sc_raw):,} registros ──────────────────')
print(df_sc_raw[['question', 'query', 'schema']].head(3).to_string())

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C28 — LIMPEZA, SPLITS E DEDUPLICAÇÃO DO SQL-CREATE-CONTEXT              ║
# ║                                                                          ║
# ║  O sql-create-context tem apenas split 'train'.                          ║
# ║  Geramos train/val/test (70/15/15) com a mesma função create_splits      ║
# ║  já definida no projeto, mantendo consistência metodológica.             ║
# ║                                                                          ║
# ║  Deduplicação em dois níveis:                                            ║
# ║  1. Interna: remover duplicatas dentro do sql-create-context             ║
# ║  2. Cross-dataset: remover perguntas que já existem no Spider            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print('🧹 Limpando sql-create-context...')
before = len(df_sc_raw)

df_sc = df_sc_raw.copy()
df_sc['question'] = df_sc['question'].str.strip()
df_sc['query']    = df_sc['query'].str.strip().str.upper()
df_sc['schema']   = df_sc['schema'].str.strip()

df_sc = df_sc.dropna(subset=['question', 'query'])
df_sc = df_sc[df_sc['question'].str.len() > 5]
df_sc = df_sc[df_sc['query'].str.len() > 10]
df_sc = df_sc[df_sc['query'].str.upper().str.startswith('SELECT')]
df_sc = df_sc.drop_duplicates(subset=['question', 'query']).reset_index(drop=True)
after_internal = len(df_sc)

spider_questions_lower = set(df_all['question'].str.lower().str.strip())
sc_questions_lower     = df_sc['question'].str.lower().str.strip()
dup_mask    = sc_questions_lower.isin(spider_questions_lower)
n_cross_dup = dup_mask.sum()
df_sc       = df_sc[~dup_mask].reset_index(drop=True)
after_dedup = len(df_sc)

print(f'  Original:               {before:,}')
print(f'  Após limpeza interna:   {after_internal:,} (-{before - after_internal:,})')
print(f'  Duplicatas com Spider:  {n_cross_dup:,}')
print(f'  Final limpo:            {after_dedup:,}')

print('\n⚙️  Gerando splits train/val/test (70/15/15)...')
df_sc = create_splits(df_sc, train_r=0.70, val_r=0.15, seed=SEED)

print('\nSplits sql-create-context:')
print(df_sc['split'].value_counts().to_string())

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C29 — EDA COMPARATIVA: SPIDER vs SQL-CREATE-CONTEXT                     ║
# ║                                                                          ║
# ║  Reutiliza SQL_KEYWORDS e CFG já definidos — sem repetição de código.    ║
# ║  Foco nas diferenças de distribuição entre os dois datasets.             ║
# ║  CORREÇÃO: adicionado salvamento do plot EDA em PLOTS_DIR.               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
df_sc['q_len_raw']   = df_sc['question'].str.split().str.len()
df_sc['sql_len_raw'] = df_sc['query'].str.split().str.len()

fig_eda, axes_eda = plt.subplots(2, 2, figsize=(16, 10))

datasets_eda = [
    (df_all[df_all['lang'] == 'en'], 'Spider EN'),
    (df_sc,                           'sql-create-context'),
]

for row, (df_eda, label) in enumerate(datasets_eda):
    for col, (metric, title, color, lim_key) in enumerate([
        ('q_len_raw',   'Perguntas (palavras)',   'steelblue', 'max_src_len'),
        ('sql_len_raw', 'Queries SQL (palavras)', 'seagreen',  'max_tgt_len'),
    ]):
        ax  = axes_eda[row, col]
        med = df_eda[metric].median()
        p95 = df_eda[metric].quantile(0.95)

        ax.hist(df_eda[metric], bins=40, color=color, edgecolor='white', alpha=0.8)
        ax.axvline(med, color='red',    linestyle='--', label=f'Mediana={med:.0f}')
        ax.axvline(p95, color='orange', linestyle=':',  label=f'P95={p95:.0f}')
        ax.axvline(CFG[lim_key], color='black', linestyle='-.',
                   label=f'Limite CFG={CFG[lim_key]}')
        ax.set_title(f'[{label}] {title}')
        ax.set_xlabel('Nº de palavras')
        ax.legend(fontsize=8)

plt.suptitle('EDA: Spider EN vs sql-create-context', fontsize=14, y=1.01)
plt.tight_layout()
fig_eda.savefig(f'{PLOTS_DIR}/fig6_eda_spider_vs_sqlctx.png',
                dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Salvo: {PLOTS_DIR}/fig6_eda_spider_vs_sqlctx.png')

# Estatísticas
print('── Estatísticas ──────────────────────────────────────────────')
for df_eda, label in datasets_eda:
    stats = df_eda[['q_len_raw', 'sql_len_raw']].describe().round(1)
    print(f'\n[{label}] (n={len(df_eda):,})')
    print(stats.to_string())

print('\n── % acima dos limites do CFG ────────────────────────────────')
for df_eda, label in datasets_eda:
    pq = (df_eda['q_len_raw']   > CFG['max_src_len']).mean() * 100
    ps = (df_eda['sql_len_raw'] > CFG['max_tgt_len']).mean() * 100
    print(f'  [{label}]')
    print(f'    Perguntas acima do limite: {pq:.1f}%')
    print(f'    Queries acima do limite:   {ps:.1f}%')

print('\n── Frequência de cláusulas SQL ───────────────────────────────')
n_spider = len(df_all[df_all['lang'] == 'en'])
n_sc     = len(df_sc)
kw_rows  = []
for kw in SQL_KEYWORDS:
    spider_count = (df_all[df_all['lang'] == 'en']['query']
                    .str.upper().str.contains(re.escape(kw)).sum())
    sc_count     = df_sc['query'].str.upper().str.contains(re.escape(kw)).sum()
    kw_rows.append({
        'Cláusula'   : kw,
        'Spider EN'  : spider_count,
        'Spider EN %': round(spider_count / n_spider * 100, 1),
        'sql-ctx'    : sc_count,
        'sql-ctx %'  : round(sc_count / n_sc * 100, 1),
    })
df_kw = pd.DataFrame(kw_rows).sort_values('Spider EN', ascending=False)
print(df_kw.to_string(index=False))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C30 — INCORPORAÇÃO DO SQL-CREATE-CONTEXT AO DATASET PRINCIPAL           ║
# ║                                                                          ║
# ║  O sql-create-context inclui a coluna 'schema' com o CREATE TABLE.       ║
# ║  Para manter compatibilidade com df_all (que não tem 'schema'),          ║
# ║  adicionamos a coluna 'schema' ao df_all com valor vazio — ela será      ║
# ║  usada apenas no Fine-Tuning (Seção 7), não no LSTM seq2seq.             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print('🔗 Incorporando sql-create-context ao dataset principal...')

if 'schema' not in df_all.columns:
    df_all['schema'] = ''

cols_needed = [c for c in df_all.columns if c != 'schema']
for col in cols_needed:
    if col not in df_sc.columns:
        df_sc[col] = None

df_all_expanded = pd.concat([df_all, df_sc], ignore_index=True)
df_all_expanded = df_all_expanded.drop_duplicates(
    subset=['question', 'query']
).reset_index(drop=True)

n_with_schema = (df_all_expanded['schema'].fillna('').str.len() > 0).sum()
print(f'Linhas com schema não-vazio: {n_with_schema:,}')
print(f'  Esperado ≈ {len(df_sc):,} (linhas do sql-create-context)')

print(f'\n── Crescimento do dataset ────────────────────────────────────')
print(f'  Spider (original):       {len(df_all):,}')
print(f'  sql-create-context:      {len(df_sc):,}')
print(f'  Total expandido:         {len(df_all_expanded):,}')
print(f'  Crescimento:             +{len(df_all_expanded)-len(df_all):,} '
      f'({(len(df_all_expanded)/len(df_all)-1)*100:.0f}%)')

print('\nPor fonte:')
print(df_all_expanded['source'].value_counts().to_string())
print('\nPor idioma × split:')
print(df_all_expanded.groupby(['lang', 'split']).size().unstack(fill_value=0).to_string())

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C31 — TOKENIZAÇÃO DO DATASET EXPANDIDO                                  ║
# ║                                                                          ║
# ║  Estratégia eficiente: tokens do Spider já foram calculados em C12.      ║
# ║  Só tokenizamos as linhas novas do sql-create-context.                   ║
# ║                                                                          ║
# ║  ATENÇÃO: df_all_expanded foi criado em C30 concatenando df_all e df_sc. ║
# ║  As colunas q_tokens e sql_tokens existem em df_all mas não em df_sc.    ║
# ║  Precisamos tokenizar apenas as linhas do sql-create-context.            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
needs_tokenization = df_all_expanded['q_tokens'].isna()
print(f'Linhas sem tokens: {needs_tokenization.sum():,} '
      f'(todas do sql-create-context)')

if needs_tokenization.sum() > 0:
    print('\n🔤 Tokenizando perguntas do sql-create-context...')
    tqdm.pandas(desc='  Perguntas')
    df_all_expanded.loc[needs_tokenization, 'q_tokens'] = \
        df_all_expanded[needs_tokenization].progress_apply(
            tokenize_row_question, axis=1
        )

    print('🔤 Tokenizando queries SQL do sql-create-context...')
    tqdm.pandas(desc='  Queries SQL')
    df_all_expanded.loc[needs_tokenization, 'sql_tokens'] = \
        df_all_expanded.loc[needs_tokenization, 'query'].progress_apply(
            tokenize_sql
        )
else:
    print('✅ Todos os tokens já calculados.')

df_all_expanded['q_len']   = df_all_expanded['q_tokens'].apply(len)
df_all_expanded['sql_len'] = df_all_expanded['sql_tokens'].apply(len)

before = len(df_all_expanded)
mask   = (
    (df_all_expanded['q_len']   <= CFG['max_src_len']) &
    (df_all_expanded['sql_len'] <= CFG['max_tgt_len'])
)
df_all_expanded = df_all_expanded[mask].reset_index(drop=True)
removed = before - len(df_all_expanded)

print(f'\nPares removidos por comprimento: {removed:,} ({removed/before*100:.1f}%)')
print(f'Pares restantes: {len(df_all_expanded):,}')
print('\nDistribuição final por idioma × split:')
print(df_all_expanded.groupby(['lang', 'split']).size().unstack(fill_value=0).to_string())

# Reconstruir vocabulários
print('\n📚 Reconstruindo vocabulários com dataset expandido...')

train_exp_mask    = df_all_expanded['split'] == 'train'
train_en_exp_mask = train_exp_mask & (df_all_expanded['lang'] == 'en')
train_pt_exp_mask = train_exp_mask & (df_all_expanded['lang'] == 'pt')

print(f'  Treino ALL: {train_exp_mask.sum():,} | '
      f'EN: {train_en_exp_mask.sum():,} | '
      f'PT: {train_pt_exp_mask.sum():,}')

src_vocab_exp    = Vocabulary('source_all_exp', min_freq=2)
src_vocab_en_exp = Vocabulary('source_en_exp',  min_freq=2)
# min_freq=1 para PT (poucos dados — não descartar nenhum token conhecido)
src_vocab_pt_exp = Vocabulary('source_pt_exp',  min_freq=1)
tgt_vocab_exp    = Vocabulary('target_sql_exp', min_freq=5)

src_vocab_exp.build_from_corpus(
    df_all_expanded.loc[train_exp_mask,    'q_tokens'].tolist())
src_vocab_en_exp.build_from_corpus(
    df_all_expanded.loc[train_en_exp_mask, 'q_tokens'].tolist())
src_vocab_pt_exp.build_from_corpus(
    df_all_expanded.loc[train_pt_exp_mask, 'q_tokens'].tolist())
tgt_vocab_exp.build_from_corpus(
    df_all_expanded.loc[train_exp_mask,    'sql_tokens'].tolist())

# NOTA: src_vocab_pt_exp usa min_freq=1 enquanto src_vocab_pt original usou
# min_freq=3. O vocabulário PT expandido é levemente maior, mas como os dados
# PT são idênticos entre Spider e Expandido, o impacto é mínimo.

print(f'\n── Cobertura dos vocabulários expandidos ────────────────────')
q_all = df_all_expanded['q_tokens'].tolist()
q_en  = df_all_expanded.loc[df_all_expanded['lang'] == 'en', 'q_tokens'].tolist()
q_pt  = df_all_expanded.loc[df_all_expanded['lang'] == 'pt', 'q_tokens'].tolist()
print(f'  src_vocab_exp    (ALL): {src_vocab_exp.coverage(q_all):.2f}%')
print(f'  src_vocab_en_exp  (EN): {src_vocab_en_exp.coverage(q_en):.2f}%')
print(f'  src_vocab_pt_exp  (PT): {src_vocab_pt_exp.coverage(q_pt):.2f}%')
print(f'  tgt_vocab_exp    (SQL): '
      f'{tgt_vocab_exp.coverage(df_all_expanded["sql_tokens"].tolist()):.2f}%')

vocab_dir = Path(CFG['vocab_dir'])
src_vocab_exp.save(str(vocab_dir    / 'src_vocab_all_exp.json'))
src_vocab_en_exp.save(str(vocab_dir / 'src_vocab_en_exp.json'))
src_vocab_pt_exp.save(str(vocab_dir / 'src_vocab_pt_exp.json'))
tgt_vocab_exp.save(str(vocab_dir    / 'tgt_vocab_exp.json'))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C31b — PERSISTÊNCIA DO DATASET EXPANDIDO (REPRODUTIBILIDADE)            ║
# ║                                                                          ║
# ║  Salva o dataset processado completo para que outros colaboradores       ║
# ║  não precisem re-executar C1–C31 para reproduzir os experimentos.        ║
# ║                                                                          ║
# ║  Arquivos gerados (para subir no GitHub ou Google Drive):                ║
# ║  ┌─────────────────────────────────────────────────────────────────┐     ║
# ║  │ data/df_all_expanded.parquet   → dataset completo tokenizado    │     ║
# ║  │ data/df_all_spider.parquet     → só Spider (referência)         │     ║
# ║  │ checkpoints/vocabs/*.json      → vocabulários (já salvos em C31)│     ║
# ║  │ requirements_data.txt          → versões das libs de dados      │     ║
# ║  └─────────────────────────────────────────────────────────────────┘     ║
# ║                                                                          ║
# ║  NOTA SOBRE TOKENS: colunas q_tokens e sql_tokens são listas Python.     ║
# ║  Parquet não suporta listas nativamente via pandas direto — salvamos     ║
# ║  como JSON string e restauramos com ast.literal_eval no carregamento.    ║
# ║                                                                          ║
# ║  PARA CARREGAR EM OUTRA SESSÃO: executar C31c (abaixo).                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import ast

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

def save_dataset(df: pd.DataFrame, path: str):
    df_save = df.copy()
    for col in ['q_tokens', 'sql_tokens']:
        if col in df_save.columns:
            df_save[col] = df_save[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False)
                if isinstance(x, list) else x
            )
    df_save.to_parquet(path, index=False)
    size_mb = Path(path).stat().st_size / 1e6
    print(f'  ✅ Salvo: {path} ({len(df_save):,} linhas, {size_mb:.1f} MB)')

print('💾 Salvando datasets processados...\n')
save_dataset(df_all_expanded, str(DATA_DIR / 'df_all_expanded.parquet'))
save_dataset(df_all,          str(DATA_DIR / 'df_all_spider.parquet'))

import datasets as hf_datasets
metadata = {
    'seed'              : SEED,
    'criado_em'         : pd.Timestamp.now().isoformat(),
    'total_pares'       : len(df_all_expanded),
    'spider_pares'      : len(df_all),
    'sql_ctx_pares'     : len(df_sc),
    'max_src_len'       : CFG['max_src_len'],
    'max_tgt_len'       : CFG['max_tgt_len'],
    'splits'            : df_all_expanded['split'].value_counts().to_dict(),
    'idiomas'           : df_all_expanded['lang'].value_counts().to_dict(),
    'fontes'            : df_all_expanded['source'].value_counts().to_dict(),
    'vocab_src_all_size': len(src_vocab_exp),
    'vocab_src_en_size' : len(src_vocab_en_exp),
    'vocab_src_pt_size' : len(src_vocab_pt_exp),
    'vocab_tgt_size'    : len(tgt_vocab_exp),
    'datasets_hf'       : ['xlangai/spider',
                            'Boakpe/spider-test-portuguese',
                            'b-mc2/sql-create-context'],
}
with open(DATA_DIR / 'dataset_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('  ✅ Metadata salva.')

reqs = [
    f'datasets=={hf_datasets.__version__}',
    f'pandas=={pd.__version__}',
    f'torch=={torch.__version__}',
    f'sqlparse=={sqlparse.__version__}',
    f'nltk=={nltk.__version__}',
]
with open(DATA_DIR / 'requirements_data.txt', 'w') as f:
    f.write('\n'.join(reqs))
print('  ✅ requirements_data.txt salvo.')

print('\n── Arquivos gerados ──────────────────────────────────────────')
for p in sorted(DATA_DIR.glob('*')):
    print(f'  {p.name:<40} {p.stat().st_size/1e6:.1f} MB')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C31c — CARREGAMENTO DO DATASET PRÉ-PROCESSADO (INÍCIO ALTERNATIVO)      ║
# ║                                                                          ║
# ║  Esta célula é um PONTO DE ENTRADA ALTERNATIVO para colaboradores        ║
# ║  que não querem re-executar todo o pipeline de dados (C1–C31).           ║
# ║                                                                          ║
# ║  PRÉ-REQUISITOS:                                                         ║
# ║  1. Executar C1 (instalação) e C2 (imports)                              ║
# ║  2. Executar C3 (SEED, DEVICE, CFG, tokens especiais)                    ║
# ║  3. Executar as definições de classe: Vocabulary (C13), Seq2Seq          ║
# ║     (C15–C18) e funções de treino (C19–C20)                              ║
# ║  4. Ter os arquivos em data/ e checkpoints/vocabs/ disponíveis           ║
# ║     (obtidos via git pull ou download manual)                            ║
# ║                                                                          ║
# ║  FLUXO NORMAL vs FLUXO ALTERNATIVO:                                      ║
# ║  Normal:      C1 → C2 → ... → C31 → C31b (salva) → C32+                  ║
# ║  Alternativo: C1 → C2 → C3 → [classes] → C31c (carrega) → C32+           ║
# ║  ATENÇÃO: NÃO executar esta célula se C27–C31b já foram executadas       ║
# ║  na mesma sessão — ela sobrescreveria df_all e df_all_expanded.          ║
# ║  Use apenas para retomar o projeto sem refazer o pipeline completo.      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# Guarda de segurança: só executa se df_all_expanded ainda não existir
if 'df_all_expanded' not in dir():
    import ast
    DATA_DIR  = Path('./data')
    vocab_dir = Path(CFG['vocab_dir'])

    def load_dataset_saved(path: str) -> pd.DataFrame:
        df = pd.read_parquet(path)
        for col in ['q_tokens', 'sql_tokens']:
            if col in df.columns:
                df[col] = df[col].apply(
                    lambda x: json.loads(x)
                    if isinstance(x, str) and x.startswith('[') else x
                )
        return df

    required = [
        DATA_DIR / 'df_all_spider.parquet',
        DATA_DIR / 'df_all_expanded.parquet',
        vocab_dir / 'src_vocab_all_exp.json',
        vocab_dir / 'src_vocab_en_exp.json',
        vocab_dir / 'src_vocab_pt_exp.json',
        vocab_dir / 'tgt_vocab_exp.json',
        vocab_dir / 'src_vocab_all.json',
        vocab_dir / 'src_vocab_en.json',
        vocab_dir / 'src_vocab_pt.json',
        vocab_dir / 'tgt_vocab.json',
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        print('❌ Arquivos faltando — execute C1–C31b primeiro:')
        for m in missing:
            print(f'   {m}')
        raise FileNotFoundError('Dados pré-processados não encontrados.')

    print('📂 Carregando datasets pré-processados...')
    df_all          = load_dataset_saved(str(DATA_DIR / 'df_all_spider.parquet'))
    df_all_expanded = load_dataset_saved(str(DATA_DIR / 'df_all_expanded.parquet'))
    print(f'  ✅ Spider:    {len(df_all):,} pares')
    print(f'  ✅ Expandido: {len(df_all_expanded):,} pares')

    print('\n📚 Carregando vocabulários...')
    src_vocab        = Vocabulary.load(str(vocab_dir / 'src_vocab_all.json'))
    src_vocab_en     = Vocabulary.load(str(vocab_dir / 'src_vocab_en.json'))
    src_vocab_pt     = Vocabulary.load(str(vocab_dir / 'src_vocab_pt.json'))
    tgt_vocab        = Vocabulary.load(str(vocab_dir / 'tgt_vocab.json'))
    src_vocab_exp    = Vocabulary.load(str(vocab_dir / 'src_vocab_all_exp.json'))
    src_vocab_en_exp = Vocabulary.load(str(vocab_dir / 'src_vocab_en_exp.json'))
    src_vocab_pt_exp = Vocabulary.load(str(vocab_dir / 'src_vocab_pt_exp.json'))
    tgt_vocab_exp    = Vocabulary.load(str(vocab_dir / 'tgt_vocab_exp.json'))

    with open(DATA_DIR / 'dataset_metadata.json', encoding='utf-8') as f:
        meta = json.load(f)
    print(f'\n📋 Criado em {meta["criado_em"][:19]} | '
          f'{meta["total_pares"]:,} pares | SEED={meta["seed"]}')
    print('✅ Pronto para C32.')
else:
    print('✅ df_all_expanded já existe na sessão — C31c ignorada.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C32 — DATALOADERS EXPANDIDOS E RE-TREINO COMPARATIVO                    ║
# ║                                                                          ║
# ║  Treina modelos ALL e EN com Spider + sql-create-context.                ║
# ║  PT permanece inalterado (sql-create-context é só EN).                   ║
# ║                                                                          ║
# ║  Checkpoints com sufixo '_exp' para não sobrescrever os originais.       ║
# ║  Isso permite comparação direta na C33.                                  ║
# ║                                                                          ║
# ║  CORREÇÃO 1: val_lang do modelo ALL expandido alterado para 'en'.        ║
# ║  Razão: df_val_all_exp tem ~11k EN e ~321 PT. Com n_samples=20 o         ║
# ║  compute_bleu_sample sorteia quase sempre perguntas EN. Usar val_lang    ║
# ║  ='en' garante que o tokenizador NLTK correto é usado e o BLEU é         ║
# ║  calculado sobre o idioma majoritário da amostra.                        ║
# ║                                                                          ║
# ║  CORREÇÃO 2: CFG_ALL_OVERRIDE usado consistentemente (igual à C21).      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print('══ Construindo DataLoaders expandidos ═══════════════════════')

print('\n▶ Loaders ALL expandido (Spider + sql-create-context):')
loaders_all_exp = build_loaders(
    df_all_expanded, src_vocab_exp, tgt_vocab_exp, CFG_ALL_OVERRIDE, lang_filter=None
)

print('\n▶ Loaders EN expandido:')
loaders_en_exp = build_loaders(
    df_all_expanded, src_vocab_en_exp, tgt_vocab_exp, CFG_EN_OVERRIDE, lang_filter='en'
)

print('\n▶ Loaders PT expandido:')
loaders_pt_exp = build_loaders(
    df_all_expanded, src_vocab_pt_exp, tgt_vocab_exp, CFG, lang_filter='pt'
)

print('\n── Verificação final ──────────────────────────────────────────')
for lang in ['en', 'pt']:
    for split in ['train', 'validation', 'test']:
        n = len(df_all_expanded[
            (df_all_expanded['lang'] == lang) &
            (df_all_expanded['split'] == split)
        ])
        status = '✅' if n > 0 else '⚠️  vazio'
        print(f'  [{lang}] {split:12s}: {n:>7,} amostras  {status}')
print('\n✅ DataLoaders expandidos prontos.')

# ── Re-treino ALL expandido ───────────────────────────────────────────────
histories_exp = {}

print('\n' + '═' * 65)
print('▶ Re-treino ALL expandido (Spider + sql-create-context)')
print(f'  Vocab src: {len(src_vocab_exp):,} tokens | '
      f'Vocab tgt: {len(tgt_vocab_exp):,} tokens')

model_all_exp = make_fresh_model(vocab_src=src_vocab_exp,
                                 cfg=CFG_ALL_OVERRIDE,
                                 vocab_tgt=tgt_vocab_exp)
opt_all_exp   = optim.Adam(model_all_exp.parameters(),
                            lr=CFG_ALL_OVERRIDE['learning_rate'])
sch_all_exp   = optim.lr_scheduler.ReduceLROnPlateau(
    opt_all_exp,
    patience = CFG_ALL_OVERRIDE['lr_patience'],
    factor   = CFG_ALL_OVERRIDE['lr_factor'],
)
df_val_all_exp = df_all_expanded[df_all_expanded['split'] == 'validation']

histories_exp['all_exp'] = train_model(
    model_all_exp,
    loaders_all_exp['train'],
    loaders_all_exp['validation'],
    opt_all_exp, sch_all_exp, criterion,
    cfg           = CFG_ALL_OVERRIDE,
    tag           = 'all_exp',
    df_val_sample = df_val_all_exp,
    val_lang      = 'en',   # dataset expandido é 99% EN
    warmup_epochs = 3,
)

# ── Re-treino EN expandido ────────────────────────────────────────────────
print('\n' + '═' * 65)
print('▶ Re-treino EN expandido (Spider EN + sql-create-context)')
print(f'  Vocab src: {len(src_vocab_en_exp):,} tokens | '
      f'Vocab tgt: {len(tgt_vocab_exp):,} tokens')

model_en_exp = make_fresh_model(vocab_src=src_vocab_en_exp,
                                cfg=CFG_EN_OVERRIDE,
                                vocab_tgt=tgt_vocab_exp)
opt_en_exp   = optim.Adam(model_en_exp.parameters(),
                           lr=CFG_EN_OVERRIDE['learning_rate'])
sch_en_exp   = optim.lr_scheduler.ReduceLROnPlateau(
    opt_en_exp,
    patience = CFG_EN_OVERRIDE['lr_patience'],
    factor   = CFG_EN_OVERRIDE['lr_factor'],
)
df_val_en_exp = df_all_expanded[
    (df_all_expanded['split'] == 'validation') &
    (df_all_expanded['lang']  == 'en')
]

histories_exp['en_exp'] = train_model(
    model_en_exp,
    loaders_en_exp['train'],
    loaders_en_exp['validation'],
    opt_en_exp, sch_en_exp, criterion,
    cfg           = CFG_EN_OVERRIDE,
    tag           = 'en_exp',
    df_val_sample = df_val_en_exp,
    val_lang      = 'en',
)

# ── Re-treino PT expandido ────────────────────────────────────────────────
print('\n' + '═' * 65)
print('▶ PT com tgt_vocab expandido (Spider PT — dados inalterados)')
print(f'  Vocab src: {len(src_vocab_pt_exp):,} tokens | '
      f'Vocab tgt: {len(tgt_vocab_exp):,} tokens')
print('  Nota: dados PT são idênticos ao Spider original.')
print('  Este modelo serve de controle para verificar que a mudança')
print('  de tgt_vocab não afeta o PT de forma significativa.')

model_pt_exp = make_fresh_model(vocab_src=src_vocab_pt_exp,
                                cfg=CFG,
                                vocab_tgt=tgt_vocab_exp)
opt_pt_exp   = optim.Adam(model_pt_exp.parameters(), lr=CFG['learning_rate'])
sch_pt_exp   = optim.lr_scheduler.ReduceLROnPlateau(
    opt_pt_exp,
    patience = CFG['lr_patience'],
    factor   = CFG['lr_factor'],
)
df_val_pt_exp = df_all_expanded[
    (df_all_expanded['split'] == 'validation') &
    (df_all_expanded['lang']  == 'pt')
]

histories_exp['pt_exp'] = train_model(
    model_pt_exp,
    loaders_pt_exp['train'],
    loaders_pt_exp['validation'],
    opt_pt_exp, sch_pt_exp, criterion,
    cfg           = CFG,
    tag           = 'pt_exp',
    df_val_sample = df_val_pt_exp,
    val_lang      = 'pt',
)

# ── Resumo pós-treino expandido ───────────────────────────────────────────
print('\n' + '═' * 65)
print('── Resumo dos modelos expandidos treinados ──────────────────────')
key_labels_exp = {
    'all_exp': 'ALL (Spider + sql-ctx)',
    'en_exp' : 'EN  (Spider + sql-ctx)',
    'pt_exp' : 'PT  (Spider — controle)',
}
for key, hist in histories_exp.items():
    print(f'\n  [{key_labels_exp[key]}]')
    print(f'    Épocas:          {len(hist["val_loss"])}')
    print(f'    Melhor val loss: {min(hist["val_loss"]):.4f}')
    print(f'    Melhor val ppl:  {min(hist["val_ppl"]):.2f}')
    best_b = max(hist['bleu']) if any(b > 0 for b in hist['bleu']) else 0.0
    print(f'    Melhor BLEU-1:   {best_b:.2f}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C33 — COMPARAÇÃO: SPIDER vs SPIDER + SQL-CREATE-CONTEXT                 ║
# ║                                                                          ║
# ║  Compara loss e perplexidade antes e depois da expansão do dataset.      ║
# ║  Linhas tracejadas = modelos originais (Spider apenas)                   ║
# ║  Linhas sólidas = modelos expandidos (Spider + sql-create-context)       ║
# ║                                                                          ║
# ║  CORREÇÕES aplicadas:                                                    ║
# ║  1. Plots salvos em PLOTS_DIR.                                           ║
# ║  2. Adicionado subplot de BLEU-1 comparativo (monitoramento).            ║
# ║  3. Avaliação BLEU-1 no TESTE real para modelos expandidos, usando       ║
# ║     a mesma função avaliar_bleu_teste() da C24 — comparação justa        ║
# ║     com os modelos Spider originais.                                     ║
# ║  4. Tabela final inclui coluna BLEU-1 teste para todos os 6 modelos.     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Carregar modelos expandidos dos checkpoints ───────────────────────────
print('📂 Carregando modelos expandidos para avaliação no teste...')

model_all_exp_loaded = load_model_from_checkpoint(
    './checkpoints/best_all_exp.pt',
    vocab_src = src_vocab_exp,
    vocab_tgt = tgt_vocab_exp,
    cfg       = CFG_ALL_OVERRIDE,
    device    = DEVICE,
)
model_en_exp_loaded = load_model_from_checkpoint(
    './checkpoints/best_en_exp.pt',
    vocab_src = src_vocab_en_exp,
    vocab_tgt = tgt_vocab_exp,
    cfg       = CFG_EN_OVERRIDE,
    device    = DEVICE,
)
model_pt_exp_loaded = load_model_from_checkpoint(
    './checkpoints/best_pt_exp.pt',
    vocab_src = src_vocab_pt_exp,
    vocab_tgt = tgt_vocab_exp,
    cfg       = CFG,
    device    = DEVICE,
)

# ── BLEU-1 no teste real para modelos expandidos ──────────────────────────
# Usando os mesmos conjuntos de teste da C24 para comparação justa.
# Os modelos expandidos foram treinados com tgt_vocab_exp (min_freq=5),
# que pode não conter tokens raros. O BLEU-1 no teste mede a qualidade
# real de geração, independente de qual vocabulário foi usado no treino.
print('\n── BLEU-1 no conjunto de teste real (n=200) ─────────────────────')

df_test_all_exp = df_all_expanded[df_all_expanded['split'] == 'test']
df_test_en_exp  = df_all_expanded[
    (df_all_expanded['split'] == 'test') &
    (df_all_expanded['lang']  == 'en')
]
df_test_pt_exp  = df_all_expanded[
    (df_all_expanded['split'] == 'test') &
    (df_all_expanded['lang']  == 'pt')
]

bleu_all_exp_test, _, _ = avaliar_bleu_teste(
    model_all_exp_loaded, df_test_all_exp, lang='en', n_samples=200)
bleu_en_exp_test,  _, _ = avaliar_bleu_teste(
    model_en_exp_loaded,  df_test_en_exp,  lang='en', n_samples=200)
bleu_pt_exp_test,  _, _ = avaliar_bleu_teste(
    model_pt_exp_loaded,  df_test_pt_exp,  lang='pt', n_samples=200)

print(f'  ALL expandido (teste ALL): {bleu_all_exp_test:.2f}')
print(f'  EN  expandido (teste EN):  {bleu_en_exp_test:.2f}')
print(f'  PT  expandido (teste PT):  {bleu_pt_exp_test:.2f}')

# ── Figura 7: Loss + PPL + BLEU — Spider vs Expandido ────────────────────
fig7, axes7 = plt.subplots(1, 3, figsize=(22, 6))

plot_configs = [
    ('ALL Spider',    histories.get('all', {}),         'purple',    '--'),
    ('ALL Expandido', histories_exp.get('all_exp', {}), 'purple',    '-'),
    ('EN Spider',     histories.get('en', {}),          'steelblue', '--'),
    ('EN Expandido',  histories_exp.get('en_exp', {}),  'steelblue', '-'),
    ('PT Spider',     histories.get('pt', {}),          'seagreen',  '--'),
    ('PT Expandido',  histories_exp.get('pt_exp', {}),  'seagreen',  '-'),
]

for label, hist, color, style in plot_configs:
    if not hist or 'val_loss' not in hist:
        continue
    epochs = range(1, len(hist['val_loss']) + 1)
    lw = 2 if style == '-' else 1.5

    axes7[0].plot(epochs, hist['val_loss'], c=color, linestyle=style,
                  label=label, alpha=0.85, linewidth=lw)
    axes7[1].plot(epochs, hist['val_ppl'],  c=color, linestyle=style,
                  label=label, alpha=0.85, linewidth=lw)

    # BLEU-1 de monitoramento (calculado durante treino)
    bleu_epochs = [e for e, b in zip(epochs, hist['bleu']) if b > 0]
    bleu_vals   = [b for b in hist['bleu'] if b > 0]
    if bleu_vals:
        axes7[2].plot(bleu_epochs, bleu_vals, c=color, linestyle=style,
                      marker='o', markersize=4, label=label, linewidth=lw)

axes7[0].set_title('Val Loss\n(tracejado = Spider; sólido = Expandido)')
axes7[0].set_xlabel('Época')
axes7[0].set_ylabel('Validation Loss')
axes7[0].legend(fontsize=7)

axes7[1].set_title('Val Perplexidade (↓ melhor)')
axes7[1].set_xlabel('Época')
axes7[1].set_ylabel('Perplexidade')
axes7[1].legend(fontsize=7)

axes7[2].set_title('BLEU-1 durante validação\n(monitoramento de treino)')
axes7[2].set_xlabel('Época')
axes7[2].set_ylabel('BLEU-1 (↑ melhor)')
axes7[2].legend(fontsize=7)

plt.suptitle('Seção 6 — Impacto do sql-create-context no Treinamento',
             fontsize=13)
plt.tight_layout()
fig7.savefig(f'{PLOTS_DIR}/fig7_spider_vs_expandido_loss_ppl_bleu.png',
             dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Salvo: {PLOTS_DIR}/fig7_spider_vs_expandido_loss_ppl_bleu.png')

# ── Figura 8: BLEU-1 no teste (barras lado a lado) ───────────────────────
fig8, ax8 = plt.subplots(figsize=(11, 5))

modelos_bar  = ['ALL\nSpider', 'ALL\nExpandido',
                 'EN\nSpider',  'EN\nExpandido',
                 'PT\nSpider',  'PT\nExpandido']
bleus_bar    = [
    bleu_all_test,     bleu_all_exp_test,
    bleu_en_test,      bleu_en_exp_test,
    bleu_pt_test,      bleu_pt_exp_test,
]
cores_bar    = ['#9C7FC8', '#6A3D9A',   # purple — ALL
                '#7DB7D8', '#2171B5',   # blue   — EN
                '#74C69D', '#1A7741']   # green  — PT

bars8 = ax8.bar(modelos_bar, bleus_bar, color=cores_bar,
                edgecolor='white', width=0.6)
for bar, val in zip(bars8, bleus_bar):
    ax8.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.3,
             f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

ax8.set_title('BLEU-1 no Conjunto de Teste Real\n'
              '(Spider vs Spider + sql-create-context)', fontsize=12)
ax8.set_ylabel('BLEU-1 (↑ melhor)')
ax8.set_ylim(0, max(bleus_bar) * 1.25 if bleus_bar else 1)
# Linha separando Spider (barras claras) de Expandido (barras escuras)
ax8.axvline(0.5, color='gray', linestyle=':', alpha=0.4)
ax8.axvline(2.5, color='gray', linestyle=':', alpha=0.4)
ax8.axvline(4.5, color='gray', linestyle=':', alpha=0.4)
plt.tight_layout()
fig8.savefig(f'{PLOTS_DIR}/fig8_bleu_teste_spider_vs_expandido.png',
             dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Salvo: {PLOTS_DIR}/fig8_bleu_teste_spider_vs_expandido.png')

# ── Tabela comparativa completa ───────────────────────────────────────────
print('\n── Tabela comparativa: Spider vs Expandido ───────────────────────')
tabela = []
configs_tabela = [
    ('ALL — Spider',         histories.get('all', {}),         'all',     False),
    ('ALL — Spider+sql-ctx', histories_exp.get('all_exp', {}), 'all_exp', True),
    ('EN  — Spider',         histories.get('en', {}),          'en',      False),
    ('EN  — Spider+sql-ctx', histories_exp.get('en_exp', {}),  'en_exp',  True),
    ('PT  — Spider',         histories.get('pt', {}),          'pt',      False),
    ('PT  — Spider+sql-ctx', histories_exp.get('pt_exp', {}),  'pt_exp',  True),
]
bleu_teste_map = {
    'all':     bleu_all_test,     'all_exp': bleu_all_exp_test,
    'en':      bleu_en_test,      'en_exp':  bleu_en_exp_test,
    'pt':      bleu_pt_test,      'pt_exp':  bleu_pt_exp_test,
}

for label, hist, key, is_exp in configs_tabela:
    if not hist or 'val_loss' not in hist:
        continue
    base   = key.replace('_exp', '')
    df_ref = df_all_expanded if is_exp else df_all
    n = (len(df_ref[df_ref['split'] == 'train']) if base == 'all'
         else len(df_ref[(df_ref['split'] == 'train') &
                          (df_ref['lang'] == base)]))
    best_b_val = (max(hist['bleu'])
                  if any(b > 0 for b in hist['bleu']) else 0.0)
    tabela.append({
        'Modelo'        : label,
        'N treino'      : f'{n:,}',
        'Épocas'        : len(hist['val_loss']),
        'Val Loss'      : f'{min(hist["val_loss"]):.4f}',
        'Val PPL'       : f'{min(hist["val_ppl"]):.1f}',
        'BLEU-1 (val)'  : f'{best_b_val:.1f}',
        'BLEU-1 (teste)': f'{bleu_teste_map.get(key, 0.0):.1f}',
    })

df_tabela = pd.DataFrame(tabela)
print(df_tabela.to_string(index=False))

# Salvar tabela como figura
fig9, ax9 = plt.subplots(figsize=(14, 3.5))
ax9.axis('off')
tbl9 = ax9.table(
    cellText  = df_tabela.values,
    colLabels = df_tabela.columns,
    cellLoc   = 'center', loc='center',
)
tbl9.auto_set_font_size(False)
tbl9.set_fontsize(9)
tbl9.scale(1.1, 2.2)
for j in range(len(df_tabela.columns)):
    tbl9[0, j].set_facecolor('#DDDDEE')
    tbl9[0, j].set_text_props(fontweight='bold')
# Destacar linhas dos modelos expandidos (ímpares)
for i in range(1, len(df_tabela) + 1):
    if i % 2 == 0:  # expandidos (linhas pares na tabela = índice ímpar)
        for j in range(len(df_tabela.columns)):
            tbl9[i, j].set_facecolor('#F0F4FF')
plt.title('Spider vs Spider + sql-create-context — Comparação Completa',
          fontsize=11, pad=12)
plt.tight_layout()
fig9.savefig(f'{PLOTS_DIR}/fig9_tabela_spider_vs_expandido.png',
             dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ {PLOTS_DIR}/fig9_tabela_spider_vs_expandido.png salvo.')
print('\n── Plots salvos nesta seção ──────────────────────────────────────')
for p in sorted(Path(PLOTS_DIR).glob('fig[6-9]*.png')):
    print(f'  {p.name}')
print('\n✅ Comparação concluída. Avançando para Fine-Tuning (Seção 7).')

## 🤖 PARTE 7 — Fine-Tuning com T5

O modelo Seq2Seq treinado do zero tem limitações claras quando o dataset é pequeno (~1k amostras em PT). Uma abordagem mais robusta é utilizar **fine-tuning** em um modelo pré-treinado como o T5, que já possui representações ricas de linguagem natural.

### Por que T5?
- Arquitetura encoder-decoder nativa (ideal para text-to-SQL)
- Pré-treinado em grandes corpora multilíngues
- Existem checkpoints já fine-tunados em SQL (ex: `mrm8488/t5-base-finetuned-wikiSQL`)

### Experimentos nesta seção:
1. **Fine-tuning sem schema** — pergunta → SQL
2. **Fine-tuning com schema** — [schema do banco] pergunta → SQL
3. **Comparação** com e sem schema

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C34 — SCHEMAS E DATASET CLASSE T5SQLDataset                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from transformers import (
    T5ForConditionalGeneration, T5TokenizerFast,
    MT5ForConditionalGeneration, AutoTokenizer,
)
import gc

# ── Funções de inferência de schema ───────────────────────────────────────
def extract_tables_from_sql(sql: str) -> List[str]:
    return list(dict.fromkeys(re.findall(
        r'(?:FROM|JOIN)\s+([A-Z_][A-Z0-9_]*)', sql.upper()
    )))

def extract_columns_from_sql(sql: str) -> List[str]:
    m = re.search(r'SELECT\s+(.+?)\s+FROM', sql.upper(), re.DOTALL)
    if not m:
        return []
    sql_kw = {'SELECT','FROM','WHERE','JOIN','GROUP','ORDER','HAVING',
               'COUNT','SUM','AVG','MAX','MIN','DISTINCT','AS','T1','T2','T3'}
    return [c for c in re.findall(r'[A-Z_][A-Z0-9_.]*', m.group(1))
            if c not in sql_kw]

def build_schema_prompt(tables: List[str], columns: List[str]) -> str:
    if not tables:
        return ''
    parts = []
    for t in tables:
        cols = [c for c in columns if not c.startswith(t + '.')]
        parts.append(f'{t.lower()} ( {", ".join(cols[:8]).lower() or "..."} )')
    return 'schema: ' + ' | '.join(parts)

def infer_schema_from_query(row) -> str:
    return build_schema_prompt(
        extract_tables_from_sql(row['query']),
        extract_columns_from_sql(row['query']),
    )

# ── Inferir schemas em df_all (Spider PT+EN) ──────────────────────────────
print('⚙️  Inferindo schemas em df_all...')
if 'schema' not in df_all.columns:
    df_all['schema'] = ''
need = df_all['schema'].fillna('') == ''
if need.any():
    tqdm.pandas(desc='  df_all schemas')
    df_all.loc[need, 'schema'] = df_all[need].progress_apply(
        infer_schema_from_query, axis=1
    )
print(f'  ✅ df_all: {need.sum():,} schemas inferidos')

# ── Inferir schemas Spider em df_all_expanded ─────────────────────────────
print('⚙️  Inferindo schemas Spider em df_all_expanded...')
spider_mask = df_all_expanded['source'].isin(
    ['xlangai/spider', 'Boakpe/spider-test-portuguese', 'c4ai']
)
if 'schema' not in df_all_expanded.columns:
    df_all_expanded['schema'] = ''
need_exp = spider_mask & (df_all_expanded['schema'].fillna('') == '')
if need_exp.any():
    tqdm.pandas(desc='  df_all_expanded schemas Spider')
    df_all_expanded.loc[need_exp, 'schema'] = \
        df_all_expanded[need_exp].progress_apply(infer_schema_from_query, axis=1)
print(f'  ✅ df_all_expanded Spider: {need_exp.sum():,} schemas inferidos')
print(f'  ✅ sql-create-context mantém schemas CREATE TABLE reais: '
      f'{(df_all_expanded["source"]=="sql-create-context").sum():,} linhas')

# ── Criar question_with_schema nos dois datasets ──────────────────────────
for _df, _name in [(df_all, 'df_all'), (df_all_expanded, 'df_all_expanded')]:
    if 'question_with_schema' not in _df.columns:
        _df['question_with_schema'] = _df.apply(
            lambda r: (f"{r['schema']} question: {r['question']}"
                       if r.get('schema', '') else f"question: {r['question']}"),
            axis=1
        )
        print(f'  ✅ {_name}: question_with_schema criada')
    else:
        # Garantir que está atualizada caso schema tenha mudado
        _df['question_with_schema'] = _df.apply(
            lambda r: (f"{r['schema']} question: {r['question']}"
                       if r.get('schema', '') else f"question: {r['question']}"),
            axis=1
        )
        print(f'  ✅ {_name}: question_with_schema atualizada')

# ── Verificação ────────────────────────────────────────────────────────────
print('\n── Tamanho dos subsets ──────────────────────────────────────────')
for split in ['train', 'validation', 'test']:
    npt  = len(df_all[(df_all['split']==split) & (df_all['lang']=='pt')])
    nen  = len(df_all[(df_all['split']==split) & (df_all['lang']=='en')])
    nexp = len(df_all_expanded[(df_all_expanded['split']==split) &
                                (df_all_expanded['lang']=='en')])
    print(f'  {split:12s}: PT+C4AI={npt:,} | EN Spider={nen:,} | EN Expandido={nexp:,}')

print('\n── Exemplos de schemas ──────────────────────────────────────────')
ex_pt = df_all[(df_all['lang']=='pt') & (df_all['schema'].fillna('')!='')].iloc[0]
print(f'[Spider PT] question: {ex_pt["question"]}')
print(f'            schema:   {ex_pt["schema"]}')
ex_en = df_all[(df_all['lang']=='en') & (df_all['schema'].fillna('')!='')].iloc[0]
print(f'[Spider EN] question: {ex_en["question"]}')
print(f'            schema:   {ex_en["schema"]}')
ex_sc = df_all_expanded[df_all_expanded['source']=='sql-create-context'].iloc[0]
print(f'[sql-ctx  ] question: {ex_sc["question"]}')
print(f'            schema:   {ex_sc["schema"][:100]}...')

# ── Classe Dataset T5 ─────────────────────────────────────────────────────
class T5SQLDataset(Dataset):
    """Dataset genérico para fine-tuning T5/mT5/PTT5 com suporte a schema."""

    def __init__(self, df: pd.DataFrame, tokenizer,
                 max_src: int, max_tgt: int, use_schema: bool):
        q_col = 'question_with_schema' if use_schema else 'question'
        if use_schema and q_col not in df.columns:
            raise ValueError(f"Coluna '{q_col}' ausente. Execute C34 primeiro.")
        self.inputs  = ('translate to SQL: ' +
                         df[q_col].fillna(df['question'])).tolist()
        self.targets = df['query'].tolist()
        self.tok     = tokenizer
        self.max_src = max_src
        self.max_tgt = max_tgt

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        enc = self.tok(self.inputs[idx], max_length=self.max_src,
                        truncation=True, padding='max_length', return_tensors='pt')
        dec = self.tok(self.targets[idx], max_length=self.max_tgt,
                        truncation=True, padding='max_length', return_tensors='pt')
        lbl = dec['input_ids'].squeeze().clone()
        lbl[lbl == self.tok.pad_token_id] = -100
        return {
            'input_ids'     : enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels'        : lbl,
        }

print('\n✅ C34 concluída — schemas prontos e T5SQLDataset definido.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C35 — REGISTRO DE MODELOS E FUNÇÕES GENÉRICAS                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Configuração dos modelos base ─────────────────────────────────────────
MODEL_CONFIGS = {
    't5sql': {
        'hf_id'  : 'mrm8488/t5-base-finetuned-wikiSQL',
        'cls'    : T5ForConditionalGeneration,
        'tok_cls': T5TokenizerFast,
        'label'  : 'T5-SQL',
        'color'  : '#E24B4A',
    },
    'mt5': {
        'hf_id'  : 'google/mt5-base',
        'cls'    : MT5ForConditionalGeneration,
        'tok_cls': AutoTokenizer,
        'label'  : 'mT5',
        'color'  : '#378ADD',
    },
    'ptt5': {
        'hf_id'  : 'unicamp-dl/ptt5-base-portuguese-vocab',
        'cls'    : T5ForConditionalGeneration,
        'tok_cls': AutoTokenizer,
        'label'  : 'PTT5',
        'color'  : '#1D9E75',
    },
}

# Registro global de resultados — preenchido pelos treinos C37–C39
ALL_RESULTS: Dict[str, dict] = {}


# ── Funções de treino e avaliação (epoch-level) ───────────────────────────

def _train_epoch(model, loader, optimizer) -> float:
    model.train()
    total, n = 0.0, 0
    bar = tqdm(loader, desc='  Treino', leave=False, ncols=92)
    for i, batch in enumerate(bar):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        loss = model(input_ids=ids, attention_mask=mask, labels=lbl).loss
        if torch.isnan(loss) or torch.isinf(loss):
            continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item(); n += 1
        bar.set_postfix({'loss': f'{total/n:.4f}'})
    return total / max(n, 1)


@torch.no_grad()
def _eval_epoch(model, loader) -> Tuple[float, float]:
    model.eval()
    total, n = 0.0, 0
    for batch in tqdm(loader, desc='  Val', leave=False, ncols=92):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)
        total += model(input_ids=ids, attention_mask=mask, labels=lbl).loss.item()
        n += 1
    avg = total / max(n, 1)
    return avg, float(np.exp(min(avg, 10)))


@torch.no_grad()
def _generate(model, tokenizer, question: str, schema: str = '') -> str:
    model.eval()
    txt = (f'translate to SQL: {schema} question: {question}'
           if schema else f'translate to SQL: {question}')
    enc = tokenizer(txt, return_tensors='pt',
                    max_length=CFG['ft_max_src_len'], truncation=True).to(DEVICE)
    ids = model.generate(
        input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
        max_length=CFG['ft_max_tgt_len'], num_beams=4, early_stopping=True,
    )
    return tokenizer.decode(ids[0], skip_special_tokens=True)


@torch.no_grad()
def _bleu_eval(model, tokenizer, df_sub: pd.DataFrame,
               use_schema: bool, n: int, order: int) -> float:
    """Calcula BLEU-N em amostra. Usado para monitoramento e avaliação final."""
    model.eval()
    if len(df_sub) == 0:
        return 0.0
    sub = df_sub.sample(min(n, len(df_sub)), random_state=SEED)
    hyps, refs = [], []
    for _, row in sub.iterrows():
        sch = str(row.get('schema', '')) if use_schema else ''
        try:
            pred = _generate(model, tokenizer, row['question'], schema=sch)
        except Exception:
            pred = ''
        if pred.strip():
            hyps.append(pred.lower().strip())
            refs.append(row['query'].lower().strip())
    if not hyps:
        return 0.0
    try:
        return SacreBLEU(tokenize='13a', max_ngram_order=order,
                         smooth_method='exp', effective_order=True
                         ).corpus_score(hyps, [refs]).score
    except Exception:
        return 0.0


# ── Função principal de treino ────────────────────────────────────────────
def train_one_model(
    model_key  : str,
    tag        : str,
    loaders    : Dict[str, DataLoader],
    df_val_mon : pd.DataFrame,
    df_test    : pd.DataFrame,
    use_schema : bool,
    description: str,
) -> None:
    """
    Treina uma variante (t5sql/mt5/ptt5), avalia no teste e salva em ALL_RESULTS.
    Carrega e descarrega o modelo para minimizar uso de RAM entre experimentos.
    """
    cfg_m = MODEL_CONFIGS[model_key]
    print(f'\n{"═"*72}')
    print(f'▶  {description}')
    print(f'   Modelo: {cfg_m["hf_id"]}')
    print(f'   Schema: {"SIM" if use_schema else "NÃO"} | '
          f'Treino: {len(loaders["train"].dataset):,} amostras')
    print(f'{"═"*72}')

    if not loaders.get('train'):
        print('  ⚠️  Loader de treino vazio — experimento ignorado.')
        return

    # Carregar modelo e tokenizer
    try:
        tokenizer = cfg_m['tok_cls'].from_pretrained(cfg_m['hf_id'])
        model     = cfg_m['cls'].from_pretrained(cfg_m['hf_id']).to(DEVICE)
    except Exception as e:
        print(f'  ❌ Falha ao carregar: {e}')
        return

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Parâmetros treináveis: {n_params:,}')

    optimizer = optim.AdamW(model.parameters(),
                             lr=CFG['ft_lr'], weight_decay=0.01)
    ckpt_path = f"{CFG['checkpoint_dir']}/best_{tag}.pt"
    history   = {'train_loss': [], 'val_loss': [], 'val_ppl': [], 'bleu1_val': []}
    best_val  = float('inf')
    patience  = 0
    schema_str = 'COM schema' if use_schema else 'SEM schema'

    print(f'  🚀 [{tag}] | {CFG["ft_epochs"]} épocas | {schema_str}')
    print('  ' + '─' * 68)

    for epoch in range(1, CFG['ft_epochs'] + 1):
        tr_loss          = _train_epoch(model, loaders['train'], optimizer)
        vl_loss, vl_ppl  = _eval_epoch(model, loaders.get('validation',
                                                            loaders['train']))
        b1_val = _bleu_eval(model, tokenizer, df_val_mon,
                             use_schema, n=30, order=1)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['val_ppl'].append(vl_ppl)
        history['bleu1_val'].append(b1_val)

        is_best = vl_loss < best_val
        if is_best:
            best_val = vl_loss
            patience = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience += 1

        b_str = f'| B1={b1_val:.2f} ' if b1_val > 0 else ''
        flag  = '  ← ✅' if is_best else f'  ({patience}/{CFG["patience"]})'
        print(f'  Ep {epoch:02d}/{CFG["ft_epochs"]} | '
              f'Train={tr_loss:.4f} | Val={vl_loss:.4f} '
              f'(ppl={vl_ppl:.1f}) {b_str}{flag}')

        if patience >= CFG['patience']:
            print(f'  ⏹️  Early stopping na época {epoch}.')
            break

    # Recarregar melhor checkpoint e avaliar no teste
    model.load_state_dict(
        torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    )
    model.eval()
    print(f'\n  📊 Avaliando no teste (n=200)...')
    b1_test = _bleu_eval(model, tokenizer, df_test, use_schema, n=200, order=1)
    b4_test = _bleu_eval(model, tokenizer, df_test, use_schema, n=200, order=4)
    best_ppl = min(history['val_ppl'])

    ALL_RESULTS[tag] = {
        'description': description,
        'model_key'  : model_key,
        'label'      : cfg_m['label'],
        'color'      : cfg_m['color'],
        'use_schema' : use_schema,
        'history'    : history,
        'bleu1_test' : b1_test,
        'bleu4_test' : b4_test,
        'best_ppl'   : best_ppl,
        'best_val'   : best_val,
        'epochs'     : len(history['val_loss']),
        'n_train'    : len(loaders['train'].dataset),
        # Manter modelo e tokenizer em CPU para inferência posterior
        'model_cpu'  : model.cpu(),
        'tokenizer'  : tokenizer,
    }

    print(f'  ✅ [{tag}] BLEU-1={b1_test:.2f} | BLEU-4={b4_test:.2f} | '
          f'PPL={best_ppl:.1f}')

    # Liberar GPU
    if DEVICE.type == 'cuda':
        gc.collect()
        torch.cuda.empty_cache()


# ── Helper: mover modelo para inferência ─────────────────────────────────
def get_model_for_inference(tag: str):
    """Retorna (model_on_device, tokenizer) ou (None, None)."""
    r = ALL_RESULTS.get(tag, {})
    if not r or r.get('model_cpu') is None:
        return None, None
    return r['model_cpu'].to(DEVICE).eval(), r['tokenizer']


print('✅ C35 concluída — MODEL_CONFIGS e funções de treino prontos.')
print(f'   Modelos: {list(MODEL_CONFIGS.keys())}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C36 — DATALOADERS PARA TODOS OS EXPERIMENTOS                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def make_loaders(df: pd.DataFrame, tokenizer,
                  use_schema: bool,
                  lang: Optional[str] = None,
                  source: Optional[str] = None,
                  tag: str = '') -> Dict[str, DataLoader]:
    """Constrói {train, validation, test} DataLoaders com filtros opcionais."""
    loaders = {}
    for split, shuffle in [('train', True), ('validation', False), ('test', False)]:
        mask = df['split'] == split
        if lang:   mask &= df['lang']   == lang
        if source: mask &= df['source'] == source
        sub = df[mask].reset_index(drop=True)
        if len(sub) == 0:
            continue
        try:
            ds = T5SQLDataset(sub, tokenizer,
                               CFG['ft_max_src_len'], CFG['ft_max_tgt_len'],
                               use_schema)
        except ValueError as e:
            print(f'  ❌ [{tag}/{split}] {e}')
            continue
        loaders[split] = DataLoader(
            ds, batch_size=CFG['ft_batch_size'], shuffle=shuffle,
            num_workers=0, pin_memory=torch.cuda.is_available()
        )
        print(f'  [{tag}|{"sch" if use_schema else "nos"}] '
              f'{split:12s}: {len(ds):,} amostras')
    return loaders


# ── Carregar tokenizers ────────────────────────────────────────────────────
print('📥 Carregando tokenizers...')
TOKENIZERS: Dict[str, object] = {}
for mk, cfg_m in MODEL_CONFIGS.items():
    try:
        TOKENIZERS[mk] = cfg_m['tok_cls'].from_pretrained(cfg_m['hf_id'])
        print(f'  ✅ {cfg_m["label"]}')
    except Exception as e:
        print(f'  ⚠️  {cfg_m["label"]} fallback t5-small: {e}')
        TOKENIZERS[mk] = T5TokenizerFast.from_pretrained('t5-small')

# ── Conjuntos de teste fixos ───────────────────────────────────────────────
DF_TEST = {
    'pt'    : df_all[(df_all['split']=='test') &
                      (df_all['lang']=='pt')].reset_index(drop=True),
    'en'    : df_all[(df_all['split']=='test') &
                      (df_all['lang']=='en')].reset_index(drop=True),
    'en_exp': df_all_expanded[(df_all_expanded['split']=='test') &
                               (df_all_expanded['lang']=='en')].reset_index(drop=True),
}
# Subsets de validação para monitoramento BLEU durante treino
DF_VAL = {
    'pt'    : df_all[(df_all['split']=='validation') &
                      (df_all['lang']=='pt')].reset_index(drop=True),
    'en'    : df_all[(df_all['split']=='validation') &
                      (df_all['lang']=='en')].reset_index(drop=True),
    'en_exp': df_all_expanded[(df_all_expanded['split']=='validation') &
                               (df_all_expanded['lang']=='en')].reset_index(drop=True),
}
print(f'\nTeste: PT={len(DF_TEST["pt"]):,} | '
      f'EN={len(DF_TEST["en"]):,} | EN_EXP={len(DF_TEST["en_exp"]):,}')

# ── Construir loaders para cada modelo × experimento ─────────────────────
# LOADERS[model_key][exp_key] → {train: DL, validation: DL, test: DL}
LOADERS: Dict[str, Dict[str, Dict]] = {}

for mk in MODEL_CONFIGS:
    tok = TOKENIZERS[mk]
    LOADERS[mk] = {}
    lbl = MODEL_CONFIGS[mk]['label']
    print(f'\n── {lbl} ──────────────────────────────────────────────────────')

    print('  PT+C4AI sem schema:')
    LOADERS[mk]['pt_nos'] = make_loaders(
        df_all, tok, False, lang='pt', tag=f'{mk}/pt_nos')

    print('  PT+C4AI com schema:')
    LOADERS[mk]['pt_sch'] = make_loaders(
        df_all, tok, True,  lang='pt', tag=f'{mk}/pt_sch')

    print('  EN Spider sem schema:')
    LOADERS[mk]['en_nos'] = make_loaders(
        df_all, tok, False, lang='en', tag=f'{mk}/en_nos')

    print('  EN Spider com schema:')
    LOADERS[mk]['en_sch'] = make_loaders(
        df_all, tok, True,  lang='en', tag=f'{mk}/en_sch')

    print('  EN Expandido sem schema:')
    LOADERS[mk]['enx_nos'] = make_loaders(
        df_all_expanded, tok, False, lang='en', tag=f'{mk}/enx_nos')

    print('  EN Expandido com schema:')
    LOADERS[mk]['enx_sch'] = make_loaders(
        df_all_expanded, tok, True,  lang='en', tag=f'{mk}/enx_sch')

print('\n✅ C36 concluída — todos os DataLoaders prontos.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C37 — TREINO T5-SQL: 6 EXPERIMENTOS                                     ║
# ║                                                                          ║
# ║  Hipótese: T5-SQL (pré-treinado em WikiSQL em inglês) terá desempenho    ║
# ║  superior em EN, especialmente com mais dados (EN Expandido).            ║
# ║  Em PT esperamos resultado mais fraco devido ao vocabulário EN-cêntrico. ║
# ╚══════════════════════════════════════════════════════════════════════════╝
MK = 't5sql'

T5SQL_EXP = [
    # (tag, loader_key, val_key, test_key, use_schema, descrição)
    ('t5_pt_nos',  'pt_nos',  'pt',     'pt',     False, 'T5-SQL | PT+C4AI      | sem schema'),
    ('t5_pt_sch',  'pt_sch',  'pt',     'pt',     True,  'T5-SQL | PT+C4AI      | com schema'),
    ('t5_en_nos',  'en_nos',  'en',     'en',     False, 'T5-SQL | EN Spider    | sem schema'),
    ('t5_en_sch',  'en_sch',  'en',     'en',     True,  'T5-SQL | EN Spider    | com schema'),
    ('t5_enx_nos', 'enx_nos', 'en_exp', 'en_exp', False, 'T5-SQL | EN Expandido | sem schema'),
    ('t5_enx_sch', 'enx_sch', 'en_exp', 'en_exp', True,  'T5-SQL | EN Expandido | com schema'),
]

for tag, lk, vk, tk, sch, desc in T5SQL_EXP:
    train_one_model(
        model_key   = MK,
        tag         = tag,
        loaders     = LOADERS[MK][lk],
        df_val_mon  = DF_VAL[vk],
        df_test     = DF_TEST[tk],
        use_schema  = sch,
        description = desc,
    )

# Resumo T5-SQL
print('\n' + '═' * 72)
print('── Resumo T5-SQL ────────────────────────────────────────────────────')
print(f'  {"Descrição":<46} {"B1":>6} {"B4":>6} {"PPL":>7} {"Épocas":>7}')
print('  ' + '─' * 70)
for tag, *_, desc in T5SQL_EXP:
    r = ALL_RESULTS.get(tag, {})
    if r:
        print(f'  {r["description"]:<46} '
              f'{r["bleu1_test"]:>6.2f} {r["bleu4_test"]:>6.2f} '
              f'{r["best_ppl"]:>7.1f} {r["epochs"]:>7}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C38 — TREINO mT5: 6 EXPERIMENTOS                                        ║
# ║                                                                          ║
# ║  Hipótese: mT5 (multilíngue, 101 idiomas) terá vantagem sobre T5-SQL     ║
# ║  em PT (vocabulário nativo) e resultado competitivo em EN.               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
MK = 'mt5'

MT5_EXP = [
    ('mt5_pt_nos',  'pt_nos',  'pt',     'pt',     False, 'mT5 | PT+C4AI      | sem schema'),
    ('mt5_pt_sch',  'pt_sch',  'pt',     'pt',     True,  'mT5 | PT+C4AI      | com schema'),
    ('mt5_en_nos',  'en_nos',  'en',     'en',     False, 'mT5 | EN Spider    | sem schema'),
    ('mt5_en_sch',  'en_sch',  'en',     'en',     True,  'mT5 | EN Spider    | com schema'),
    ('mt5_enx_nos', 'enx_nos', 'en_exp', 'en_exp', False, 'mT5 | EN Expandido | sem schema'),
    ('mt5_enx_sch', 'enx_sch', 'en_exp', 'en_exp', True,  'mT5 | EN Expandido | com schema'),
]

for tag, lk, vk, tk, sch, desc in MT5_EXP:
    train_one_model(
        model_key   = MK,
        tag         = tag,
        loaders     = LOADERS[MK][lk],
        df_val_mon  = DF_VAL[vk],
        df_test     = DF_TEST[tk],
        use_schema  = sch,
        description = desc,
    )

print('\n' + '═' * 72)
print('── Resumo mT5 ───────────────────────────────────────────────────────')
print(f'  {"Descrição":<44} {"B1":>6} {"B4":>6} {"PPL":>7} {"Épocas":>7}')
print('  ' + '─' * 68)
for tag, *_, desc in MT5_EXP:
    r = ALL_RESULTS.get(tag, {})
    if r:
        print(f'  {r["description"]:<44} '
              f'{r["bleu1_test"]:>6.2f} {r["bleu4_test"]:>6.2f} '
              f'{r["best_ppl"]:>7.1f} {r["epochs"]:>7}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C39 — TREINO PTT5: 2 EXPERIMENTOS (PT+C4AI APENAS)                      ║
# ║                                                                          ║
# ║  PTT5 é especializado em PT via continuation pretraining na Wikipedia    ║
# ║  PT. Esperado: PTT5 ≥ mT5 ≥ T5-SQL em tarefas PT.                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
MK = 'ptt5'

PTT5_EXP = [
    ('ptt5_pt_nos', 'pt_nos', 'pt', 'pt', False, 'PTT5 | PT+C4AI | sem schema'),
    ('ptt5_pt_sch', 'pt_sch', 'pt', 'pt', True,  'PTT5 | PT+C4AI | com schema'),
]

for tag, lk, vk, tk, sch, desc in PTT5_EXP:
    train_one_model(
        model_key   = MK,
        tag         = tag,
        loaders     = LOADERS[MK][lk],
        df_val_mon  = DF_VAL[vk],
        df_test     = DF_TEST[tk],
        use_schema  = sch,
        description = desc,
    )

print('\n' + '═' * 72)
print('── Resumo PTT5 ──────────────────────────────────────────────────────')
for tag, *_, desc in PTT5_EXP:
    r = ALL_RESULTS.get(tag, {})
    if r:
        print(f'  {r["description"]:<44} '
              f'B1={r["bleu1_test"]:.2f} | B4={r["bleu4_test"]:.2f} | '
              f'PPL={r["best_ppl"]:.1f} | Épocas={r["epochs"]}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C40 — AVALIAÇÃO GLOBAL: TABELA + PLOTS                                  ║
# ║                                                                          ║
# ║  fig10  Tabela comparativa global (todos os 14 experimentos)             ║
# ║  fig11  Curvas de loss por modelo (3 subplots: T5/mT5/PTT5)              ║
# ║  fig12  BLEU-1 e BLEU-4 no teste — todos os experimentos (barras)        ║
# ║  fig13  Perplexidade de validação — todos os experimentos                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
if not ALL_RESULTS:
    print('⚠️  ALL_RESULTS vazio — execute C37–C39 primeiro.')
else:
    # ── Tabela global ─────────────────────────────────────────────────────
    rows = []
    for tag, r in ALL_RESULTS.items():
        rows.append({
            'Modelo'        : r['label'],
            'Dados'         : r['description'].split('|')[1].strip() if '|' in r['description'] else '',
            'Schema'        : 'Sim' if r['use_schema'] else 'Não',
            'N treino'      : f'{r["n_train"]:,}',
            'Épocas'        : r['epochs'],
            'Val Loss'      : f'{r["best_val"]:.4f}',
            'PPL val'       : f'{r["best_ppl"]:.1f}',
            'BLEU-1 val'    : f'{max(r["history"]["bleu1_val"], default=0.0):.2f}',
            'BLEU-1 teste'  : f'{r["bleu1_test"]:.2f}',
            'BLEU-4 teste'  : f'{r["bleu4_test"]:.2f}',
        })
    df_global = pd.DataFrame(rows, index=list(ALL_RESULTS.keys()))

    print('═' * 82)
    print('  TABELA GLOBAL — 14 EXPERIMENTOS DE FINE-TUNING')
    print('═' * 82)
    print(df_global.to_string())

    # ── fig10: Tabela como figura ─────────────────────────────────────────
    df_fig10 = df_global[['Modelo','Dados','Schema','N treino',
                            'PPL val','BLEU-1 teste','BLEU-4 teste']].copy()
    fig10, ax10 = plt.subplots(figsize=(18, 0.6 + 0.42 * len(df_fig10)))
    ax10.axis('off')
    tbl = ax10.table(cellText=df_fig10.values, colLabels=df_fig10.columns,
                      cellLoc='center', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1.0, 1.9)
    for j in range(len(df_fig10.columns)):
        tbl[0, j].set_facecolor('#CCCCDD'); tbl[0, j].set_text_props(fontweight='bold')
    model_bg = {'T5-SQL': '#FFF0EE', 'mT5': '#EEF4FF', 'PTT5': '#EEFFF5'}
    for i in range(1, len(df_fig10) + 1):
        bg = model_bg.get(df_global.iloc[i-1]['Modelo'], '#FFFFFF')
        for j in range(len(df_fig10.columns)):
            tbl[i, j].set_facecolor(bg)
    plt.title('Tabela Global — Fine-Tuning Seção 7 (T5-SQL, mT5, PTT5)',
               fontsize=11, pad=10)
    plt.tight_layout()
    fig10.savefig(f'{PLOTS_DIR}/fig10_ft_tabela_global.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  ✅ Salvo: {PLOTS_DIR}/fig10_ft_tabela_global.png')

    # ── fig11: Curvas de loss por modelo (3 subplots) ─────────────────────
    grupos = {
        'T5-SQL': [t for t in ALL_RESULTS if t.startswith('t5_')],
        'mT5'   : [t for t in ALL_RESULTS if t.startswith('mt5_')],
        'PTT5'  : [t for t in ALL_RESULTS if t.startswith('ptt5_')],
    }
    exp_colors = {
        'pt_nos': '#FF8C69', 'pt_sch': '#CC2200',
        'en_nos': '#87CEEB', 'en_sch': '#1565C0',
        'enx_nos':'#90EE90', 'enx_sch':'#1B5E20',
    }
    exp_labels_short = {
        'pt_nos': 'PT sem sch', 'pt_sch': 'PT com sch',
        'en_nos': 'EN sem sch', 'en_sch': 'EN com sch',
        'enx_nos':'ENx sem sch','enx_sch':'ENx com sch',
    }

    fig11, axes11 = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
    for ax, (grp, tags) in zip(axes11, grupos.items()):
        for tag in tags:
            if tag not in ALL_RESULTS: continue
            r   = ALL_RESULTS[tag]
            his = r['history']
            eps = range(1, len(his['train_loss']) + 1)
            # Determinar chave de cor pelo sufixo do tag
            sfx = '_'.join(tag.split('_')[1:])
            c   = exp_colors.get(sfx, 'gray')
            lbl = exp_labels_short.get(sfx, sfx)
            ax.plot(eps, his['train_loss'], c=c, linestyle='--', alpha=0.35, linewidth=1)
            ax.plot(eps, his['val_loss'],   c=c, linestyle='-',  linewidth=2, label=lbl)
        ax.set_title(f'Loss — {grp}')
        ax.set_xlabel('Época'); ax.set_ylabel('Loss')
        ax.legend(fontsize=8)

    plt.suptitle('Curvas de Loss por Modelo (tracejado claro = train; sólido = val)',
                  fontsize=13)
    plt.tight_layout()
    fig11.savefig(f'{PLOTS_DIR}/fig11_ft_loss_curvas.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  ✅ Salvo: {PLOTS_DIR}/fig11_ft_loss_curvas.png')

    # ── fig12: BLEU-1 e BLEU-4 no teste ──────────────────────────────────
    all_tags   = list(ALL_RESULTS.keys())
    b1_vals    = [ALL_RESULTS[t]['bleu1_test'] for t in all_tags]
    b4_vals    = [ALL_RESULTS[t]['bleu4_test'] for t in all_tags]
    bar_labels = [ALL_RESULTS[t]['description'].replace(' | ', '\n') for t in all_tags]
    bar_colors = [ALL_RESULTS[t]['color']  for t in all_tags]
    x = np.arange(len(all_tags)); w = 0.35

    fig12, ax12 = plt.subplots(figsize=(max(16, len(all_tags) * 1.4), 7))
    bars1 = ax12.bar(x - w/2, b1_vals, w, label='BLEU-1',
                      color=bar_colors, alpha=0.9, edgecolor='white')
    bars4 = ax12.bar(x + w/2, b4_vals, w, label='BLEU-4',
                      color=bar_colors, alpha=0.5, edgecolor='white', hatch='//')
    for bar, val in [(b, v) for b, v in zip(bars1, b1_vals) if v > 0]:
        ax12.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                  f'{val:.1f}', ha='center', fontsize=7, fontweight='bold')
    for bar, val in [(b, v) for b, v in zip(bars4, b4_vals) if v > 0]:
        ax12.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                  f'{val:.1f}', ha='center', fontsize=7, fontweight='bold')
    ax12.set_xticks(x)
    ax12.set_xticklabels(bar_labels, rotation=35, ha='right', fontsize=7.5)
    ax12.set_ylabel('Score BLEU (↑ melhor)')
    ax12.set_title('BLEU-1 e BLEU-4 no Conjunto de Teste — Todos os Experimentos',
                    fontsize=12)
    ax12.legend(fontsize=10)
    max_b = max(b1_vals + b4_vals, default=1)
    ax12.set_ylim(0, max_b * 1.28)
    # Linhas verticais separando os grupos de modelos
    ax12.axvline(5.5, color='gray', linestyle=':', alpha=0.5)
    ax12.axvline(11.5, color='gray', linestyle=':', alpha=0.5)
    ax12.text(2.5,  max_b * 1.22, 'T5-SQL', ha='center', fontsize=9, color='#E24B4A')
    ax12.text(8.5,  max_b * 1.22, 'mT5',    ha='center', fontsize=9, color='#378ADD')
    ax12.text(12.5, max_b * 1.22, 'PTT5',   ha='center', fontsize=9, color='#1D9E75')
    plt.tight_layout()
    fig12.savefig(f'{PLOTS_DIR}/fig12_ft_bleu_global.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  ✅ Salvo: {PLOTS_DIR}/fig12_ft_bleu_global.png')

    # ── fig13: Perplexidade ───────────────────────────────────────────────
    ppl_vals = [ALL_RESULTS[t]['best_ppl'] for t in all_tags]
    fig13, ax13 = plt.subplots(figsize=(max(14, len(all_tags) * 1.3), 5))
    bars_p = ax13.bar(x, ppl_vals, color=bar_colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars_p, ppl_vals):
        ax13.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                  f'{val:.1f}', ha='center', fontsize=7)
    ax13.set_xticks(x)
    ax13.set_xticklabels(bar_labels, rotation=35, ha='right', fontsize=7.5)
    ax13.set_ylabel('Perplexidade de Validação (↓ melhor)')
    ax13.set_title('Perplexidade — Todos os Experimentos', fontsize=12)
    ax13.axvline(5.5, color='gray', linestyle=':', alpha=0.5)
    ax13.axvline(11.5, color='gray', linestyle=':', alpha=0.5)
    plt.tight_layout()
    fig13.savefig(f'{PLOTS_DIR}/fig13_ft_ppl_global.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  ✅ Salvo: {PLOTS_DIR}/fig13_ft_ppl_global.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C41 — COMPARAÇÃO T5-SQL vs mT5                                         ║
# ║                                                                          ║
# ║  Compara T5-SQL e mT5 em cada cenário equivalente.                      ║
# ║  Δ BLEU-4 = mT5 − T5-SQL: verde = mT5 melhor; vermelho = T5-SQL melhor.║
# ╚══════════════════════════════════════════════════════════════════════════╝
pairs_comp = [
    ('t5_pt_nos',  'mt5_pt_nos',  'PT+C4AI sem schema'),
    ('t5_pt_sch',  'mt5_pt_sch',  'PT+C4AI com schema'),
    ('t5_en_nos',  'mt5_en_nos',  'EN Spider sem schema'),
    ('t5_en_sch',  'mt5_en_sch',  'EN Spider com schema'),
    ('t5_enx_nos', 'mt5_enx_nos', 'EN Expandido sem schema'),
    ('t5_enx_sch', 'mt5_enx_sch', 'EN Expandido com schema'),
]

# Tabela comparativa
comp_rows = []
for t5t, mt5t, label in pairs_comp:
    r5  = ALL_RESULTS.get(t5t, {})
    rm5 = ALL_RESULTS.get(mt5t, {})
    if not r5 or not rm5:
        continue
    comp_rows.append({
        'Cenário'      : label,
        'T5 B1'        : f'{r5["bleu1_test"]:.2f}',
        'T5 B4'        : f'{r5["bleu4_test"]:.2f}',
        'T5 PPL'       : f'{r5["best_ppl"]:.1f}',
        'mT5 B1'       : f'{rm5["bleu1_test"]:.2f}',
        'mT5 B4'       : f'{rm5["bleu4_test"]:.2f}',
        'mT5 PPL'      : f'{rm5["best_ppl"]:.1f}',
        'Δ B4 (mT5−T5)': f'{rm5["bleu4_test"] - r5["bleu4_test"]:+.2f}',
    })

df_comp = pd.DataFrame(comp_rows)
print('═' * 78)
print('  COMPARAÇÃO T5-SQL vs mT5')
print('═' * 78)
print(df_comp.to_string(index=False))

if comp_rows:
    labels_c = [r['Cenário'] for r in comp_rows]
    t5_b4    = [float(r['T5 B4'])  for r in comp_rows]
    mt5_b4   = [float(r['mT5 B4']) for r in comp_rows]
    deltas   = [float(r['Δ B4 (mT5−T5)']) for r in comp_rows]
    x_c = np.arange(len(labels_c)); w_c = 0.3

    fig14, (ax14a, ax14b) = plt.subplots(1, 2, figsize=(18, 6))

    # BLEU-4 lado a lado
    b_t5  = ax14a.bar(x_c - w_c/2, t5_b4,  w_c, label='T5-SQL',
                       color='#E24B4A', alpha=0.85, edgecolor='white')
    b_mt5 = ax14a.bar(x_c + w_c/2, mt5_b4, w_c, label='mT5',
                       color='#378ADD', alpha=0.85, edgecolor='white')
    for bar, val in list(zip(b_t5, t5_b4)) + list(zip(b_mt5, mt5_b4)):
        if val > 0:
            ax14a.text(bar.get_x() + bar.get_width()/2,
                       bar.get_height() + 0.15, f'{val:.1f}',
                       ha='center', fontsize=8, fontweight='bold')
    ax14a.set_xticks(x_c)
    ax14a.set_xticklabels(labels_c, rotation=22, ha='right', fontsize=9)
    ax14a.set_ylabel('BLEU-4 (↑ melhor)')
    ax14a.set_title('BLEU-4 no Teste: T5-SQL vs mT5')
    ax14a.legend(fontsize=10)

    # Δ BLEU-4
    delta_colors = ['#1D9E75' if d >= 0 else '#E24B4A' for d in deltas]
    ax14b.bar(x_c, deltas, color=delta_colors, alpha=0.85, edgecolor='white')
    ax14b.axhline(0, color='black', linewidth=0.8)
    for i, (xp, val) in enumerate(zip(x_c, deltas)):
        ax14b.text(xp, val + (0.08 if val >= 0 else -0.18),
                   f'{val:+.2f}', ha='center', fontsize=8, fontweight='bold')
    ax14b.set_xticks(x_c)
    ax14b.set_xticklabels(labels_c, rotation=22, ha='right', fontsize=9)
    ax14b.set_ylabel('Δ BLEU-4 (mT5 − T5-SQL)')
    ax14b.set_title('Diferença BLEU-4\nverde = mT5 melhor | vermelho = T5-SQL melhor')

    plt.suptitle('Comparação T5-SQL vs mT5', fontsize=13)
    plt.tight_layout()
    fig14.savefig(f'{PLOTS_DIR}/fig14_comp_t5_mt5.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  ✅ Salvo: {PLOTS_DIR}/fig14_comp_t5_mt5.png')

    # Tabela como figura
    fig14b, ax14b2 = plt.subplots(figsize=(18, 1.8 + 0.45 * len(df_comp)))
    ax14b2.axis('off')
    tbl14 = ax14b2.table(cellText=df_comp.values, colLabels=df_comp.columns,
                          cellLoc='center', loc='center')
    tbl14.auto_set_font_size(False); tbl14.set_fontsize(9); tbl14.scale(1.0, 2.0)
    for j in range(len(df_comp.columns)):
        tbl14[0, j].set_facecolor('#CCCCDD')
        tbl14[0, j].set_text_props(fontweight='bold')
    # Colorir Δ: verde se positivo, vermelho se negativo
    for i in range(1, len(df_comp) + 1):
        d = float(df_comp.iloc[i-1]['Δ B4 (mT5−T5)'])
        tbl14[i, -1].set_facecolor('#D4EDDA' if d >= 0 else '#F8D7DA')
    plt.title('Tabela Comparativa: T5-SQL vs mT5', fontsize=11, pad=10)
    plt.tight_layout()
    fig14b.savefig(f'{PLOTS_DIR}/fig14b_tabela_t5_mt5.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  ✅ Salvo: {PLOTS_DIR}/fig14b_tabela_t5_mt5.png')

    # Inferência qualitativa: T5-SQL vs mT5
    cases_41 = [
        {'lang':'pt', 'q':'Quais são os nomes de todos os cantores?',
         'schema':'CREATE TABLE singer (singer_id INTEGER, name VARCHAR, country VARCHAR, age INTEGER)',
         'ref':'SELECT name FROM singer',
         't5_nos':'t5_pt_nos','mt5_nos':'mt5_pt_nos',
         't5_sch':'t5_pt_sch','mt5_sch':'mt5_pt_sch'},
        {'lang':'en', 'q':'How many concerts were held in 2014?',
         'schema':'CREATE TABLE concert (concert_id INTEGER, concert_name VARCHAR, year INTEGER)',
         'ref':'SELECT count(*) FROM concert WHERE year = 2014',
         't5_nos':'t5_en_nos','mt5_nos':'mt5_en_nos',
         't5_sch':'t5_en_sch','mt5_sch':'mt5_en_sch'},
        {'lang':'en', 'q':'What is the average salary per department?',
         'schema':'CREATE TABLE employees (id INTEGER, name VARCHAR, salary FLOAT, dept VARCHAR)',
         'ref':'SELECT dept, avg(salary) FROM employees GROUP BY dept',
         't5_nos':'t5_enx_nos','mt5_nos':'mt5_enx_nos',
         't5_sch':'t5_enx_sch','mt5_sch':'mt5_enx_sch'},
    ]

    print('\n── Inferência qualitativa: T5-SQL vs mT5 ────────────────────────')
    for case in cases_41:
        print(f"\n📝 [{case['lang'].upper()}] {case['q']}")
        print(f"   Ref: {case['ref']}")
        for label, tag, use_sch in [
            ('T5-SQL sem schema', case['t5_nos'],  False),
            ('T5-SQL com schema', case['t5_sch'],  True),
            ('mT5    sem schema', case['mt5_nos'], False),
            ('mT5    com schema', case['mt5_sch'], True),
        ]:
            m, tok = get_model_for_inference(tag)
            if m is None:
                print(f'   {label:<22}: [não treinado]')
                continue
            sch  = case['schema'] if use_sch else ''
            pred = _generate(m, tok, case['q'], schema=sch)
            print(f'   {label:<22}: {pred or "[vazio]"}')
            m.cpu(); gc.collect()
        print('─' * 72)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C42 — COMPARAÇÃO PT-ONLY: T5-SQL vs mT5 vs PTT5                         ║
# ║                                                                          ║
# ║  Foco exclusivo nos dados PT+C4AI (sem e com schema).                    ║
# ║  Hipótese de ranking esperado: PTT5 ≥ mT5 ≥ T5-SQL                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
pt_tags_nosql = ['t5_pt_nos',  'mt5_pt_nos',  'ptt5_pt_nos']
pt_tags_schema= ['t5_pt_sch',  'mt5_pt_sch',  'ptt5_pt_sch']
model_labels  = ['T5-SQL', 'mT5', 'PTT5']
colors_pt     = ['#E24B4A', '#378ADD', '#1D9E75']

print('═' * 72)
print('  COMPARAÇÃO PT-ONLY: T5-SQL vs mT5 vs PTT5')
print('═' * 72)

# Tabela
pt_comp_rows = []
for ns_tag, sc_tag, mlabel in zip(pt_tags_nosql, pt_tags_schema, model_labels):
    r_ns = ALL_RESULTS.get(ns_tag, {})
    r_sc = ALL_RESULTS.get(sc_tag, {})
    pt_comp_rows.append({
        'Modelo'           : mlabel,
        'B1 sem schema'    : f'{r_ns.get("bleu1_test", 0.0):.2f}',
        'B4 sem schema'    : f'{r_ns.get("bleu4_test", 0.0):.2f}',
        'PPL sem schema'   : f'{r_ns.get("best_ppl",   0.0):.1f}',
        'B1 com schema'    : f'{r_sc.get("bleu1_test", 0.0):.2f}',
        'B4 com schema'    : f'{r_sc.get("bleu4_test", 0.0):.2f}',
        'PPL com schema'   : f'{r_sc.get("best_ppl",   0.0):.1f}',
    })

df_pt_comp = pd.DataFrame(pt_comp_rows)
print(df_pt_comp.to_string(index=False))

# fig15a: BLEU-4 sem e com schema (barras agrupadas por modelo)
b4_ns  = [ALL_RESULTS.get(t, {}).get('bleu4_test', 0.0) for t in pt_tags_nosql]
b4_sc  = [ALL_RESULTS.get(t, {}).get('bleu4_test', 0.0) for t in pt_tags_schema]
ppl_ns = [ALL_RESULTS.get(t, {}).get('best_ppl',   0.0) for t in pt_tags_nosql]
ppl_sc = [ALL_RESULTS.get(t, {}).get('best_ppl',   0.0) for t in pt_tags_schema]
x42    = np.arange(3); w42 = 0.3

fig15, axes15 = plt.subplots(1, 2, figsize=(16, 6))

# BLEU-4
ax = axes15[0]
b_ns_bars = ax.bar(x42 - w42/2, b4_ns, w42, label='Sem schema',
                    color=colors_pt, alpha=0.85, edgecolor='white')
b_sc_bars = ax.bar(x42 + w42/2, b4_sc, w42, label='Com schema',
                    color=colors_pt, alpha=0.5,  edgecolor='white', hatch='//')
for bar, val in list(zip(b_ns_bars, b4_ns)) + list(zip(b_sc_bars, b4_sc)):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x42); ax.set_xticklabels(model_labels, fontsize=11)
ax.set_ylabel('BLEU-4 (↑ melhor)')
ax.set_title('BLEU-4 no Teste — PT+C4AI')
ax.legend(fontsize=10)

# Perplexidade
ax2 = axes15[1]
p_ns_bars = ax2.bar(x42 - w42/2, ppl_ns, w42, label='Sem schema',
                     color=colors_pt, alpha=0.85, edgecolor='white')
p_sc_bars = ax2.bar(x42 + w42/2, ppl_sc, w42, label='Com schema',
                     color=colors_pt, alpha=0.5,  edgecolor='white', hatch='//')
for bar, val in list(zip(p_ns_bars, ppl_ns)) + list(zip(p_sc_bars, ppl_sc)):
    if val > 0:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}', ha='center', fontsize=9)
ax2.set_xticks(x42); ax2.set_xticklabels(model_labels, fontsize=11)
ax2.set_ylabel('Perplexidade (↓ melhor)')
ax2.set_title('Perplexidade — PT+C4AI')
ax2.legend(fontsize=10)

plt.suptitle('Comparação PT-Only: T5-SQL vs mT5 vs PTT5', fontsize=13)
plt.tight_layout()
fig15.savefig(f'{PLOTS_DIR}/fig15_comp_pt_only.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Salvo: {PLOTS_DIR}/fig15_comp_pt_only.png')

# Tabela como figura
fig15b, ax15b = plt.subplots(figsize=(16, 2.0 + 0.5 * 3))
ax15b.axis('off')
tbl15 = ax15b.table(cellText=df_pt_comp.values, colLabels=df_pt_comp.columns,
                     cellLoc='center', loc='center')
tbl15.auto_set_font_size(False); tbl15.set_fontsize(10); tbl15.scale(1.0, 2.2)
for j in range(len(df_pt_comp.columns)):
    tbl15[0, j].set_facecolor('#CCCCDD')
    tbl15[0, j].set_text_props(fontweight='bold')
for i, c in enumerate(['#FFF0EE', '#EEF4FF', '#EEFFF5'], start=1):
    for j in range(len(df_pt_comp.columns)):
        tbl15[i, j].set_facecolor(c)
plt.title('Tabela Comparativa PT-Only: T5-SQL vs mT5 vs PTT5', fontsize=11, pad=10)
plt.tight_layout()
fig15b.savefig(f'{PLOTS_DIR}/fig15b_tabela_pt_only.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Salvo: {PLOTS_DIR}/fig15b_tabela_pt_only.png')

# Inferência qualitativa PT-only
cases_42 = [
    {'q':'Quais são os nomes de todos os cantores?',
     'schema':'CREATE TABLE singer (singer_id INTEGER, name VARCHAR, country VARCHAR, age INTEGER)',
     'ref':'SELECT name FROM singer'},
    {'q':'Quantos concertos foram realizados em 2014?',
     'schema':'CREATE TABLE concert (concert_id INTEGER, concert_name VARCHAR, year INTEGER)',
     'ref':'SELECT count(*) FROM concert WHERE year = 2014'},
    {'q':'Qual é a média de idade dos cantores por país?',
     'schema':'CREATE TABLE singer (singer_id INTEGER, name VARCHAR, country VARCHAR, age INTEGER)',
     'ref':'SELECT country, avg(age) FROM singer GROUP BY country'},
]

print('\n── Inferência qualitativa PT-Only ────────────────────────────────')
for case in cases_42:
    print(f"\n📝 [PT] {case['q']}")
    print(f"   Ref: {case['ref']}")
    for ns_tag, sc_tag, mlabel in zip(pt_tags_nosql, pt_tags_schema, model_labels):
        for tag, use_sch, sfx in [(ns_tag, False, 'sem sch'), (sc_tag, True, 'com sch')]:
            m, tok = get_model_for_inference(tag)
            sch  = case['schema'] if use_sch else ''
            pred = _generate(m, tok, case['q'], schema=sch) if m else '[não treinado]'
            print(f'   {mlabel} ({sfx}): {pred or "[vazio]"}')
            if m: m.cpu(); gc.collect()
    print('─' * 72)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C43 — PAINEL FINAL: LSTM vs FINE-TUNED (PT E EN)                        ║
# ║                                                                          ║
# ║  Painel PT: LSTM PT (Seção 5) × T5 PT sem/com × mT5 PT sem/com           ║
# ║             × PTT5 PT sem/com                                            ║
# ║  Painel EN: LSTM EN Spider (S5) × LSTM EN Exp (S6) × T5 EN 4 vars        ║
# ║             × mT5 EN 4 vars                                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Coletar métricas LSTM das seções anteriores ───────────────────────────
lstm_metrics = {}

# BLEU-1 do conjunto de teste calculado na C24
if 'bleu_pt_test' in dir():
    lstm_metrics['lstm_pt'] = {
        'label':'LSTM PT (S5)', 'color':'#2ECC71',
        'bleu1':bleu_pt_test,   'bleu4':0.0,
        'ppl': min(histories.get('pt',{}).get('val_ppl',[0])),
    }
if 'bleu_en_test' in dir():
    lstm_metrics['lstm_en'] = {
        'label':'LSTM EN Spider (S5)', 'color':'#8E44AD',
        'bleu1':bleu_en_test, 'bleu4':0.0,
        'ppl': min(histories.get('en',{}).get('val_ppl',[0])),
    }
if 'bleu_en_exp_test' in dir():
    lstm_metrics['lstm_en_exp'] = {
        'label':'LSTM EN Exp. (S6)', 'color':'#6C3483',
        'bleu1':bleu_en_exp_test, 'bleu4':0.0,
        'ppl': min(histories_exp.get('en_exp',{}).get('val_ppl',[0])),
    }


# ── Painel PT ─────────────────────────────────────────────────────────────
pt_panel_configs = []
if 'lstm_pt' in lstm_metrics:
    m = lstm_metrics['lstm_pt']
    pt_panel_configs.append((m['label'], m['bleu1'], 0.0, m['ppl'], m['color'], ''))

for tag, label, sch_sfx in [
    ('t5_pt_nos',  'T5-SQL sem', 'sem'),
    ('t5_pt_sch',  'T5-SQL com', 'com'),
    ('mt5_pt_nos', 'mT5 sem',    'sem'),
    ('mt5_pt_sch', 'mT5 com',    'com'),
    ('ptt5_pt_nos','PTT5 sem',   'sem'),
    ('ptt5_pt_sch','PTT5 com',   'com'),
]:
    r = ALL_RESULTS.get(tag, {})
    if r:
        pt_panel_configs.append(
            (label, r['bleu1_test'], r['bleu4_test'], r['best_ppl'], r['color'],
             'com' if 'sch' in tag else 'sem')
        )

labels_pt  = [c[0] for c in pt_panel_configs]
b1_pt      = [c[1] for c in pt_panel_configs]
b4_pt      = [c[2] for c in pt_panel_configs]
ppl_pt     = [c[3] for c in pt_panel_configs]
colors_pan = [c[4] for c in pt_panel_configs]
hatch_pt   = ['//' if c[5]=='com' else '' for c in pt_panel_configs]

x_pt = np.arange(len(labels_pt))

fig16, axes16 = plt.subplots(1, 2, figsize=(max(18, len(labels_pt)*1.8), 7))

ax = axes16[0]
for i, (b1, b4, lbl, col, ht) in enumerate(zip(b1_pt, b4_pt, labels_pt, colors_pan, hatch_pt)):
    ax.bar(i - 0.18, b1, 0.33, color=col, alpha=0.9,  edgecolor='white', label='BLEU-1' if i==0 else '')
    ax.bar(i + 0.18, b4, 0.33, color=col, alpha=0.5,  edgecolor='white', hatch='//', label='BLEU-4' if i==0 else '')
    if b1 > 0: ax.text(i-0.18, b1+0.1, f'{b1:.1f}', ha='center', fontsize=7, fontweight='bold')
    if b4 > 0: ax.text(i+0.18, b4+0.1, f'{b4:.1f}', ha='center', fontsize=7, fontweight='bold')
ax.set_xticks(x_pt); ax.set_xticklabels(labels_pt, rotation=30, ha='right', fontsize=8.5)
ax.set_ylabel('BLEU (↑ melhor)')
ax.set_title('BLEU no Teste — PT: LSTM vs Fine-Tuned')
# Linha separando LSTM dos modelos FT
if 'lstm_pt' in lstm_metrics:
    ax.axvline(0.5, color='black', linestyle=':', alpha=0.4)
    ax.text(0, max(b1_pt+b4_pt, default=1))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C44 — RESUMO FINAL DO PROJETO COMPLETO                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print('═' * 76)
print('  RESUMO FINAL — TEXT-TO-SQL NLP PROJECT')
print('═' * 76)

print(f'\n📊 DADOS')
print(f'  Spider original:              {len(df_all):,} pares (EN + PT + C4AI)')
print(f'  sql-create-context:           {len(df_sc):,} pares (EN)')
print(f'  Dataset expandido (Seção 6):  {len(df_all_expanded):,} pares')
print(f'    Spider PT+C4AI: {len(df_all[df_all["lang"]=="pt"]):,} | '
      f'Spider EN: {len(df_all[df_all["lang"]=="en"]):,}')

print(f'\n🏗️  LSTM — SEÇÃO 5 (Spider)')
for key, lbl in [('all','ALL'),('en','EN'),('pt','PT')]:
    if key not in histories: continue
    h  = histories[key]
    bt = {'all': getattr(__builtins__,'bleu_all_test',0) if 'bleu_all_test' in dir() else 0,
          'en' : bleu_en_test  if 'bleu_en_test'  in dir() else 0,
          'pt' : bleu_pt_test  if 'bleu_pt_test'  in dir() else 0}.get(key, 0)
    print(f'  [{lbl}] val_loss={min(h["val_loss"]):.4f} | '
          f'ppl={min(h["val_ppl"]):.1f} | BLEU-1 teste={bt:.1f}')

print(f'\n🏗️  LSTM — SEÇÃO 6 (Spider + sql-create-context)')
for key, lbl in [('all_exp','ALL_EXP'),('en_exp','EN_EXP'),('pt_exp','PT_EXP')]:
    if key not in histories_exp: continue
    h  = histories_exp[key]
    bt = {'all_exp': bleu_all_exp_test if 'bleu_all_exp_test' in dir() else 0,
          'en_exp' : bleu_en_exp_test  if 'bleu_en_exp_test'  in dir() else 0,
          'pt_exp' : bleu_pt_exp_test  if 'bleu_pt_exp_test'  in dir() else 0}.get(key, 0)
    print(f'  [{lbl}] val_loss={min(h["val_loss"]):.4f} | '
          f'ppl={min(h["val_ppl"]):.1f} | BLEU-1 teste={bt:.1f}')

print(f'\n🤖 FINE-TUNING — SEÇÃO 7 (14 experimentos)')
print(f'  {"Modelo":<8} {"Experimento":<42} {"B1":>6} {"B4":>6} {"PPL":>7}')
print('  ' + '─' * 68)
for tag, r in ALL_RESULTS.items():
    print(f'  {r["label"]:<8} {r["description"][:42]:<42} '
          f'{r["bleu1_test"]:>6.2f} {r["bleu4_test"]:>6.2f} '
          f'{r["best_ppl"]:>7.1f}')

print(f'\n🏆 MELHORES POR CATEGORIA (BLEU-4 no teste)')
cats = {
    'PT sem schema': [t for t in ALL_RESULTS if 'pt_nos' in t],
    'PT com schema': [t for t in ALL_RESULTS if 'pt_sch' in t],
    'EN Spider sem': [t for t in ALL_RESULTS if 'en_nos' in t and 'enx' not in t],
    'EN Spider com': [t for t in ALL_RESULTS if 'en_sch' in t and 'enx' not in t],
    'EN Exp. sem'  : [t for t in ALL_RESULTS if 'enx_nos' in t],
    'EN Exp. com'  : [t for t in ALL_RESULTS if 'enx_sch' in t],
}
for cat, tags in cats.items():
    if not tags: continue
    best = max(tags, key=lambda t: ALL_RESULTS.get(t,{}).get('bleu4_test', 0))
    r    = ALL_RESULTS[best]
    print(f'  {cat:<20}: {r["label"]:<8} {r["description"][:36]} '
          f'(B4={r["bleu4_test"]:.2f})')

print(f'\n📁 CHECKPOINTS: {CFG["checkpoint_dir"]}/')
for p in sorted(Path(CFG['checkpoint_dir']).glob('*.pt')):
    print(f'  {p.name:<50} {p.stat().st_size/1e6:.1f} MB')

print(f'\n📈 PLOTS SALVOS: {PLOTS_DIR}/')
for p in sorted(Path(PLOTS_DIR).glob('*.png')):
    print(f'  {p.name}')

print('\n✅ Projeto concluído.')